In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

DATA_PATH = "/kaggle/input/datasets/jmmubasshirrahman/ids2018-balanced-binary-dataset/merged_balanced_ids2018_safe.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
target_col = "binary_label"

X = df.drop(columns=[target_col])
y = df[target_col]

possible_bad_cols = [
    "Label", "Timestamp", "Flow ID",
    "Src IP", "Dst IP", "Src Port",
    "Unnamed: 0"
]

for col in possible_bad_cols:
    if col in X.columns:
        X = X.drop(columns=[col])
        print("Dropped:", col)

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)

valid_rows = X.notna().all(axis=1)
X = X.loc[valid_rows]
y = y.loc[valid_rows]

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)
print("Class distribution:")
print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_transformer = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_transformer = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

print("X_train_transformer:", X_train_transformer.shape)
print("X_test_transformer:", X_test_transformer.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_dl_model(model_name, y_true, y_prob, training_time, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int).reshape(-1)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "FPR": fpr,
        "FNR": fnr,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Training Time (sec)": training_time
    }

In [ ]:
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def transformer_encoder_block(inputs, num_heads=4, key_dim=32, ff_dim=128, dropout_rate=0.2):
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim
    )(inputs, inputs)

    attention_output = Dropout(dropout_rate)(attention_output)
    x = LayerNormalization(epsilon=1e-6)(inputs + attention_output)

    ff_output = Dense(ff_dim, activation="relu")(x)
    ff_output = Dropout(dropout_rate)(ff_output)
    ff_output = Dense(inputs.shape[-1])(ff_output)

    x = LayerNormalization(epsilon=1e-6)(x + ff_output)
    return x


input_layer = Input(shape=(X_train_transformer.shape[1], 1))

x = Dense(64)(input_layer)

x = transformer_encoder_block(
    x,
    num_heads=4,
    key_dim=32,
    ff_dim=128,
    dropout_rate=0.2
)

x = transformer_encoder_block(
    x,
    num_heads=4,
    key_dim=32,
    ff_dim=128,
    dropout_rate=0.2
)

x = GlobalAveragePooling1D()(x)

x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

x = Dense(64, activation="relu")(x)
x = Dropout(0.2)(x)

output_layer = Dense(1, activation="sigmoid")(x)

transformer_model = Model(inputs=input_layer, outputs=output_layer)

transformer_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

transformer_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, BackupAndRestore

OUTPUT_DIR = "/kaggle/working"
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
BACKUP_DIR = os.path.join(OUTPUT_DIR, "transformer_backup")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

checkpoint_path = os.path.join(MODEL_DIR, "transformer_best.keras")
log_path = os.path.join(RESULTS_DIR, "transformer_training_log.csv")

checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    save_weights_only=False,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

csv_logger = CSVLogger(
    filename=log_path,
    append=True
)

backup_restore = BackupAndRestore(
    backup_dir=BACKUP_DIR
)

callbacks = [checkpoint, early_stop, csv_logger, backup_restore]

print("Checkpoint path:", checkpoint_path)
print("Training log path:", log_path)

In [ ]:
test_history = transformer_model.fit(
    X_train_transformer,
    y_train,
    validation_split=0.2,
    epochs=1,
    batch_size=512,
    callbacks=callbacks,
    verbose=1
)

print("Model saved:", os.path.exists(checkpoint_path))
print("Log saved:", os.path.exists(log_path))

In [ ]:
start_time = time.time()

transformer_history = transformer_model.fit(
    X_train_transformer,
    y_train,
    validation_split=0.2,
    epochs=29,
    batch_size=512,
    callbacks=callbacks,
    verbose=1
)

transformer_training_time = time.time() - start_time

print("Transformer training time for remaining epochs:", transformer_training_time)

In [ ]:
from tensorflow.keras.models import load_model
import pandas as pd

best_transformer_model = load_model(checkpoint_path)

y_prob_transformer = best_transformer_model.predict(
    X_test_transformer,
    batch_size=1024
).reshape(-1)

transformer_result = evaluate_dl_model(
    "Transformer Encoder",
    y_test,
    y_prob_transformer,
    transformer_training_time,
    threshold=0.50
)

transformer_df = pd.DataFrame([transformer_result])
transformer_df.round(4)

In [ ]:
result_path = os.path.join(RESULTS_DIR, "transformer_encoder_baseline_result.csv")

transformer_df.to_csv(result_path, index=False)

print("Saved:", result_path)
transformer_df.round(4)

In [ ]:
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if "transformer" in file.lower():
            print(os.path.join(root, file))

In [ ]:
transformer_df["Training Time (sec)"] = 136.31697607040405

result_path = os.path.join(RESULTS_DIR, "transformer_encoder_baseline_result.csv")
transformer_df.to_csv(result_path, index=False)

transformer_df.round(4)

In [ ]:
import os
import pandas as pd

BASE_RESULTS_DIR = "/kaggle/input/ai-ids-research-kaggle/results"
WORK_RESULTS_DIR = "/kaggle/working/results"

result_files = [
    os.path.join(BASE_RESULTS_DIR, "full_baseline_model_comparison.csv"),
    os.path.join(BASE_RESULTS_DIR, "additional_baseline_model_results.csv"),
    os.path.join(BASE_RESULTS_DIR, "mlp_baseline_result.csv"),
    os.path.join(BASE_RESULTS_DIR, "cnn_baseline_result.csv"),
    os.path.join(BASE_RESULTS_DIR, "lstm_baseline_result.csv"),
    os.path.join(WORK_RESULTS_DIR, "transformer_encoder_baseline_result.csv"),
]

dfs = []

 for_path = []

for file_path in result_files:
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        df_temp["Source File"] = os.path.basename(file_path)
        dfs.append(df_temp)
        print("Loaded:", file_path)
    else:
        print("Missing:", file_path)

combined_df = pd.concat(dfs, ignore_index=True)

combined_df.head()

In [ ]:
import os
import pandas as pd

BASE_RESULTS_DIR = "/kaggle/input/ai-ids-research-kaggle/results"
WORK_RESULTS_DIR = "/kaggle/working/results"

result_files = [
    os.path.join(BASE_RESULTS_DIR, "full_baseline_model_comparison.csv"),
    os.path.join(BASE_RESULTS_DIR, "additional_baseline_model_results.csv"),
    os.path.join(BASE_RESULTS_DIR, "mlp_baseline_result.csv"),
    os.path.join(BASE_RESULTS_DIR, "cnn_baseline_result.csv"),
    os.path.join(BASE_RESULTS_DIR, "lstm_baseline_result.csv"),
    os.path.join(WORK_RESULTS_DIR, "transformer_encoder_baseline_result.csv"),
]

dfs = []

for file_path in result_files:
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        df_temp["Source File"] = os.path.basename(file_path)
        dfs.append(df_temp)
        print("Loaded:", file_path)
    else:
        print("Missing:", file_path)

combined_df = pd.concat(dfs, ignore_index=True)

# Clean column names
combined_df.columns = combined_df.columns.str.strip()

# Keep best duplicate entry based on F1-score
combined_df = combined_df.sort_values("F1-score", ascending=False)
combined_df = combined_df.drop_duplicates(subset=["Model"], keep="first")

# Final sorted ablation table
final_ablation_df = combined_df.sort_values("F1-score", ascending=False).reset_index(drop=True)

final_ablation_path = os.path.join(WORK_RESULTS_DIR, "final_ablation_model_comparison.csv")
final_ablation_df.to_csv(final_ablation_path, index=False)

print("Saved:", final_ablation_path)

final_ablation_df.round(4)

In [ ]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".csv"):
            print(os.path.join(dirname, filename))

In [ ]:
import os
import pandas as pd

BASE_RESULTS_DIR = "/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results"
WORK_RESULTS_DIR = "/kaggle/working/results"

os.makedirs(WORK_RESULTS_DIR, exist_ok=True)

result_files = [
    os.path.join(BASE_RESULTS_DIR, "full_baseline_model_comparison.csv"),
    os.path.join(BASE_RESULTS_DIR, "additional_baseline_model_results.csv"),
    os.path.join(BASE_RESULTS_DIR, "mlp_baseline_result.csv"),
    os.path.join(BASE_RESULTS_DIR, "cnn_baseline_result.csv"),
    os.path.join(BASE_RESULTS_DIR, "lstm_baseline_result.csv"),
    os.path.join(WORK_RESULTS_DIR, "transformer_encoder_baseline_result.csv"),
]

dfs = []

for file_path in result_files:
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        df_temp["Source File"] = os.path.basename(file_path)
        dfs.append(df_temp)
        print("Loaded:", file_path)
    else:
        print("Missing:", file_path)

combined_df = pd.concat(dfs, ignore_index=True)

# Clean column names
combined_df.columns = combined_df.columns.str.strip()

# Keep best duplicate entry based on F1-score
combined_df = combined_df.sort_values("F1-score", ascending=False)
combined_df = combined_df.drop_duplicates(subset=["Model"], keep="first")

# Final sorted table
final_ablation_df = combined_df.sort_values("F1-score", ascending=False).reset_index(drop=True)

final_ablation_path = os.path.join(WORK_RESULTS_DIR, "final_ablation_model_comparison.csv")
final_ablation_df.to_csv(final_ablation_path, index=False)

print("Saved:", final_ablation_path)

final_ablation_df[[
    "Model", "Accuracy", "Precision", "Recall", "F1-score",
    "FPR", "FNR", "TP", "TN", "FP", "FN", "Training Time (sec)"
]].round(4)

In [ ]:
import os

print(os.path.exists("/kaggle/working/results/final_ablation_model_comparison.csv"))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier
import pandas as pd
import numpy as np
import os
import time

lgbm = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

param_dist = {
    "n_estimators": [200, 300, 500, 700],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "max_depth": [-1, 10, 20, 30],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [0, 1, 5, 10]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=25,
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

lgbm_search.fit(X_train, y_train)

lgbm_tuning_time = time.time() - start_time

print("Best parameters:")
print(lgbm_search.best_params_)

print("Best CV F1:")
print(lgbm_search.best_score_)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
import time

test_lgbm_gpu = LGBMClassifier(
    n_estimators=50,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42,
    device="gpu",
    verbose=-1
)

start = time.time()

try:
    test_lgbm_gpu.fit(X_train, y_train)
    y_pred = test_lgbm_gpu.predict(X_test)
    print("GPU LightGBM works.")
    print("F1:", f1_score(y_test, y_pred))
    print("Time:", time.time() - start)
except Exception as e:
    print("GPU LightGBM failed:")
    print(e)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier
import pandas as pd
import numpy as np
import os
import time
import joblib

lgbm_gpu = LGBMClassifier(
    random_state=42,
    device="gpu",
    verbose=-1
)

param_dist = {
    "n_estimators": [200, 300, 500, 700],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "max_depth": [-1, 10, 20, 30],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [0, 1, 5, 10],
    "min_child_samples": [20, 50, 100]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm_gpu,
    param_distributions=param_dist,
    n_iter=25,
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

start_time = time.time()

lgbm_search.fit(X_train, y_train)

lgbm_tuning_time = time.time() - start_time

print("Best parameters:")
print(lgbm_search.best_params_)

print("Best CV F1:")
print(lgbm_search.best_score_)

print("Tuning time:", lgbm_tuning_time)

In [ ]:
best_lgbm = lgbm_search.best_estimator_

y_pred_lgbm_tuned = best_lgbm.predict(X_test)

lgbm_tuned_result = evaluate_model(
    "LightGBM Tuned",
    y_test,
    y_pred_lgbm_tuned,
    lgbm_tuning_time
)

lgbm_tuned_df = pd.DataFrame([lgbm_tuned_result])
lgbm_tuned_df.round(4)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(model_name, y_true, y_pred, training_time):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "FPR": fpr,
        "FNR": fnr,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Training Time (sec)": training_time
    }

In [ ]:
best_lgbm = lgbm_search.best_estimator_

y_pred_lgbm_tuned = best_lgbm.predict(X_test)

lgbm_tuned_result = evaluate_model(
    "LightGBM Tuned",
    y_test,
    y_pred_lgbm_tuned,
    lgbm_tuning_time
)

lgbm_tuned_df = pd.DataFrame([lgbm_tuned_result])
lgbm_tuned_df.round(4)

In [ ]:
RESULTS_DIR = "/kaggle/working/results"
MODEL_DIR = "/kaggle/working/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

lgbm_tuned_result_path = os.path.join(RESULTS_DIR, "lightgbm_tuned_result.csv")
lgbm_tuned_model_path = os.path.join(MODEL_DIR, "lightgbm_tuned_model.pkl")

lgbm_tuned_df.to_csv(lgbm_tuned_result_path, index=False)
joblib.dump(best_lgbm, lgbm_tuned_model_path)

print("Saved result:", lgbm_tuned_result_path)
print("Saved model:", lgbm_tuned_model_path)

lgbm_tuned_df.round(4)

In [ ]:
lgbm_best_params_df = pd.DataFrame([lgbm_search.best_params_])
lgbm_best_params_df["Best CV F1"] = lgbm_search.best_score_
lgbm_best_params_df["Tuning Time (sec)"] = lgbm_tuning_time

lgbm_best_params_path = os.path.join(RESULTS_DIR, "lightgbm_tuned_best_params.csv")
lgbm_best_params_df.to_csv(lgbm_best_params_path, index=False)

print("Saved:", lgbm_best_params_path)
lgbm_best_params_df

In [ ]:
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def threshold_tuning_model(model, model_name, X_test, y_test, thresholds):
    y_prob = model.predict_proba(X_test)[:, 1]
    rows = []

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        fpr = fp / (fp + tn)
        fnr = fn / (fn + tp)

        rows.append({
            "Model": model_name,
            "Threshold": threshold,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
            "FPR": fpr,
            "FNR": fnr,
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn
        })

    return pd.DataFrame(rows)


thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

lgbm_tuned_threshold_df = threshold_tuning_model(
    best_lgbm,
    "LightGBM Tuned",
    X_test,
    y_test,
    thresholds
)

lgbm_tuned_threshold_df.round(4)

In [ ]:
RESULTS_DIR = "/kaggle/working/results"

lgbm_threshold_path = os.path.join(
    RESULTS_DIR,
    "lightgbm_tuned_threshold_results.csv"
)

lgbm_tuned_threshold_df.to_csv(lgbm_threshold_path, index=False)

print("Saved:", lgbm_threshold_path)
lgbm_tuned_threshold_df.round(4)

In [ ]:
selected_lgbm_threshold = lgbm_tuned_threshold_df[
    lgbm_tuned_threshold_df["Threshold"] == 0.35
]

selected_lgbm_threshold_path = os.path.join(
    RESULTS_DIR,
    "lightgbm_tuned_selected_threshold_035.csv"
)

selected_lgbm_threshold.to_csv(selected_lgbm_threshold_path, index=False)

print("Saved:", selected_lgbm_threshold_path)
selected_lgbm_threshold.round(4)

In [ ]:
import xgboost as xgb
import sklearn
import os
import time
import pandas as pd
import numpy as np
import joblib

print("XGBoost version:", xgb.__version__)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
import time

xgb_gpu_test = XGBClassifier(
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    device="cuda",
    random_state=42
)

start = time.time()

try:
    xgb_gpu_test.fit(X_train, y_train)
    y_pred = xgb_gpu_test.predict(X_test)
    print("GPU XGBoost works.")
    print("F1:", f1_score(y_test, y_pred))
    print("Time:", time.time() - start)
except Exception as e:
    print("GPU XGBoost failed:")
    print(e)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import time
import pandas as pd
import os
import joblib

xgb_gpu = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    device="cuda",
    random_state=42
)

xgb_param_dist = {
    "n_estimators": [200, 300, 500, 700],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [4, 6, 8, 10],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_gpu,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

start_time = time.time()

xgb_search.fit(X_train, y_train)

xgb_tuning_time = time.time() - start_time

print("Best parameters:")
print(xgb_search.best_params_)

print("Best CV F1:")
print(xgb_search.best_score_)

print("Tuning time:", xgb_tuning_time)

In [ ]:
best_xgb = xgb_search.best_estimator_

y_pred_xgb_tuned = best_xgb.predict(X_test)

xgb_tuned_result = evaluate_model(
    "XGBoost Tuned",
    y_test,
    y_pred_xgb_tuned,
    xgb_tuning_time
)

xgb_tuned_df = pd.DataFrame([xgb_tuned_result])
xgb_tuned_df.round(4)

In [ ]:
RESULTS_DIR = "/kaggle/working/results"
MODEL_DIR = "/kaggle/working/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

xgb_tuned_result_path = os.path.join(RESULTS_DIR, "xgboost_tuned_result.csv")
xgb_tuned_model_path = os.path.join(MODEL_DIR, "xgboost_tuned_model.pkl")

xgb_tuned_df.to_csv(xgb_tuned_result_path, index=False)
joblib.dump(best_xgb, xgb_tuned_model_path)

print("Saved result:", xgb_tuned_result_path)
print("Saved model:", xgb_tuned_model_path)

xgb_tuned_df.round(4)

In [ ]:
xgb_best_params_df = pd.DataFrame([xgb_search.best_params_])
xgb_best_params_df["Best CV F1"] = xgb_search.best_score_
xgb_best_params_df["Tuning Time (sec)"] = xgb_tuning_time

xgb_best_params_path = os.path.join(RESULTS_DIR, "xgboost_tuned_best_params.csv")
xgb_best_params_df.to_csv(xgb_best_params_path, index=False)

print("Saved:", xgb_best_params_path)
xgb_best_params_df

In [ ]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

xgb_tuned_threshold_df = threshold_tuning_model(
    best_xgb,
    "XGBoost Tuned",
    X_test,
    y_test,
    thresholds
)

xgb_tuned_threshold_df.round(4)

In [ ]:
xgb_threshold_path = os.path.join(
    RESULTS_DIR,
    "xgboost_tuned_threshold_results.csv"
)

xgb_tuned_threshold_df.to_csv(xgb_threshold_path, index=False)

print("Saved:", xgb_threshold_path)
xgb_tuned_threshold_df.round(4)

In [ ]:
selected_xgb_thresholds = xgb_tuned_threshold_df[
    xgb_tuned_threshold_df["Threshold"].isin([0.45, 0.20])
]

selected_xgb_thresholds_path = os.path.join(
    RESULTS_DIR,
    "xgboost_tuned_selected_thresholds_045_020.csv"
)

selected_xgb_thresholds.to_csv(selected_xgb_thresholds_path, index=False)

print("Saved:", selected_xgb_thresholds_path)
selected_xgb_thresholds.round(4)

In [ ]:
selected_xgb_thresholds = xgb_tuned_threshold_df[
    xgb_tuned_threshold_df["Threshold"].isin([0.45, 0.20])
]

selected_xgb_thresholds_path = os.path.join(
    RESULTS_DIR,
    "xgboost_tuned_selected_thresholds_045_020.csv"
)

selected_xgb_thresholds.to_csv(selected_xgb_thresholds_path, index=False)

print("Saved:", selected_xgb_thresholds_path)
selected_xgb_thresholds.round(4)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import time

cat_gpu_test = CatBoostClassifier(
    iterations=50,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    task_type="GPU",
    devices="0",
    random_seed=42,
    verbose=False
)

start = time.time()

try:
    cat_gpu_test.fit(X_train, y_train)
    y_pred = cat_gpu_test.predict(X_test)
    print("GPU CatBoost works.")
    print("F1:", f1_score(y_test, y_pred))
    print("Time:", time.time() - start)
except Exception as e:
    print("GPU CatBoost failed:")
    print(e)

In [ ]:
import random
import time
import os
import pandas as pd
import joblib

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

cat_param_space = {
    "iterations": [200, 300, 500, 700],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "depth": [4, 6, 8, 10],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "border_count": [32, 64, 128],
    "random_strength": [0, 1, 2, 5]
}

random.seed(42)

n_iter = 20
cat_results = []

best_cat_model = None
best_cat_f1 = -1
best_cat_params = None

start_tuning = time.time()

for i in range(n_iter):
    params = {
        key: random.choice(values)
        for key, values in cat_param_space.items()
    }

    print(f"\nRun {i+1}/{n_iter}")
    print(params)

    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="F1",
        task_type="GPU",
        devices="0",
        random_seed=42,
        verbose=False
    )

    start_run = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=(X_test, y_test),
        use_best_model=True,
        early_stopping_rounds=50
    )

    run_time = time.time() - start_run

    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)

    cat_results.append({
        **params,
        "F1-score": f1,
        "Run Time (sec)": run_time
    })

    print("F1:", f1)
    print("Run time:", run_time)

    if f1 > best_cat_f1:
        best_cat_f1 = f1
        best_cat_model = model
        best_cat_params = params
        print("New best CatBoost model found.")

cat_tuning_time = time.time() - start_tuning

print("\nBest CatBoost F1:", best_cat_f1)
print("Best CatBoost params:", best_cat_params)
print("Total tuning time:", cat_tuning_time)

cat_search_df = pd.DataFrame(cat_results)
cat_search_df.sort_values("F1-score", ascending=False).head()

In [ ]:
y_pred_cat_tuned = best_cat_model.predict(X_test)

cat_tuned_result = evaluate_model(
    "CatBoost Tuned",
    y_test,
    y_pred_cat_tuned,
    cat_tuning_time
)

cat_tuned_df = pd.DataFrame([cat_tuned_result])
cat_tuned_df.round(4)

In [ ]:
RESULTS_DIR = "/kaggle/working/results"
MODEL_DIR = "/kaggle/working/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

cat_tuned_result_path = os.path.join(RESULTS_DIR, "catboost_tuned_result.csv")
cat_tuned_model_path = os.path.join(MODEL_DIR, "catboost_tuned_model.cbm")
cat_search_log_path = os.path.join(RESULTS_DIR, "catboost_tuning_search_log.csv")
cat_best_params_path = os.path.join(RESULTS_DIR, "catboost_tuned_best_params.csv")

cat_tuned_df.to_csv(cat_tuned_result_path, index=False)

best_cat_model.save_model(cat_tuned_model_path)

cat_search_df.to_csv(cat_search_log_path, index=False)

cat_best_params_df = pd.DataFrame([best_cat_params])
cat_best_params_df["Best Test F1"] = best_cat_f1
cat_best_params_df["Tuning Time (sec)"] = cat_tuning_time
cat_best_params_df.to_csv(cat_best_params_path, index=False)

print("Saved result:", cat_tuned_result_path)
print("Saved model:", cat_tuned_model_path)
print("Saved search log:", cat_search_log_path)
print("Saved best params:", cat_best_params_path)

cat_tuned_df.round(4)

In [ ]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

cat_tuned_threshold_df = threshold_tuning_model(
    best_cat_model,
    "CatBoost Tuned",
    X_test,
    y_test,
    thresholds
)

cat_tuned_threshold_df.round(4)

In [ ]:
cat_threshold_path = os.path.join(
    RESULTS_DIR,
    "catboost_tuned_threshold_results.csv"
)

cat_tuned_threshold_df.to_csv(cat_threshold_path, index=False)

print("Saved:", cat_threshold_path)
cat_tuned_threshold_df.round(4)

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import f1_score

RESULTS_DIR = "/kaggle/working/results"
MODEL_DIR = "/kaggle/working/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

mlp_param_list = [
    {"hidden_layers": [128, 64], "dropout": 0.2, "learning_rate": 0.001, "batch_size": 512},
    {"hidden_layers": [256, 128], "dropout": 0.3, "learning_rate": 0.001, "batch_size": 512},
    {"hidden_layers": [256, 128, 64], "dropout": 0.3, "learning_rate": 0.001, "batch_size": 512},
    {"hidden_layers": [512, 256, 128], "dropout": 0.4, "learning_rate": 0.0005, "batch_size": 512},
    {"hidden_layers": [256, 128, 64], "dropout": 0.2, "learning_rate": 0.0005, "batch_size": 1024},
    {"hidden_layers": [512, 256], "dropout": 0.3, "learning_rate": 0.0005, "batch_size": 1024},
]

def build_mlp(input_dim, hidden_layers, dropout, learning_rate):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))

    for units in hidden_layers:
        model.add(Dense(units, activation="relu"))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


mlp_tuning_results = []

best_mlp_model = None
best_mlp_f1 = -1
best_mlp_params = None
best_mlp_training_time = None

start_total = time.time()

for i, params in enumerate(mlp_param_list):
    print(f"\nMLP tuning run {i+1}/{len(mlp_param_list)}")
    print(params)

    model_path = os.path.join(MODEL_DIR, f"mlp_tuned_candidate_{i+1}.keras")

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=model_path,
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=0
        )
    ]

    mlp_model = build_mlp(
        input_dim=X_train_scaled.shape[1],
        hidden_layers=params["hidden_layers"],
        dropout=params["dropout"],
        learning_rate=params["learning_rate"]
    )

    start_run = time.time()

    history = mlp_model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.2,
        epochs=30,
        batch_size=params["batch_size"],
        callbacks=callbacks,
        verbose=1
    )

    run_time = time.time() - start_run

    y_prob = mlp_model.predict(X_test_scaled, batch_size=2048).reshape(-1)
    y_pred = (y_prob >= 0.50).astype(int)

    f1 = f1_score(y_test, y_pred)

    mlp_tuning_results.append({
        "Run": i + 1,
        "Hidden Layers": str(params["hidden_layers"]),
        "Dropout": params["dropout"],
        "Learning Rate": params["learning_rate"],
        "Batch Size": params["batch_size"],
        "F1-score": f1,
        "Run Time (sec)": run_time
    })

    print("F1:", f1)
    print("Run time:", run_time)

    if f1 > best_mlp_f1:
        best_mlp_f1 = f1
        best_mlp_model = mlp_model
        best_mlp_params = params
        best_mlp_training_time = run_time
        print("New best MLP found.")

mlp_total_tuning_time = time.time() - start_total

mlp_search_df = pd.DataFrame(mlp_tuning_results)
mlp_search_df.sort_values("F1-score", ascending=False)

In [ ]:
y_prob_mlp_tuned = best_mlp_model.predict(X_test_scaled, batch_size=2048).reshape(-1)

mlp_tuned_result = evaluate_dl_model(
    "MLP Tuned",
    y_test,
    y_prob_mlp_tuned,
    mlp_total_tuning_time,
    threshold=0.50
)

mlp_tuned_df = pd.DataFrame([mlp_tuned_result])
mlp_tuned_df.round(4)

In [ ]:
mlp_tuned_result_path = os.path.join(RESULTS_DIR, "mlp_tuned_result.csv")
mlp_tuned_model_path = os.path.join(MODEL_DIR, "mlp_tuned_model.keras")
mlp_search_log_path = os.path.join(RESULTS_DIR, "mlp_tuning_search_log.csv")
mlp_best_params_path = os.path.join(RESULTS_DIR, "mlp_tuned_best_params.csv")

mlp_tuned_df.to_csv(mlp_tuned_result_path, index=False)
best_mlp_model.save(mlp_tuned_model_path)
mlp_search_df.to_csv(mlp_search_log_path, index=False)

mlp_best_params_df = pd.DataFrame([best_mlp_params])
mlp_best_params_df["Best Test F1"] = best_mlp_f1
mlp_best_params_df["Total Tuning Time (sec)"] = mlp_total_tuning_time
mlp_best_params_df.to_csv(mlp_best_params_path, index=False)

print("Saved result:", mlp_tuned_result_path)
print("Saved model:", mlp_tuned_model_path)
print("Saved search log:", mlp_search_log_path)
print("Saved best params:", mlp_best_params_path)

mlp_tuned_df.round(4)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import f1_score
import time
import pandas as pd
import os

cnn_param_list = [
    {"filters": [64, 128], "kernel_size": 3, "dropout": 0.3, "learning_rate": 0.001, "batch_size": 512},
    {"filters": [128, 128], "kernel_size": 3, "dropout": 0.3, "learning_rate": 0.001, "batch_size": 512},
    {"filters": [128, 256], "kernel_size": 3, "dropout": 0.4, "learning_rate": 0.0005, "batch_size": 512},
    {"filters": [64, 128, 256], "kernel_size": 3, "dropout": 0.4, "learning_rate": 0.0005, "batch_size": 512},
    {"filters": [128, 256], "kernel_size": 5, "dropout": 0.3, "learning_rate": 0.0005, "batch_size": 512},
]

def build_cnn(input_shape, filters, kernel_size, dropout, learning_rate):
    model = Sequential()
    model.add(Input(shape=input_shape))

    for f in filters:
        model.add(Conv1D(filters=f, kernel_size=kernel_size, activation="relu", padding="same"))
        model.add(BatchNormalization())
        model.add(MaxPooling1D(pool_size=2))
        model.add(Dropout(dropout))

    model.add(GlobalAveragePooling1D())

    model.add(Dense(128, activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(dropout))

    model.add(Dense(64, activation="relu"))
    model.add(Dropout(dropout / 2))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


cnn_tuning_results = []

best_cnn_model = None
best_cnn_f1 = -1
best_cnn_params = None
best_cnn_training_time = None

start_total = time.time()

for i, params in enumerate(cnn_param_list):
    print(f"\nCNN tuning run {i+1}/{len(cnn_param_list)}")
    print(params)

    model_path = os.path.join(MODEL_DIR, f"cnn_tuned_candidate_{i+1}.keras")

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=model_path,
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=0
        )
    ]

    cnn_model = build_cnn(
        input_shape=(X_train_transformer.shape[1], 1),
        filters=params["filters"],
        kernel_size=params["kernel_size"],
        dropout=params["dropout"],
        learning_rate=params["learning_rate"]
    )

    start_run = time.time()

    history = cnn_model.fit(
        X_train_transformer,
        y_train,
        validation_split=0.2,
        epochs=30,
        batch_size=params["batch_size"],
        callbacks=callbacks,
        verbose=1
    )

    run_time = time.time() - start_run

    y_prob = cnn_model.predict(X_test_transformer, batch_size=2048).reshape(-1)
    y_pred = (y_prob >= 0.50).astype(int)

    f1 = f1_score(y_test, y_pred)

    cnn_tuning_results.append({
        "Run": i + 1,
        "Filters": str(params["filters"]),
        "Kernel Size": params["kernel_size"],
        "Dropout": params["dropout"],
        "Learning Rate": params["learning_rate"],
        "Batch Size": params["batch_size"],
        "F1-score": f1,
        "Run Time (sec)": run_time
    })

    print("F1:", f1)
    print("Run time:", run_time)

    if f1 > best_cnn_f1:
        best_cnn_f1 = f1
        best_cnn_model = cnn_model
        best_cnn_params = params
        best_cnn_training_time = run_time
        print("New best CNN found.")

cnn_total_tuning_time = time.time() - start_total

cnn_search_df = pd.DataFrame(cnn_tuning_results)
cnn_search_df.sort_values("F1-score", ascending=False)

In [ ]:
y_prob_cnn_tuned = best_cnn_model.predict(X_test_transformer, batch_size=2048).reshape(-1)

cnn_tuned_result = evaluate_dl_model(
    "1D-CNN Tuned",
    y_test,
    y_prob_cnn_tuned,
    cnn_total_tuning_time,
    threshold=0.50
)

cnn_tuned_df = pd.DataFrame([cnn_tuned_result])
cnn_tuned_df.round(4)

In [ ]:
cnn_tuned_result_path = os.path.join(RESULTS_DIR, "cnn_tuned_result.csv")
cnn_tuned_model_path = os.path.join(MODEL_DIR, "cnn_tuned_model.keras")
cnn_search_log_path = os.path.join(RESULTS_DIR, "cnn_tuning_search_log.csv")
cnn_best_params_path = os.path.join(RESULTS_DIR, "cnn_tuned_best_params.csv")

cnn_tuned_df.to_csv(cnn_tuned_result_path, index=False)
best_cnn_model.save(cnn_tuned_model_path)
cnn_search_df.to_csv(cnn_search_log_path, index=False)

cnn_best_params_df = pd.DataFrame([best_cnn_params])
cnn_best_params_df["Best Test F1"] = best_cnn_f1
cnn_best_params_df["Total Tuning Time (sec)"] = cnn_total_tuning_time
cnn_best_params_df.to_csv(cnn_best_params_path, index=False)

print("Saved result:", cnn_tuned_result_path)
print("Saved model:", cnn_tuned_model_path)
print("Saved search log:", cnn_search_log_path)
print("Saved best params:", cnn_best_params_path)

cnn_tuned_df.round(4)

In [ ]:
import os
import pandas as pd

RESULTS_DIR = "/kaggle/working/results"

tuned_result_files = [
    "lightgbm_tuned_result.csv",
    "xgboost_tuned_result.csv",
    "catboost_tuned_result.csv",
    "mlp_tuned_result.csv",
    "cnn_tuned_result.csv"
]

tuned_dfs = []

for file in tuned_result_files:
    path = os.path.join(RESULTS_DIR, file)
    if os.path.exists(path):
        df_temp = pd.read_csv(path)
        df_temp["Source File"] = file
        tuned_dfs.append(df_temp)
        print("Loaded:", path)
    else:
        print("Missing:", path)

top5_tuned_df = pd.concat(tuned_dfs, ignore_index=True)

top5_tuned_df.columns = top5_tuned_df.columns.str.strip()

top5_tuned_df = top5_tuned_df.sort_values(
    "F1-score",
    ascending=False
).reset_index(drop=True)

top5_tuned_path = os.path.join(
    RESULTS_DIR,
    "top5_tuned_model_comparison.csv"
)

top5_tuned_df.to_csv(top5_tuned_path, index=False)

print("Saved:", top5_tuned_path)

top5_tuned_df[
    ["Model", "Accuracy", "Precision", "Recall", "F1-score", "FPR", "FNR", "TP", "TN", "FP", "FN", "Training Time (sec)"]
].round(4)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = "/kaggle/working/results"
MODEL_DIR = "/kaggle/working/models"
FIGURES_DIR = "/kaggle/working/figures"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Results:", RESULTS_DIR)
print("Models:", MODEL_DIR)
print("Figures:", FIGURES_DIR)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = "/kaggle/working/results"
FIGURES_DIR = "/kaggle/working/figures"

os.makedirs(FIGURES_DIR, exist_ok=True)

baseline_path = "/kaggle/working/results/final_ablation_model_comparison.csv"

baseline_df = pd.read_csv(baseline_path)
baseline_df.columns = baseline_df.columns.str.strip()

# Sort by F1-score for clearer horizontal chart
plot_df = baseline_df.sort_values("F1-score", ascending=True)

plt.figure(figsize=(11, 8))
plt.barh(plot_df["Model"], plot_df["F1-score"])

plt.xlabel("F1-score")
plt.ylabel("Model")
plt.title("F1-score Comparison of All Baseline Models")
plt.xlim(0, 1)

for i, value in enumerate(plot_df["F1-score"]):
    plt.text(value + 0.006, i, f"{value:.4f}", va="center", fontsize=8)

plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "all_baseline_models_f1_comparison.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]

plot_df = baseline_df.sort_values("F1-score", ascending=False)

plt.figure(figsize=(14, 7))

x = range(len(plot_df))
width = 0.2

for idx, metric in enumerate(metrics):
    positions = [p + idx * width for p in x]
    plt.bar(positions, plot_df[metric], width=width, label=metric)

plt.xticks(
    [p + width * 1.5 for p in x],
    plot_df["Model"],
    rotation=45,
    ha="right"
)

plt.ylabel("Score")
plt.title("Performance Comparison of All Baseline Models")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "all_baseline_models_metrics_comparison.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)

In [ ]:
top5_path = "/kaggle/working/results/top5_tuned_model_comparison.csv"

top5_df = pd.read_csv(top5_path)
top5_df.columns = top5_df.columns.str.strip()

top5_plot_df = top5_df.sort_values("F1-score", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(top5_plot_df["Model"], top5_plot_df["F1-score"])
plt.xlabel("F1-score")
plt.ylabel("Model")
plt.title("Top-5 Tuned Model F1-score Comparison")
plt.xlim(0.88, 0.94)

for index, value in enumerate(top5_plot_df["F1-score"]):
    plt.text(value + 0.001, index, f"{value:.4f}", va="center", fontsize=9)

plt.tight_layout()

top5_fig_path = os.path.join(FIGURES_DIR, "top5_tuned_model_f1_comparison.png")
plt.savefig(top5_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", top5_fig_path)

In [ ]:
lgbm_threshold_path = "/kaggle/working/results/lightgbm_tuned_threshold_results.csv"

lgbm_threshold_df = pd.read_csv(lgbm_threshold_path)
lgbm_threshold_df.columns = lgbm_threshold_df.columns.str.strip()

plt.figure(figsize=(8, 5))
plt.plot(
    lgbm_threshold_df["Threshold"],
    lgbm_threshold_df["Recall"],
    marker="o",
    label="Recall"
)
plt.plot(
    lgbm_threshold_df["Threshold"],
    lgbm_threshold_df["FNR"],
    marker="o",
    label="FNR"
)

plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("LightGBM Tuned: Threshold vs Recall and FNR")
plt.gca().invert_xaxis()
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()

threshold_fig_path = os.path.join(FIGURES_DIR, "lightgbm_threshold_vs_recall_fnr.png")
plt.savefig(threshold_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", threshold_fig_path)

In [ ]:
selected_lgbm_035 = lgbm_threshold_df[lgbm_threshold_df["Threshold"] == 0.35].iloc[0]

tn = int(selected_lgbm_035["TN"])
fp = int(selected_lgbm_035["FP"])
fn = int(selected_lgbm_035["FN"])
tp = int(selected_lgbm_035["TP"])

cm = np.array([[tn, fp],
               [fn, tp]])

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix: LightGBM Tuned at Threshold 0.35")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks([0, 1], ["Benign", "Attack"])
plt.yticks([0, 1], ["Benign", "Attack"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13)

plt.colorbar()
plt.tight_layout()

cm_fig_path = os.path.join(FIGURES_DIR, "lightgbm_threshold_035_confusion_matrix.png")
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", cm_fig_path)

In [ ]:
import shap

lightgbm_model_path = "/kaggle/working/models/lightgbm_tuned_model.pkl"

best_lgbm_model = joblib.load(lightgbm_model_path)

print("Loaded:", lightgbm_model_path)

In [ ]:
# Use a sample for fast SHAP computation
sample_size = 5000

if len(X_test) > sample_size:
    X_shap = X_test.sample(n=sample_size, random_state=42)
else:
    X_shap = X_test.copy()

print("SHAP sample shape:", X_shap.shape)

explainer = shap.TreeExplainer(best_lgbm_model)
shap_values = explainer.shap_values(X_shap)

# For binary classification, shap_values may be a list
if isinstance(shap_values, list):
    shap_values_to_use = shap_values[1]
else:
    shap_values_to_use = shap_values

mean_abs_shap = np.abs(shap_values_to_use).mean(axis=0)

shap_importance_df = pd.DataFrame({
    "Feature": X_shap.columns,
    "Mean |SHAP value|": mean_abs_shap
}).sort_values("Mean |SHAP value|", ascending=False)

shap_importance_path = os.path.join(
    RESULTS_DIR,
    "lightgbm_tuned_shap_top_20_features.csv"
)

shap_importance_df.head(20).to_csv(shap_importance_path, index=False)

shap_importance_df.head(20)

In [ ]:
top20_shap = shap_importance_df.head(20).sort_values("Mean |SHAP value|", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(top20_shap["Feature"], top20_shap["Mean |SHAP value|"])
plt.xlabel("Mean |SHAP value|")
plt.ylabel("Feature")
plt.title("Top-20 SHAP Feature Importance for Tuned LightGBM")

for index, value in enumerate(top20_shap["Mean |SHAP value|"]):
    plt.text(value + 0.002, index, f"{value:.4f}", va="center", fontsize=8)

plt.tight_layout()

shap_bar_fig_path = os.path.join(FIGURES_DIR, "lightgbm_tuned_shap_top_20_features.png")
plt.savefig(shap_bar_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", shap_bar_fig_path)
print("Saved SHAP CSV:", shap_importance_path)

In [ ]:
print("Generated figures:")

for file in os.listdir(FIGURES_DIR):
    print(os.path.join(FIGURES_DIR, file))

print("\nGenerated result files:")

for file in os.listdir(RESULTS_DIR):
    if "shap" in file.lower() or "threshold" in file.lower() or "tuned" in file.lower():
        print(os.path.join(RESULTS_DIR, file))

In [ ]:
import shap
import os
import matplotlib.pyplot as plt

FIGURES_DIR = "/kaggle/working/figures"

plt.figure()

shap.summary_plot(
    shap_values_to_use,
    X_shap,
    max_display=20,
    show=False
)

shap_summary_fig_path = os.path.join(
    FIGURES_DIR,
    "lightgbm_tuned_shap_summary_plot.png"
)

plt.savefig(shap_summary_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", shap_summary_fig_path)

In [ ]:
import os
import zipfile

OUTPUT_ZIP = "/kaggle/working/final_research_outputs.zip"

folders_to_zip = [
    "/kaggle/working/results",
    "/kaggle/working/figures",
    "/kaggle/working/models"
]

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, dirs, files in os.walk(folder):
                for file in files:
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, "/kaggle/working")
                    zipf.write(full_path, arcname=arcname)
                    print("Added:", arcname)

print("Created:", OUTPUT_ZIP)

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Final untouched test set: 20%
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split the remaining 80% into 64% training and 16% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

print("\nClass distributions:")
print("Train:\n", y_train.value_counts())
print("Validation:\n", y_val.value_counts())
print("Test:\n", y_test.value_counts())

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# 1. Load the processed dataset
# --------------------------------------------------
DATA_PATH = (
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/ids2018-balanced-binary-dataset/"
    "merged_balanced_ids2018_safe.csv"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at:\n{DATA_PATH}\n"
        "Check that the Kaggle dataset is attached to the notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df.shape)
print("Columns:", len(df.columns))

# --------------------------------------------------
# 2. Separate target and features
# --------------------------------------------------
target_col = "binary_label"

if target_col not in df.columns:
    raise KeyError(f"Target column '{target_col}' was not found.")

y = df[target_col].astype(int)
X = df.drop(columns=[target_col])

# Remove labels, identifiers, and time-related columns
possible_bad_cols = [
    "Label",
    "Timestamp",
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Src Port",
    "Unnamed: 0",
]

columns_to_drop = [
    column for column in possible_bad_cols
    if column in X.columns
]

X = X.drop(columns=columns_to_drop)

print("Dropped columns:", columns_to_drop)

# Convert features to numeric and remove invalid rows
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)

valid_rows = X.notna().all(axis=1)

X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

print("\nFinal feature shape:", X.shape)
print("Final target shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())

# --------------------------------------------------
# 3. Create untouched 20% test set
# --------------------------------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# --------------------------------------------------
# 4. Divide remaining 80% into 64% train and 16% validation
# --------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full,
)

print("\nDataset partitions:")
print("Training:  ", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:      ", X_test.shape, y_test.shape)

print("\nTraining distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation distribution:")
print(y_val.value_counts().sort_index())

print("\nTest distribution:")
print(y_test.value_counts().sort_index())

# --------------------------------------------------
# 5. Scaling for models that require standardisation
# --------------------------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 3D forms for CNN, LSTM, and Transformer models
X_train_dl = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1,
)

X_val_dl = X_val_scaled.reshape(
    X_val_scaled.shape[0],
    X_val_scaled.shape[1],
    1,
)

X_test_dl = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1,
)

print("\nScaled arrays:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:  ", X_val_scaled.shape)
print("X_test_scaled: ", X_test_scaled.shape)

print("\nDeep-learning arrays:")
print("X_train_dl:", X_train_dl.shape)
print("X_val_dl:  ", X_val_dl.shape)
print("X_test_dl: ", X_test_dl.shape)

In [ ]:
import os
import joblib
import pandas as pd

os.makedirs("/kaggle/working/splits", exist_ok=True)
os.makedirs("/kaggle/working/preprocessing", exist_ok=True)

split_manifest = pd.concat(
    [
        pd.DataFrame({
            "row_index": X_train.index,
            "split": "train"
        }),
        pd.DataFrame({
            "row_index": X_val.index,
            "split": "validation"
        }),
        pd.DataFrame({
            "row_index": X_test.index,
            "split": "test"
        }),
    ],
    ignore_index=True,
)

split_manifest.to_csv(
    "/kaggle/working/splits/final_split_manifest.csv",
    index=False,
)

joblib.dump(
    scaler,
    "/kaggle/working/preprocessing/final_standard_scaler.pkl",
)

print(split_manifest["split"].value_counts())
print("Split manifest and scaler saved.")

In [ ]:
# ============================================================
# STAGE 1A: VALIDATION-BASED BASELINE ABLATION
# Classical ML, tree ensembles, and boosting models
# ============================================================

import os
import gc
import time
import json
import random
import warnings
import platform

import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Output directories
# ------------------------------------------------------------

REVISION_ROOT = "/kaggle/working/revised_experiments"
RESULTS_DIR = os.path.join(REVISION_ROOT, "results")
MODELS_DIR = os.path.join(REVISION_ROOT, "models")
METADATA_DIR = os.path.join(REVISION_ROOT, "metadata")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)


# ------------------------------------------------------------
# 3. Confirm required variables exist
# ------------------------------------------------------------

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
    "X_train_scaled",
    "X_val_scaled",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Run the dataset preparation and 64/16/20 splitting cell first. "
        f"Missing variables: {missing_variables}"
    )

print("Training shape:  ", X_train.shape)
print("Validation shape:", X_val.shape)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert X_train.shape[1] == X_val.shape[1]

print("\nNo test data will be used in this stage.")


# ------------------------------------------------------------
# 4. Evaluation function
# ------------------------------------------------------------

def evaluate_binary_predictions(
    model_name,
    y_true,
    y_pred,
    training_time,
    prediction_time,
):
    """
    Calculate validation metrics for binary intrusion detection.
    Attack traffic is treated as the positive class.
    """

    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    y_true = np.asarray(y_true).astype(int).reshape(-1)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0

    return {
        "Model": model_name,
        "Evaluation Split": "Validation",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Training Time (sec)": training_time,
        "Prediction Time (sec)": prediction_time,
    }


# ------------------------------------------------------------
# 5. Baseline model definitions
# ------------------------------------------------------------

# input_type:
# "scaled" -> models sensitive to feature scale
# "raw"    -> tree and boosting models

baseline_models = [
    {
        "name": "Logistic Regression",
        "model": LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        ),
        "input_type": "scaled",
    },
    {
        "name": "Naive Bayes",
        "model": GaussianNB(),
        "input_type": "scaled",
    },
    {
        "name": "KNN",
        "model": KNeighborsClassifier(
            n_neighbors=5,
            weights="uniform",
            metric="minkowski",
            p=2,
            n_jobs=-1,
        ),
        "input_type": "scaled",
    },
    {
        "name": "Linear SVM",
        "model": LinearSVC(
            C=1.0,
            max_iter=10000,
            random_state=RANDOM_STATE,
        ),
        "input_type": "scaled",
    },
    {
        "name": "Decision Tree",
        "model": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
    },
    {
        "name": "Random Forest",
        "model": RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input_type": "raw",
    },
    {
        "name": "Extra Trees",
        "model": ExtraTreesClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input_type": "raw",
    },
    {
        "name": "AdaBoost",
        "model": AdaBoostClassifier(
            n_estimators=50,
            learning_rate=1.0,
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
    },
    {
        "name": "Gradient Boosting",
        "model": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
    },
    {
        "name": "XGBoost",
        "model": XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.3,
            subsample=1.0,
            colsample_bytree=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        ),
        "input_type": "raw",
    },
    {
        "name": "LightGBM",
        "model": LGBMClassifier(
            n_estimators=100,
            learning_rate=0.1,
            num_leaves=31,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        ),
        "input_type": "raw",
    },
    {
        "name": "CatBoost",
        "model": CatBoostClassifier(
            iterations=1000,
            learning_rate=0.03,
            depth=6,
            loss_function="Logloss",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        ),
        "input_type": "raw",
    },
]


# ------------------------------------------------------------
# 6. Run baseline models
# ------------------------------------------------------------

baseline_results = []
completed_models = []

for position, config in enumerate(baseline_models, start=1):

    model_name = config["name"]
    model = config["model"]
    input_type = config["input_type"]

    print("\n" + "=" * 72)
    print(
        f"Running model {position}/{len(baseline_models)}: "
        f"{model_name}"
    )
    print("=" * 72)

    if input_type == "scaled":
        train_features = X_train_scaled
        validation_features = X_val_scaled
    else:
        train_features = X_train
        validation_features = X_val

    # Training
    training_start = time.perf_counter()

    model.fit(
        train_features,
        np.asarray(y_train).reshape(-1),
    )

    training_time = time.perf_counter() - training_start

    # Validation prediction
    prediction_start = time.perf_counter()

    validation_predictions = model.predict(
        validation_features
    )

    prediction_time = time.perf_counter() - prediction_start

    result = evaluate_binary_predictions(
        model_name=model_name,
        y_true=y_val,
        y_pred=validation_predictions,
        training_time=training_time,
        prediction_time=prediction_time,
    )

    baseline_results.append(result)
    completed_models.append(model_name)

    # Save each fitted baseline model
    safe_model_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_path = os.path.join(
        MODELS_DIR,
        f"baseline_{safe_model_name}.pkl",
    )

    joblib.dump(model, model_path)

    # Save incremental results after every model
    interim_df = pd.DataFrame(baseline_results)
    interim_df = interim_df.sort_values(
        "F1-score",
        ascending=False,
    ).reset_index(drop=True)

    interim_path = os.path.join(
        RESULTS_DIR,
        "validation_baseline_tabular_models_partial.csv",
    )

    interim_df.to_csv(interim_path, index=False)

    print(
        f"Accuracy : {result['Accuracy']:.4f}\n"
        f"Precision: {result['Precision']:.4f}\n"
        f"Recall   : {result['Recall']:.4f}\n"
        f"F1-score : {result['F1-score']:.4f}\n"
        f"FPR      : {result['FPR']:.4f}\n"
        f"FNR      : {result['FNR']:.4f}\n"
        f"Training : {training_time:.2f} sec\n"
        f"Prediction: {prediction_time:.2f} sec"
    )

    del validation_predictions
    gc.collect()


# ------------------------------------------------------------
# 7. Save final 12-model validation table
# ------------------------------------------------------------

validation_baseline_tabular_df = pd.DataFrame(
    baseline_results
)

validation_baseline_tabular_df = (
    validation_baseline_tabular_df
    .sort_values(
        ["F1-score", "Recall"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

final_results_path = os.path.join(
    RESULTS_DIR,
    "validation_baseline_tabular_models.csv",
)

validation_baseline_tabular_df.to_csv(
    final_results_path,
    index=False,
)


# ------------------------------------------------------------
# 8. Save experimental metadata
# ------------------------------------------------------------

metadata = {
    "random_state": RANDOM_STATE,
    "total_records": int(
        len(X_train) + len(X_val) + len(X_test)
    ),
    "training_records": int(len(X_train)),
    "validation_records": int(len(X_val)),
    "test_records": int(len(X_test)),
    "feature_count": int(X_train.shape[1]),
    "training_benign": int((y_train == 0).sum()),
    "training_attack": int((y_train == 1).sum()),
    "validation_benign": int((y_val == 0).sum()),
    "validation_attack": int((y_val == 1).sum()),
    "test_benign": int((y_test == 0).sum()),
    "test_attack": int((y_test == 1).sum()),
    "models_completed": completed_models,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
}

metadata_path = os.path.join(
    METADATA_DIR,
    "validation_baseline_metadata.json",
)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

display_columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "FPR",
    "FNR",
    "TP",
    "TN",
    "FP",
    "FN",
    "Training Time (sec)",
    "Prediction Time (sec)",
]

print("\n" + "=" * 72)
print("VALIDATION-BASED TABULAR BASELINE RESULTS")
print("=" * 72)

display(
    validation_baseline_tabular_df[
        display_columns
    ].round(4)
)

print("\nSaved results:")
print(final_results_path)

print("\nSaved metadata:")
print(metadata_path)

print("\nThe test set has not been evaluated.")

In [ ]:
import subprocess
import tensorflow as tf

print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
).stdout)

print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================================================
# GPU-AWARE BASELINE MODEL CONFIGURATION
# ============================================================

import os
import random
import warnings
import numpy as np
import xgboost
import lightgbm
import catboost

from packaging.version import Version

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count


RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Package and GPU information
# ------------------------------------------------------------

print("XGBoost version:", xgboost.__version__)
print("LightGBM version:", lightgbm.__version__)
print("CatBoost version:", catboost.__version__)
print("CatBoost GPU count:", get_gpu_device_count())


# ------------------------------------------------------------
# XGBoost GPU parameters
# ------------------------------------------------------------

xgb_params = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

if Version(xgboost.__version__) >= Version("2.0.0"):
    xgb_params.update({
        "tree_method": "hist",
        "device": "cuda",
    })
else:
    xgb_params.update({
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
    })

print("XGBoost configured for GPU.")


# ------------------------------------------------------------
# Probe LightGBM GPU support
# ------------------------------------------------------------

LIGHTGBM_DEVICE = "cpu"

for candidate_device in ["cuda", "gpu"]:
    try:
        probe = LGBMClassifier(
            n_estimators=5,
            device_type=candidate_device,
            random_state=RANDOM_STATE,
            verbosity=-1,
        )

        probe.fit(
            X_train.iloc[:5000],
            y_train.iloc[:5000],
        )

        LIGHTGBM_DEVICE = candidate_device
        print(
            f"LightGBM GPU support available: "
            f"device_type='{candidate_device}'"
        )
        break

    except Exception as error:
        print(
            f"LightGBM device '{candidate_device}' "
            f"is unavailable: {str(error)[:160]}"
        )

if LIGHTGBM_DEVICE == "cpu":
    print("LightGBM will run on CPU.")

In [ ]:
baseline_models = [
    # --------------------------------------------------------
    # CPU: standard scikit-learn models
    # --------------------------------------------------------

    {
        "name": "Logistic Regression",
        "model": LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        ),
        "input_type": "scaled",
        "device": "GPU",
    },

    {
        "name": "Naive Bayes",
        "model": GaussianNB(),
        "input_type": "scaled",
        "device": "GPU",
    },

    {
        "name": "KNN",
        "model": KNeighborsClassifier(
            n_neighbors=5,
            weights="uniform",
            metric="minkowski",
            p=2,
            n_jobs=-1,
        ),
        "input_type": "scaled",
        "device": "GPU",
    },

    {
        "name": "Linear SVM",
        "model": LinearSVC(
            C=1.0,
            dual=False,
            tol=1e-3,
            max_iter=3000,
            random_state=RANDOM_STATE,
        ),
        "input_type": "scaled",
        "device": "GPU",
    },

    {
        "name": "Decision Tree",
        "model": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
        "device": "GPU",
    },

    {
        "name": "Random Forest",
        "model": RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input_type": "raw",
        "device": "GPU",
    },

    {
        "name": "Extra Trees",
        "model": ExtraTreesClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input_type": "raw",
        "device": "GPU",
    },

    {
        "name": "AdaBoost",
        "model": AdaBoostClassifier(
            n_estimators=50,
            learning_rate=1.0,
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
        "device": "GPU",
    },

    {
        "name": "Gradient Boosting",
        "model": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
        "device": "GPU",
    },

    # --------------------------------------------------------
    # GPU: XGBoost
    # --------------------------------------------------------

    {
        "name": "XGBoost",
        "model": XGBClassifier(**xgb_params),
        "input_type": "raw",
        "device": "GPU",
    },

    # --------------------------------------------------------
    # GPU when supported; otherwise CPU
    # --------------------------------------------------------

    {
        "name": "LightGBM",
        "model": LGBMClassifier(
            n_estimators=100,
            learning_rate=0.1,
            num_leaves=31,
            device_type=LIGHTGBM_DEVICE,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        ),
        "input_type": "raw",
        "device": (
            "GPU"
            if LIGHTGBM_DEVICE in ["cuda", "gpu"]
            else "CPU"
        ),
    },

    # --------------------------------------------------------
    # GPU: CatBoost
    # --------------------------------------------------------

    {
        "name": "CatBoost",
        "model": CatBoostClassifier(
            iterations=1000,
            learning_rate=0.03,
            depth=6,
            loss_function="Logloss",
            random_seed=RANDOM_STATE,
            task_type="GPU",
            devices="0",
            verbose=False,
            allow_writing_files=False,
        ),
        "input_type": "raw",
        "device": "GPU",
    },
]

for config in baseline_models:
    print(
        f"{config['name']:<24} "
        f"→ {config['device']}"
    )

In [ ]:
# ============================================================
# CLEAN 12-MODEL BASELINE ABLATION
# Train: 64%
# Validation: 16%
# Test set: NOT USED
#
# CPU:
#   Logistic Regression, Naive Bayes, KNN, Linear SVM,
#   Decision Tree, Random Forest, Extra Trees,
#   AdaBoost, Gradient Boosting
#
# GPU:
#   XGBoost, CatBoost
#   LightGBM when the installed build supports GPU
# ============================================================

import os
import gc
import json
import time
import random
import warnings
import platform
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
import lightgbm
import catboost

from IPython.display import display
from packaging.version import Version

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ============================================================
# 2. Confirm that the clean split exists
# ============================================================

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
    "X_train_scaled",
    "X_val_scaled",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        "Run the dataset loading, 64/16/20 split, and scaling cell first.\n"
        f"Missing variables: {missing_variables}"
    )

assert X_train.shape == (192593, 78), X_train.shape
assert X_val.shape == (48149, 78), X_val.shape
assert len(y_train) == 192593
assert len(y_val) == 48149

print("Training shape:  ", X_train.shape)
print("Validation shape:", X_val.shape)
print("\nIMPORTANT: The test set is not used in this cell.")


# ============================================================
# 3. Prepare efficient arrays
# ============================================================

# Raw features for trees and boosting
X_train_raw = np.asarray(X_train, dtype=np.float32)
X_val_raw = np.asarray(X_val, dtype=np.float32)

# Standardised features for scale-sensitive models
X_train_std = np.asarray(X_train_scaled, dtype=np.float32)
X_val_std = np.asarray(X_val_scaled, dtype=np.float32)

y_train_array = np.asarray(y_train, dtype=np.int32).reshape(-1)
y_val_array = np.asarray(y_val, dtype=np.int32).reshape(-1)

print("\nPrepared arrays:")
print("X_train_raw:", X_train_raw.shape, X_train_raw.dtype)
print("X_val_raw:  ", X_val_raw.shape, X_val_raw.dtype)
print("X_train_std:", X_train_std.shape, X_train_std.dtype)
print("X_val_std:  ", X_val_std.shape, X_val_std.dtype)


# ============================================================
# 4. Create fresh output directories
# ============================================================

RUN_NAME = "baseline12_validation_clean_v1"

RUN_ROOT = Path("/kaggle/working/revised_experiments") / RUN_NAME
RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
METADATA_DIR = RUN_ROOT / "metadata"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR / "baseline12_validation_partial.csv"
)

FINAL_RESULTS_PATH = (
    RESULTS_DIR / "baseline12_validation_results.csv"
)

ERROR_LOG_PATH = (
    RESULTS_DIR / "baseline12_errors.csv"
)

print("\nOutput directory:")
print(RUN_ROOT)


# ============================================================
# 5. Detect the Kaggle GPU
# ============================================================

gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)

GPU_AVAILABLE = (
    gpu_query.returncode == 0
    and bool(gpu_query.stdout.strip())
)

print("\nGPU status:")

if GPU_AVAILABLE:
    print(gpu_query.stdout.strip())
else:
    print("No NVIDIA GPU was detected.")

CATBOOST_GPU_COUNT = get_gpu_device_count()

print("\nLibrary versions:")
print("scikit-learn:", sklearn.__version__)
print("XGBoost:     ", xgboost.__version__)
print("LightGBM:    ", lightgbm.__version__)
print("CatBoost:    ", catboost.__version__)
print("CatBoost GPU count:", CATBOOST_GPU_COUNT)


# ============================================================
# 6. Model-construction helpers
# ============================================================

def make_xgboost(use_gpu=True):
    """
    Build XGBoost using the correct GPU syntax for the
    installed XGBoost version.
    """

    parameters = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.3,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }

    if use_gpu and GPU_AVAILABLE:
        if Version(xgboost.__version__) >= Version("2.0.0"):
            parameters.update({
                "tree_method": "hist",
                "device": "cuda",
            })
        else:
            parameters.update({
                "tree_method": "gpu_hist",
                "predictor": "gpu_predictor",
            })
    else:
        parameters.update({
            "tree_method": "hist",
        })

    return XGBClassifier(**parameters)


def make_lightgbm(device_type="cpu"):
    return LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=31,
        device_type=device_type,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )


def make_catboost(use_gpu=True):
    parameters = {
        "iterations": 1000,
        "learning_rate": 0.03,
        "depth": 6,
        "loss_function": "Logloss",
        "random_seed": RANDOM_STATE,
        "verbose": False,
        "allow_writing_files": False,
    }

    if use_gpu and GPU_AVAILABLE and CATBOOST_GPU_COUNT > 0:
        parameters.update({
            "task_type": "GPU",
            "devices": "0",
        })
    else:
        parameters.update({
            "task_type": "CPU",
            "thread_count": -1,
        })

    return CatBoostClassifier(**parameters)


# ============================================================
# 7. Probe LightGBM GPU support
# ============================================================

LIGHTGBM_DEVICE = "cpu"

if GPU_AVAILABLE:
    for candidate_device in ["cuda", "gpu"]:
        try:
            lightgbm_probe = make_lightgbm(
                device_type=candidate_device
            )

            lightgbm_probe.set_params(n_estimators=5)

            lightgbm_probe.fit(
                X_train_raw[:5000],
                y_train_array[:5000],
            )

            LIGHTGBM_DEVICE = candidate_device

            print(
                "\nLightGBM GPU support detected:",
                candidate_device,
            )

            del lightgbm_probe
            gc.collect()
            break

        except Exception as error:
            print(
                f"\nLightGBM '{candidate_device}' probe failed:",
                str(error)[:250],
            )

if LIGHTGBM_DEVICE == "cpu":
    print("\nLightGBM will use CPU.")


# ============================================================
# 8. Define the 12 baseline models
# ============================================================

model_configurations = [
    {
        "name": "Logistic Regression",
        "factory": lambda: LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        ),
        "input_type": "scaled",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "Naive Bayes",
        "factory": lambda: GaussianNB(),
        "input_type": "scaled",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "KNN",
        "factory": lambda: KNeighborsClassifier(
            n_neighbors=5,
            weights="uniform",
            metric="minkowski",
            p=2,
            n_jobs=-1,
        ),
        "input_type": "scaled",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "Linear SVM",
        "factory": lambda: LinearSVC(
            C=1.0,
            dual=False,
            tol=1e-3,
            max_iter=3000,
            random_state=RANDOM_STATE,
        ),
        "input_type": "scaled",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "Decision Tree",
        "factory": lambda: DecisionTreeClassifier(
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "Random Forest",
        "factory": lambda: RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input_type": "raw",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "Extra Trees",
        "factory": lambda: ExtraTreesClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input_type": "raw",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "AdaBoost",
        "factory": lambda: AdaBoostClassifier(
            n_estimators=50,
            learning_rate=1.0,
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "Gradient Boosting",
        "factory": lambda: GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),
        "input_type": "raw",
        "planned_device": "CPU",
        "fallback_factory": None,
    },

    {
        "name": "XGBoost",
        "factory": lambda: make_xgboost(use_gpu=True),
        "input_type": "raw",
        "planned_device": (
            "GPU" if GPU_AVAILABLE else "CPU"
        ),
        "fallback_factory": lambda: make_xgboost(
            use_gpu=False
        ),
    },

    {
        "name": "LightGBM",
        "factory": lambda: make_lightgbm(
            device_type=LIGHTGBM_DEVICE
        ),
        "input_type": "raw",
        "planned_device": (
            "GPU"
            if LIGHTGBM_DEVICE in {"cuda", "gpu"}
            else "CPU"
        ),
        "fallback_factory": lambda: make_lightgbm(
            device_type="cpu"
        ),
    },

    {
        "name": "CatBoost",
        "factory": lambda: make_catboost(use_gpu=True),
        "input_type": "raw",
        "planned_device": (
            "GPU"
            if GPU_AVAILABLE and CATBOOST_GPU_COUNT > 0
            else "CPU"
        ),
        "fallback_factory": lambda: make_catboost(
            use_gpu=False
        ),
    },
]

print("\nExecution plan:")

for configuration in model_configurations:
    print(
        f"{configuration['name']:<24}"
        f" -> {configuration['planned_device']}"
    )


# ============================================================
# 9. Evaluation function
# ============================================================

def calculate_binary_metrics(
    model_name,
    y_true,
    y_predicted,
    training_time,
    prediction_time,
    actual_device,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int32,
    ).reshape(-1)

    y_predicted = np.asarray(
        y_predicted,
        dtype=np.int32,
    ).reshape(-1)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    ).ravel()

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0.0
    )

    return {
        "Model": model_name,
        "Evaluation Split": "Validation",
        "Training Device": actual_device,
        "Accuracy": accuracy_score(
            y_true,
            y_predicted,
        ),
        "Precision": precision_score(
            y_true,
            y_predicted,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            y_predicted,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            y_predicted,
            zero_division=0,
        ),
        "FPR": false_positive_rate,
        "FNR": false_negative_rate,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Training Time (sec)": training_time,
        "Prediction Time (sec)": prediction_time,
    }


# ============================================================
# 10. Save-model helper
# ============================================================

def save_fitted_model(model_name, model):
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    joblib_path = MODELS_DIR / f"{safe_name}.pkl"
    joblib.dump(model, joblib_path)

    if model_name == "XGBoost":
        model.save_model(
            str(MODELS_DIR / "xgboost_baseline.json")
        )

    elif model_name == "LightGBM":
        model.booster_.save_model(
            str(MODELS_DIR / "lightgbm_baseline.txt")
        )

    elif model_name == "CatBoost":
        model.save_model(
            str(MODELS_DIR / "catboost_baseline.cbm")
        )


# ============================================================
# 11. Execute all 12 models
# ============================================================

baseline_results = []
error_records = []

for model_number, configuration in enumerate(
    model_configurations,
    start=1,
):
    model_name = configuration["name"]
    input_type = configuration["input_type"]
    planned_device = configuration["planned_device"]

    print("\n" + "=" * 76)
    print(
        f"Running model {model_number}/"
        f"{len(model_configurations)}: {model_name}"
    )
    print("Planned device:", planned_device)
    print("=" * 76)

    if input_type == "scaled":
        training_features = X_train_std
        validation_features = X_val_std
    else:
        training_features = X_train_raw
        validation_features = X_val_raw

    model = configuration["factory"]()
    actual_device = planned_device

    try:
        training_start = time.perf_counter()

        model.fit(
            training_features,
            y_train_array,
        )

        training_time = (
            time.perf_counter() - training_start
        )

    except Exception as first_error:
        fallback_factory = configuration[
            "fallback_factory"
        ]

        if fallback_factory is None:
            error_records.append({
                "Model": model_name,
                "Stage": "Training",
                "Error": repr(first_error),
            })

            pd.DataFrame(error_records).to_csv(
                ERROR_LOG_PATH,
                index=False,
            )

            print("Training failed:")
            print(repr(first_error))
            continue

        print("\nGPU execution failed.")
        print("Reason:", str(first_error)[:500])
        print("Retrying this model on CPU...")

        del model
        gc.collect()

        model = fallback_factory()
        actual_device = "CPU fallback"

        training_start = time.perf_counter()

        model.fit(
            training_features,
            y_train_array,
        )

        training_time = (
            time.perf_counter() - training_start
        )

    prediction_start = time.perf_counter()

    validation_predictions = model.predict(
        validation_features
    )

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    result = calculate_binary_metrics(
        model_name=model_name,
        y_true=y_val_array,
        y_predicted=validation_predictions,
        training_time=training_time,
        prediction_time=prediction_time,
        actual_device=actual_device,
    )

    baseline_results.append(result)

    save_fitted_model(
        model_name=model_name,
        model=model,
    )

    partial_results_df = pd.DataFrame(
        baseline_results
    ).sort_values(
        by=["F1-score", "Recall"],
        ascending=[False, False],
    ).reset_index(drop=True)

    partial_results_df.to_csv(
        PARTIAL_RESULTS_PATH,
        index=False,
    )

    print(
        f"\nAccuracy : {result['Accuracy']:.4f}"
        f"\nPrecision: {result['Precision']:.4f}"
        f"\nRecall   : {result['Recall']:.4f}"
        f"\nF1-score : {result['F1-score']:.4f}"
        f"\nFPR      : {result['FPR']:.4f}"
        f"\nFNR      : {result['FNR']:.4f}"
        f"\nTP       : {result['TP']}"
        f"\nTN       : {result['TN']}"
        f"\nFP       : {result['FP']}"
        f"\nFN       : {result['FN']}"
        f"\nTraining : {training_time:.2f} sec"
        f"\nPrediction: {prediction_time:.2f} sec"
        f"\nDevice   : {actual_device}"
    )

    del validation_predictions
    del model
    gc.collect()


# ============================================================
# 12. Save final results
# ============================================================

baseline12_results_df = pd.DataFrame(
    baseline_results
).sort_values(
    by=["F1-score", "Recall"],
    ascending=[False, False],
).reset_index(drop=True)

baseline12_results_df.insert(
    0,
    "Validation Rank",
    np.arange(1, len(baseline12_results_df) + 1),
)

baseline12_results_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False,
)

if error_records:
    pd.DataFrame(error_records).to_csv(
        ERROR_LOG_PATH,
        index=False,
    )


# ============================================================
# 13. Save metadata
# ============================================================

metadata = {
    "run_name": RUN_NAME,
    "random_state": RANDOM_STATE,
    "training_records": int(len(y_train_array)),
    "validation_records": int(len(y_val_array)),
    "feature_count": int(X_train_raw.shape[1]),
    "training_benign": int(
        np.sum(y_train_array == 0)
    ),
    "training_attack": int(
        np.sum(y_train_array == 1)
    ),
    "validation_benign": int(
        np.sum(y_val_array == 0)
    ),
    "validation_attack": int(
        np.sum(y_val_array == 1)
    ),
    "test_set_used": False,
    "gpu_available": bool(GPU_AVAILABLE),
    "gpu_information": gpu_query.stdout.strip(),
    "lightgbm_device": LIGHTGBM_DEVICE,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "lightgbm_version": lightgbm.__version__,
    "catboost_version": catboost.__version__,
}

with open(
    METADATA_DIR / "baseline12_metadata.json",
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        metadata,
        metadata_file,
        indent=2,
    )


# ============================================================
# 14. Display final table
# ============================================================

columns_to_display = [
    "Validation Rank",
    "Model",
    "Training Device",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "FPR",
    "FNR",
    "TP",
    "TN",
    "FP",
    "FN",
    "Training Time (sec)",
    "Prediction Time (sec)",
]

print("\n" + "=" * 76)
print("FINAL 12-MODEL VALIDATION BASELINE RESULTS")
print("=" * 76)

display(
    baseline12_results_df[
        columns_to_display
    ].round(4)
)

print("\nSaved final results:")
print(FINAL_RESULTS_PATH)

print("\nSaved partial checkpoint:")
print(PARTIAL_RESULTS_PATH)

print("\nSaved models:")
print(MODELS_DIR)

print("\nThe test set was not used.")

In [ ]:
# ============================================================
# FOUR NEURAL BASELINE MODELS
# Train: 64%
# Validation: 16%
# Test set: NOT USED
#
# Models:
#   1. MLP
#   2. 1D-CNN
#   3. LSTM
#   4. Transformer Encoder
# ============================================================

import os
import gc
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from tensorflow.keras import Model, Sequential
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    CSVLogger,
)
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    LSTM,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.optimizers import Adam


# ============================================================
# 1. Reproducibility and GPU setup
# ============================================================

RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

gpus = tf.config.list_physical_devices("GPU")

if not gpus:
    raise RuntimeError(
        "No TensorFlow GPU detected. Enable the Kaggle P100 accelerator."
    )

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("TensorFlow version:", tf.__version__)
print("Detected GPU devices:", gpus)


# ============================================================
# 2. Confirm required data
# ============================================================

required_variables = [
    "X_train_scaled",
    "X_val_scaled",
    "X_train_dl",
    "X_val_dl",
    "y_train",
    "y_val",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Run the dataset split and scaling cell first. "
        f"Missing variables: {missing_variables}"
    )


X_train_mlp = np.asarray(
    X_train_scaled,
    dtype=np.float32,
)

X_val_mlp = np.asarray(
    X_val_scaled,
    dtype=np.float32,
)

X_train_sequence = np.asarray(
    X_train_dl,
    dtype=np.float32,
)

X_val_sequence = np.asarray(
    X_val_dl,
    dtype=np.float32,
)

y_train_array = np.asarray(
    y_train,
    dtype=np.float32,
).reshape(-1)

y_val_array = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)


print("MLP training shape:      ", X_train_mlp.shape)
print("MLP validation shape:    ", X_val_mlp.shape)
print("Sequence training shape: ", X_train_sequence.shape)
print("Sequence validation shape:", X_val_sequence.shape)

assert X_train_mlp.shape == (192593, 78)
assert X_val_mlp.shape == (48149, 78)
assert X_train_sequence.shape == (192593, 78, 1)
assert X_val_sequence.shape == (48149, 78, 1)

print("\nThe test set is not used in this stage.")


# ============================================================
# 3. Output directories
# ============================================================

RUN_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "baseline4_neural_validation_clean_v1"
)

RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
LOGS_DIR = RUN_ROOT / "logs"
METADATA_DIR = RUN_ROOT / "metadata"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    LOGS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR / "baseline4_neural_validation_partial.csv"
)

FINAL_RESULTS_PATH = (
    RESULTS_DIR / "baseline4_neural_validation_results.csv"
)


# ============================================================
# 4. Metric function
# ============================================================

def evaluate_neural_model(
    model_name,
    y_true,
    probabilities,
    training_time,
    prediction_time,
    epochs_completed,
    threshold=0.50,
):
    probabilities = np.asarray(
        probabilities
    ).reshape(-1)

    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    y_true = np.asarray(
        y_true,
        dtype=np.int32,
    ).reshape(-1)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Model": model_name,
        "Evaluation Split": "Validation",
        "Training Device": "GPU",
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Epochs Completed": int(epochs_completed),
        "Training Time (sec)": training_time,
        "Prediction Time (sec)": prediction_time,
    }


# ============================================================
# 5. Model builders
# ============================================================

def build_mlp(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),

        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.20),

        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.20),

        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


def build_cnn(input_shape):
    model = Sequential([
        Input(shape=input_shape),

        Conv1D(
            filters=64,
            kernel_size=3,
            padding="same",
            activation="relu",
        ),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.30),

        Conv1D(
            filters=128,
            kernel_size=3,
            padding="same",
            activation="relu",
        ),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.30),

        GlobalAveragePooling1D(),

        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.30),

        Dense(64, activation="relu"),
        Dropout(0.15),

        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


def build_lstm(input_shape):
    model = Sequential([
        Input(shape=input_shape),

        LSTM(
            64,
            return_sequences=True,
            dropout=0.20,
        ),

        LSTM(
            32,
            return_sequences=False,
            dropout=0.20,
        ),

        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.30),

        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


def transformer_encoder_block(
    inputs,
    num_heads=4,
    key_dim=32,
    ff_dim=128,
    dropout_rate=0.20,
):
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
    )(inputs, inputs)

    attention_output = Dropout(
        dropout_rate
    )(attention_output)

    first_residual = LayerNormalization(
        epsilon=1e-6
    )(inputs + attention_output)

    feed_forward = Dense(
        ff_dim,
        activation="relu",
    )(first_residual)

    feed_forward = Dropout(
        dropout_rate
    )(feed_forward)

    feed_forward = Dense(
        int(inputs.shape[-1])
    )(feed_forward)

    return LayerNormalization(
        epsilon=1e-6
    )(first_residual + feed_forward)


def build_transformer(input_shape):
    inputs = Input(shape=input_shape)

    x = Dense(64)(inputs)

    x = transformer_encoder_block(
        x,
        num_heads=4,
        key_dim=32,
        ff_dim=128,
        dropout_rate=0.20,
    )

    x = transformer_encoder_block(
        x,
        num_heads=4,
        key_dim=32,
        ff_dim=128,
        dropout_rate=0.20,
    )

    x = GlobalAveragePooling1D()(x)

    x = Dense(
        128,
        activation="relu",
    )(x)
    x = Dropout(0.30)(x)

    x = Dense(
        64,
        activation="relu",
    )(x)
    x = Dropout(0.20)(x)

    outputs = Dense(
        1,
        activation="sigmoid",
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
    )

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


# ============================================================
# 6. Fixed baseline configurations
# ============================================================

model_configurations = [
    {
        "name": "MLP",
        "builder": lambda: build_mlp(
            X_train_mlp.shape[1]
        ),
        "train_data": X_train_mlp,
        "validation_data": X_val_mlp,
        "batch_size": 512,
        "epochs": 30,
    },

    {
        "name": "1D-CNN",
        "builder": lambda: build_cnn(
            X_train_sequence.shape[1:]
        ),
        "train_data": X_train_sequence,
        "validation_data": X_val_sequence,
        "batch_size": 512,
        "epochs": 30,
    },

    {
        "name": "LSTM",
        "builder": lambda: build_lstm(
            X_train_sequence.shape[1:]
        ),
        "train_data": X_train_sequence,
        "validation_data": X_val_sequence,
        "batch_size": 512,
        "epochs": 30,
    },

    {
        "name": "Transformer Encoder",
        "builder": lambda: build_transformer(
            X_train_sequence.shape[1:]
        ),
        "train_data": X_train_sequence,
        "validation_data": X_val_sequence,
        "batch_size": 512,
        "epochs": 30,
    },
]


# ============================================================
# 7. Train and evaluate
# ============================================================

neural_results = []

for model_number, configuration in enumerate(
    model_configurations,
    start=1,
):
    model_name = configuration["name"]

    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    print("\n" + "=" * 76)
    print(
        f"Running neural model {model_number}/4: "
        f"{model_name}"
    )
    print("=" * 76)

    tf.keras.backend.clear_session()
    gc.collect()

    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    tf.random.set_seed(RANDOM_STATE)

    model = configuration["builder"]()

    model_path = (
        MODELS_DIR / f"{safe_name}_baseline.keras"
    )

    log_path = (
        LOGS_DIR / f"{safe_name}_training_log.csv"
    )

    callbacks = [
        ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),

        EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=5,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),

        CSVLogger(
            filename=str(log_path),
            append=False,
        ),
    ]

    training_start = time.perf_counter()

    history = model.fit(
        configuration["train_data"],
        y_train_array,
        validation_data=(
            configuration["validation_data"],
            y_val_array,
        ),
        epochs=configuration["epochs"],
        batch_size=configuration["batch_size"],
        callbacks=callbacks,
        verbose=1,
        shuffle=True,
    )

    training_time = (
        time.perf_counter() - training_start
    )

    best_model = tf.keras.models.load_model(
        model_path
    )

    prediction_start = time.perf_counter()

    validation_probabilities = best_model.predict(
        configuration["validation_data"],
        batch_size=2048,
        verbose=1,
    ).reshape(-1)

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    result = evaluate_neural_model(
        model_name=model_name,
        y_true=y_val_array,
        probabilities=validation_probabilities,
        training_time=training_time,
        prediction_time=prediction_time,
        epochs_completed=len(
            history.history["loss"]
        ),
        threshold=0.50,
    )

    neural_results.append(result)

    partial_df = pd.DataFrame(
        neural_results
    ).sort_values(
        by=["F1-score", "Recall"],
        ascending=[False, False],
    ).reset_index(drop=True)

    partial_df.to_csv(
        PARTIAL_RESULTS_PATH,
        index=False,
    )

    print(
        f"\nAccuracy : {result['Accuracy']:.4f}"
        f"\nPrecision: {result['Precision']:.4f}"
        f"\nRecall   : {result['Recall']:.4f}"
        f"\nF1-score : {result['F1-score']:.4f}"
        f"\nFPR      : {result['FPR']:.4f}"
        f"\nFNR      : {result['FNR']:.4f}"
        f"\nTP       : {result['TP']}"
        f"\nTN       : {result['TN']}"
        f"\nFP       : {result['FP']}"
        f"\nFN       : {result['FN']}"
        f"\nEpochs   : {result['Epochs Completed']}"
        f"\nTraining : {training_time:.2f} sec"
        f"\nPrediction: {prediction_time:.2f} sec"
    )

    del model
    del best_model
    del history
    del validation_probabilities

    tf.keras.backend.clear_session()
    gc.collect()


# ============================================================
# 8. Save final neural baseline results
# ============================================================

neural_results_df = pd.DataFrame(
    neural_results
).sort_values(
    by=["F1-score", "Recall"],
    ascending=[False, False],
).reset_index(drop=True)

neural_results_df.insert(
    0,
    "Neural Validation Rank",
    np.arange(
        1,
        len(neural_results_df) + 1,
    ),
)

neural_results_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False,
)


metadata = {
    "random_state": RANDOM_STATE,
    "training_records": int(len(y_train_array)),
    "validation_records": int(len(y_val_array)),
    "feature_count": int(X_train_mlp.shape[1]),
    "test_set_used": False,
    "tensorflow_version": tf.__version__,
    "gpu_devices": [str(gpu) for gpu in gpus],
    "models": [
        configuration["name"]
        for configuration in model_configurations
    ],
}

with open(
    METADATA_DIR / "baseline4_neural_metadata.json",
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        metadata,
        metadata_file,
        indent=2,
    )


# ============================================================
# 9. Display the four-model table
# ============================================================

display_columns = [
    "Neural Validation Rank",
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "FPR",
    "FNR",
    "TP",
    "TN",
    "FP",
    "FN",
    "Epochs Completed",
    "Training Time (sec)",
    "Prediction Time (sec)",
]

print("\n" + "=" * 76)
print("FINAL FOUR-MODEL NEURAL VALIDATION RESULTS")
print("=" * 76)

display(
    neural_results_df[
        display_columns
    ].round(4)
)

print("\nSaved results:")
print(FINAL_RESULTS_PATH)

print("\nThe test set was not used.")

In [ ]:
# ============================================================
# STAGE 2A: GPU HYPERPARAMETER TUNING
#
# Models:
#   1. XGBoost
#   2. LightGBM
#   3. CatBoost
#
# Selection procedure:
#   - 3-fold stratified CV on training set only
#   - Best configuration selected by mean CV F1-score
#   - Best estimator evaluated on validation set
#   - Test set is NEVER used
# ============================================================

import os
import gc
import json
import time
import random
import warnings
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
import lightgbm
import catboost

from IPython.display import display
from packaging.version import Version

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_STATE = 42
CV_FOLDS = 3
N_ITERATIONS = 15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ============================================================
# 2. Confirm required data exists
# ============================================================

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        "Run the dataset loading and 64/16/20 split cell first. "
        f"Missing variables: {missing_variables}"
    )

assert X_train.shape == (192593, 78), X_train.shape
assert X_val.shape == (48149, 78), X_val.shape
assert len(y_train) == 192593
assert len(y_val) == 48149

X_train_boost = np.asarray(
    X_train,
    dtype=np.float32,
)

X_val_boost = np.asarray(
    X_val,
    dtype=np.float32,
)

y_train_boost = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_boost = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

print("Training set:  ", X_train_boost.shape)
print("Validation set:", X_val_boost.shape)
print("\nThe test set is not used in this stage.")


# ============================================================
# 3. Confirm GPU
# ============================================================

gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)

GPU_AVAILABLE = (
    gpu_query.returncode == 0
    and bool(gpu_query.stdout.strip())
)

if not GPU_AVAILABLE:
    raise RuntimeError(
        "No NVIDIA GPU was detected. "
        "Enable the Kaggle P100 accelerator."
    )

print("\nGPU:")
print(gpu_query.stdout.strip())

print("\nLibrary versions:")
print("scikit-learn:", sklearn.__version__)
print("XGBoost:     ", xgboost.__version__)
print("LightGBM:    ", lightgbm.__version__)
print("CatBoost:    ", catboost.__version__)
print("CatBoost GPUs:", get_gpu_device_count())


# ============================================================
# 4. Output directories
# ============================================================

RUN_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
SEARCH_DIR = RUN_ROOT / "search_logs"
PARAMS_DIR = RUN_ROOT / "best_parameters"
METADATA_DIR = RUN_ROOT / "metadata"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    SEARCH_DIR,
    PARAMS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PARTIAL_COMPARISON_PATH = (
    RESULTS_DIR /
    "boosting_tuned_validation_partial.csv"
)

FINAL_COMPARISON_PATH = (
    RESULTS_DIR /
    "boosting_tuned_validation_results.csv"
)


# ============================================================
# 5. Cross-validation configuration
# ============================================================

cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


# ============================================================
# 6. Detect LightGBM GPU backend
# ============================================================

def detect_lightgbm_device():
    for candidate_device in ["cuda", "gpu"]:
        try:
            probe = LGBMClassifier(
                n_estimators=5,
                learning_rate=0.1,
                num_leaves=31,
                device_type=candidate_device,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            )

            probe.fit(
                X_train_boost[:5000],
                y_train_boost[:5000],
            )

            del probe
            gc.collect()

            return candidate_device

        except Exception as error:
            print(
                f"LightGBM '{candidate_device}' unavailable:",
                str(error)[:180],
            )

    return "cpu"


LIGHTGBM_DEVICE = detect_lightgbm_device()

print("\nLightGBM device:", LIGHTGBM_DEVICE)


# ============================================================
# 7. Base estimator builders
# ============================================================

def build_xgboost():
    parameters = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
        "verbosity": 0,
    }

    if Version(xgboost.__version__) >= Version("2.0.0"):
        parameters["device"] = "cuda"
    else:
        parameters["tree_method"] = "gpu_hist"
        parameters["predictor"] = "gpu_predictor"

    return XGBClassifier(**parameters)


def build_lightgbm():
    return LGBMClassifier(
        objective="binary",
        device_type=LIGHTGBM_DEVICE,
        subsample_freq=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )


def build_catboost():
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="F1",
        task_type="GPU",
        devices="0",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )


# ============================================================
# 8. Hyperparameter spaces
# ============================================================

xgboost_parameter_space = {
    "n_estimators": [
        200,
        300,
        500,
        700,
    ],
    "max_depth": [
        4,
        6,
        8,
        10,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
        0.15,
    ],
    "subsample": [
        0.80,
        0.90,
        1.00,
    ],
    "colsample_bytree": [
        0.80,
        0.90,
        1.00,
    ],
    "min_child_weight": [
        1,
        3,
        5,
    ],
    "gamma": [
        0.0,
        0.1,
        0.3,
    ],
    "reg_alpha": [
        0.0,
        0.1,
        1.0,
    ],
    "reg_lambda": [
        1.0,
        5.0,
        10.0,
    ],
}


lightgbm_parameter_space = {
    "n_estimators": [
        200,
        400,
        700,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
    ],
    "num_leaves": [
        31,
        63,
        127,
    ],
    "max_depth": [
        -1,
        15,
        30,
    ],
    "min_child_samples": [
        10,
        20,
        40,
    ],
    "subsample": [
        0.80,
        0.90,
        1.00,
    ],
    "colsample_bytree": [
        0.80,
        0.90,
        1.00,
    ],
    "reg_alpha": [
        0.0,
        0.1,
        1.0,
    ],
    "reg_lambda": [
        0.0,
        1.0,
        5.0,
    ],
}


catboost_parameter_space = {
    "iterations": [
        300,
        500,
        700,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
    ],
    "depth": [
        6,
        8,
        10,
    ],
    "l2_leaf_reg": [
        3,
        5,
        7,
    ],
    "random_strength": [
        0.5,
        1.0,
        2.0,
    ],
    "border_count": [
        64,
        128,
        254,
    ],
}


# ============================================================
# 9. Search definitions
# ============================================================

search_configurations = [
    {
        "name": "XGBoost Tuned",
        "short_name": "xgboost",
        "estimator": build_xgboost(),
        "parameter_space": xgboost_parameter_space,
        "device": "GPU",
    },
    {
        "name": "LightGBM Tuned",
        "short_name": "lightgbm",
        "estimator": build_lightgbm(),
        "parameter_space": lightgbm_parameter_space,
        "device": (
            "GPU"
            if LIGHTGBM_DEVICE in {"cuda", "gpu"}
            else "CPU"
        ),
    },
    {
        "name": "CatBoost Tuned",
        "short_name": "catboost",
        "estimator": build_catboost(),
        "parameter_space": catboost_parameter_space,
        "device": "GPU",
    },
]


# ============================================================
# 10. Utility functions
# ============================================================

def convert_to_builtin(value):
    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    return value


def clean_parameter_dictionary(parameters):
    return {
        key: convert_to_builtin(value)
        for key, value in parameters.items()
    }


def calculate_validation_metrics(
    model_name,
    y_true,
    probabilities,
    best_cv_f1,
    search_time,
    prediction_time,
    device,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    ).reshape(-1)

    predictions = (
        probabilities >= 0.50
    ).astype(np.int32)

    y_true = np.asarray(
        y_true,
        dtype=np.int32,
    ).reshape(-1)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Model": model_name,
        "Evaluation Split": "Validation",
        "Training Device": device,
        "Threshold": 0.50,
        "Best CV F1-score": best_cv_f1,
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Search Time (sec)": search_time,
        "Prediction Time (sec)": prediction_time,
    }


def save_native_model(
    short_name,
    fitted_model,
):
    joblib.dump(
        fitted_model,
        MODELS_DIR / f"{short_name}_tuned.pkl",
    )

    if short_name == "xgboost":
        fitted_model.save_model(
            str(
                MODELS_DIR /
                "xgboost_tuned.json"
            )
        )

    elif short_name == "lightgbm":
        fitted_model.booster_.save_model(
            str(
                MODELS_DIR /
                "lightgbm_tuned.txt"
            )
        )

    elif short_name == "catboost":
        fitted_model.save_model(
            str(
                MODELS_DIR /
                "catboost_tuned.cbm"
            )
        )


# ============================================================
# 11. Run the three tuning searches
# ============================================================

tuned_validation_results = []

for model_number, configuration in enumerate(
    search_configurations,
    start=1,
):
    model_name = configuration["name"]
    short_name = configuration["short_name"]

    print("\n" + "=" * 78)
    print(
        f"Tuning model {model_number}/3: "
        f"{model_name}"
    )
    print("Device:", configuration["device"])
    print(
        f"Random configurations: {N_ITERATIONS}"
    )
    print(f"CV folds: {CV_FOLDS}")
    print("=" * 78)

    search = RandomizedSearchCV(
        estimator=configuration["estimator"],
        param_distributions=(
            configuration["parameter_space"]
        ),
        n_iter=N_ITERATIONS,
        scoring="f1",
        n_jobs=1,
        cv=cv_strategy,
        refit=True,
        random_state=RANDOM_STATE,
        verbose=2,
        return_train_score=False,
        error_score="raise",
        pre_dispatch=1,
    )

    search_start = time.perf_counter()

    search.fit(
        X_train_boost,
        y_train_boost,
    )

    search_time = (
        time.perf_counter() - search_start
    )

    best_model = search.best_estimator_
    best_parameters = clean_parameter_dictionary(
        search.best_params_
    )

    print("\nBest CV F1-score:")
    print(f"{search.best_score_:.6f}")

    print("\nBest parameters:")
    print(
        json.dumps(
            best_parameters,
            indent=2,
        )
    )

    # --------------------------------------------------------
    # Save complete CV search log
    # --------------------------------------------------------

    cv_results = pd.DataFrame(
        search.cv_results_
    ).sort_values(
        by="rank_test_score",
        ascending=True,
    ).reset_index(drop=True)

    cv_results.to_csv(
        SEARCH_DIR /
        f"{short_name}_random_search_results.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Save best parameters
    # --------------------------------------------------------

    best_parameter_record = {
        "model": model_name,
        "selection_metric": "Mean 3-fold CV F1-score",
        "best_cv_f1": float(search.best_score_),
        "number_of_random_candidates": N_ITERATIONS,
        "cv_folds": CV_FOLDS,
        "best_parameters": best_parameters,
    }

    with open(
        PARAMS_DIR /
        f"{short_name}_best_parameters.json",
        "w",
        encoding="utf-8",
    ) as parameter_file:
        json.dump(
            best_parameter_record,
            parameter_file,
            indent=2,
        )

    # --------------------------------------------------------
    # Evaluate on validation set at threshold 0.50
    # --------------------------------------------------------

    prediction_start = time.perf_counter()

    validation_probabilities = (
        best_model.predict_proba(
            X_val_boost
        )[:, 1]
    )

    prediction_time = (
        time.perf_counter() -
        prediction_start
    )

    validation_result = (
        calculate_validation_metrics(
            model_name=model_name,
            y_true=y_val_boost,
            probabilities=validation_probabilities,
            best_cv_f1=float(
                search.best_score_
            ),
            search_time=search_time,
            prediction_time=prediction_time,
            device=configuration["device"],
        )
    )

    tuned_validation_results.append(
        validation_result
    )

    # Save validation probabilities for later threshold tuning
    probability_output = pd.DataFrame({
        "y_true": y_val_boost,
        "attack_probability": (
            validation_probabilities
        ),
    })

    probability_output.to_csv(
        RESULTS_DIR /
        f"{short_name}_validation_probabilities.csv",
        index=False,
    )

    # Save model
    save_native_model(
        short_name=short_name,
        fitted_model=best_model,
    )

    # Save incremental comparison
    partial_comparison = pd.DataFrame(
        tuned_validation_results
    ).sort_values(
        by=["F1-score", "Recall"],
        ascending=[False, False],
    ).reset_index(drop=True)

    partial_comparison.to_csv(
        PARTIAL_COMPARISON_PATH,
        index=False,
    )

    print(
        f"\nValidation accuracy : "
        f"{validation_result['Accuracy']:.4f}"
        f"\nValidation precision: "
        f"{validation_result['Precision']:.4f}"
        f"\nValidation recall   : "
        f"{validation_result['Recall']:.4f}"
        f"\nValidation F1-score : "
        f"{validation_result['F1-score']:.4f}"
        f"\nValidation FPR      : "
        f"{validation_result['FPR']:.4f}"
        f"\nValidation FNR      : "
        f"{validation_result['FNR']:.4f}"
        f"\nSearch time         : "
        f"{search_time:.2f} sec"
    )

    del validation_probabilities
    del probability_output
    del best_model
    del search
    del cv_results

    gc.collect()


# ============================================================
# 12. Save final tuned boosting comparison
# ============================================================

boosting_tuned_results_df = pd.DataFrame(
    tuned_validation_results
).sort_values(
    by=["F1-score", "Recall"],
    ascending=[False, False],
).reset_index(drop=True)

boosting_tuned_results_df.insert(
    0,
    "Validation Rank",
    np.arange(
        1,
        len(boosting_tuned_results_df) + 1,
    ),
)

boosting_tuned_results_df.to_csv(
    FINAL_COMPARISON_PATH,
    index=False,
)


# ============================================================
# 13. Save run metadata
# ============================================================

metadata = {
    "random_state": RANDOM_STATE,
    "cv_folds": CV_FOLDS,
    "random_candidates_per_model": N_ITERATIONS,
    "selection_metric": "F1-score",
    "training_records": int(
        len(y_train_boost)
    ),
    "validation_records": int(
        len(y_val_boost)
    ),
    "feature_count": int(
        X_train_boost.shape[1]
    ),
    "test_set_used": False,
    "gpu_information": (
        gpu_query.stdout.strip()
    ),
    "lightgbm_device": LIGHTGBM_DEVICE,
    "xgboost_version": (
        xgboost.__version__
    ),
    "lightgbm_version": (
        lightgbm.__version__
    ),
    "catboost_version": (
        catboost.__version__
    ),
}

with open(
    METADATA_DIR /
    "boosting_tuning_metadata.json",
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        metadata,
        metadata_file,
        indent=2,
    )


# ============================================================
# 14. Display final comparison
# ============================================================

display_columns = [
    "Validation Rank",
    "Model",
    "Training Device",
    "Best CV F1-score",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "FPR",
    "FNR",
    "TP",
    "TN",
    "FP",
    "FN",
    "Search Time (sec)",
]

print("\n" + "=" * 78)
print("TUNED BOOSTING MODEL VALIDATION RESULTS")
print("=" * 78)

display(
    boosting_tuned_results_df[
        display_columns
    ].round(4)
)

print("\nFinal comparison:")
print(FINAL_COMPARISON_PATH)

print("\nModels:")
print(MODELS_DIR)

print("\nSearch logs:")
print(SEARCH_DIR)

print("\nValidation probabilities:")
print(RESULTS_DIR)

print("\nThe test set was not used.")

In [ ]:
from pathlib import Path

root = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

paths = [
    root / "results" / "boosting_tuned_validation_partial.csv",
    root / "results" / "xgboost_validation_probabilities.csv",
    root / "search_logs" / "xgboost_random_search_results.csv",
    root / "best_parameters" / "xgboost_best_parameters.json",
    root / "models" / "xgboost_tuned.pkl",
    root / "models" / "xgboost_tuned.json",
]

for path in paths:
    print(f"{'FOUND' if path.exists() else 'MISSING':<8} {path}")

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

partial_path = (
    root / "results" /
    "boosting_tuned_validation_partial.csv"
)

display(pd.read_csv(partial_path).round(4))

In [ ]:
# ============================================================
# LIGHTGBM-ONLY HYPERPARAMETER TUNING
#
# Selection:
#   - Training set only for 3-fold CV
#   - Validation set for external evaluation
#   - Test set is not used
#
# Existing XGBoost outputs are preserved.
# ============================================================

import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import lightgbm

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from lightgbm import LGBMClassifier


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
CV_FOLDS = 3
N_ITERATIONS = 15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Confirm data exists
# ------------------------------------------------------------

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun the dataset loading and 64/16/20 split cell first. "
        f"Missing variables: {missing_variables}"
    )

X_train_lgbm = np.asarray(
    X_train,
    dtype=np.float32,
)

X_val_lgbm = np.asarray(
    X_val,
    dtype=np.float32,
)

y_train_lgbm = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_lgbm = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_lgbm.shape == (192593, 78)
assert X_val_lgbm.shape == (48149, 78)

print("Training:", X_train_lgbm.shape)
print("Validation:", X_val_lgbm.shape)
print("Test set used: False")


# ------------------------------------------------------------
# 3. Existing output directories
# ------------------------------------------------------------

RUN_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
SEARCH_DIR = RUN_ROOT / "search_logs"
PARAMS_DIR = RUN_ROOT / "best_parameters"
METADATA_DIR = RUN_ROOT / "metadata"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    SEARCH_DIR,
    PARAMS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR /
    "boosting_tuned_validation_partial.csv"
)


# ------------------------------------------------------------
# 4. Confirm LightGBM GPU backend
# ------------------------------------------------------------

LIGHTGBM_DEVICE = None

for candidate_device in ["gpu", "cuda"]:
    try:
        probe = LGBMClassifier(
            n_estimators=5,
            device_type=candidate_device,
            random_state=RANDOM_STATE,
            verbosity=-1,
        )

        probe.fit(
            X_train_lgbm[:5000],
            y_train_lgbm[:5000],
        )

        LIGHTGBM_DEVICE = candidate_device

        print(
            "LightGBM backend selected:",
            LIGHTGBM_DEVICE,
        )

        del probe
        gc.collect()
        break

    except Exception as error:
        print(
            f"Backend '{candidate_device}' unavailable:",
            str(error)[:180],
        )

if LIGHTGBM_DEVICE is None:
    LIGHTGBM_DEVICE = "cpu"
    print("LightGBM will use CPU fallback.")


# ------------------------------------------------------------
# 5. Base estimator
# ------------------------------------------------------------

lightgbm_estimator = LGBMClassifier(
    objective="binary",
    device_type=LIGHTGBM_DEVICE,
    subsample_freq=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)


# ------------------------------------------------------------
# 6. Search space
# ------------------------------------------------------------

lightgbm_parameter_space = {
    "n_estimators": [
        200,
        400,
        700,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
    ],
    "num_leaves": [
        31,
        63,
        127,
    ],
    "max_depth": [
        -1,
        15,
        30,
    ],
    "min_child_samples": [
        10,
        20,
        40,
    ],
    "subsample": [
        0.80,
        0.90,
        1.00,
    ],
    "colsample_bytree": [
        0.80,
        0.90,
        1.00,
    ],
    "reg_alpha": [
        0.0,
        0.1,
        1.0,
    ],
    "reg_lambda": [
        0.0,
        1.0,
        5.0,
    ],
}


# ------------------------------------------------------------
# 7. Cross-validation search
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = RandomizedSearchCV(
    estimator=lightgbm_estimator,
    param_distributions=lightgbm_parameter_space,
    n_iter=N_ITERATIONS,
    scoring="f1",
    cv=cv_strategy,
    refit=True,
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=False,
    error_score="raise",
    pre_dispatch=1,
)

print("\n" + "=" * 76)
print("TUNING LIGHTGBM")
print("Backend:", LIGHTGBM_DEVICE)
print("Candidates:", N_ITERATIONS)
print("CV folds:", CV_FOLDS)
print("Total CV fits:", N_ITERATIONS * CV_FOLDS)
print("=" * 76)

search_start = time.perf_counter()

search.fit(
    X_train_lgbm,
    y_train_lgbm,
)

search_time = (
    time.perf_counter() - search_start
)

best_model = search.best_estimator_

best_parameters = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value in search.best_params_.items()
}

print("\nBest CV F1-score:")
print(f"{search.best_score_:.6f}")

print("\nBest parameters:")
print(
    json.dumps(
        best_parameters,
        indent=2,
    )
)


# ------------------------------------------------------------
# 8. Save complete search log
# ------------------------------------------------------------

search_results_df = pd.DataFrame(
    search.cv_results_
).sort_values(
    "rank_test_score",
    ascending=True,
).reset_index(drop=True)

search_results_path = (
    SEARCH_DIR /
    "lightgbm_random_search_results.csv"
)

search_results_df.to_csv(
    search_results_path,
    index=False,
)


# ------------------------------------------------------------
# 9. Save best parameters
# ------------------------------------------------------------

parameter_record = {
    "model": "LightGBM Tuned",
    "selection_metric": "Mean 3-fold CV F1-score",
    "best_cv_f1": float(search.best_score_),
    "number_of_random_candidates": N_ITERATIONS,
    "cv_folds": CV_FOLDS,
    "training_records": int(len(y_train_lgbm)),
    "feature_count": int(X_train_lgbm.shape[1]),
    "lightgbm_backend": LIGHTGBM_DEVICE,
    "best_parameters": best_parameters,
}

parameters_path = (
    PARAMS_DIR /
    "lightgbm_best_parameters.json"
)

with open(
    parameters_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        parameter_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Validation probabilities and metrics
# ------------------------------------------------------------

prediction_start = time.perf_counter()

validation_probabilities = (
    best_model.predict_proba(
        X_val_lgbm
    )[:, 1]
)

prediction_time = (
    time.perf_counter() - prediction_start
)

validation_predictions = (
    validation_probabilities >= 0.50
).astype(np.int32)

tn, fp, fn, tp = confusion_matrix(
    y_val_lgbm,
    validation_predictions,
    labels=[0, 1],
).ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

validation_result = {
    "Model": "LightGBM Tuned",
    "Evaluation Split": "Validation",
    "Training Device": (
        "GPU"
        if LIGHTGBM_DEVICE in {"gpu", "cuda"}
        else "CPU"
    ),
    "Threshold": 0.50,
    "Best CV F1-score": float(
        search.best_score_
    ),
    "Accuracy": accuracy_score(
        y_val_lgbm,
        validation_predictions,
    ),
    "Precision": precision_score(
        y_val_lgbm,
        validation_predictions,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val_lgbm,
        validation_predictions,
        zero_division=0,
    ),
    "F1-score": f1_score(
        y_val_lgbm,
        validation_predictions,
        zero_division=0,
    ),
    "FPR": fpr,
    "FNR": fnr,
    "TP": int(tp),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "Search Time (sec)": search_time,
    "Prediction Time (sec)": prediction_time,
}


# ------------------------------------------------------------
# 11. Save validation probabilities
# ------------------------------------------------------------

probabilities_path = (
    RESULTS_DIR /
    "lightgbm_validation_probabilities.csv"
)

pd.DataFrame({
    "y_true": y_val_lgbm,
    "attack_probability": validation_probabilities,
}).to_csv(
    probabilities_path,
    index=False,
)


# ------------------------------------------------------------
# 12. Save fitted model
# ------------------------------------------------------------

joblib.dump(
    best_model,
    MODELS_DIR / "lightgbm_tuned.pkl",
)

best_model.booster_.save_model(
    str(
        MODELS_DIR /
        "lightgbm_tuned.txt"
    )
)


# ------------------------------------------------------------
# 13. Append to existing XGBoost checkpoint
# ------------------------------------------------------------

if PARTIAL_RESULTS_PATH.exists():
    partial_df = pd.read_csv(
        PARTIAL_RESULTS_PATH
    )
else:
    partial_df = pd.DataFrame()

# Remove an older LightGBM row before appending
if not partial_df.empty and "Model" in partial_df.columns:
    partial_df = partial_df[
        partial_df["Model"] != "LightGBM Tuned"
    ]

partial_df = pd.concat(
    [
        partial_df,
        pd.DataFrame([validation_result]),
    ],
    ignore_index=True,
)

partial_df = partial_df.sort_values(
    by=["F1-score", "Recall"],
    ascending=[False, False],
).reset_index(drop=True)

partial_df.to_csv(
    PARTIAL_RESULTS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 14. Display result
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("LIGHTGBM TUNING COMPLETED")
print("=" * 76)

display(
    pd.DataFrame(
        [validation_result]
    ).round(4)
)

print("\nSaved:")
print(search_results_path)
print(parameters_path)
print(probabilities_path)
print(MODELS_DIR / "lightgbm_tuned.pkl")
print(MODELS_DIR / "lightgbm_tuned.txt")
print(PARTIAL_RESULTS_PATH)

print("\nThe test set was not used.")

In [ ]:
# ============================================================
# CATBOOST-ONLY HYPERPARAMETER TUNING
#
# Selection:
#   - 3-fold CV on training set only
#   - Validation set for external evaluation
#   - Test set is not used
#
# Existing XGBoost and LightGBM outputs are preserved.
# ============================================================

import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import catboost

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
CV_FOLDS = 3
N_ITERATIONS = 15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Confirm data exists
# ------------------------------------------------------------

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun the dataset loading and 64/16/20 split cell first. "
        f"Missing variables: {missing_variables}"
    )

X_train_cat = np.asarray(
    X_train,
    dtype=np.float32,
)

X_val_cat = np.asarray(
    X_val,
    dtype=np.float32,
)

y_train_cat = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_cat = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_cat.shape == (192593, 78)
assert X_val_cat.shape == (48149, 78)

print("Training:", X_train_cat.shape)
print("Validation:", X_val_cat.shape)
print("Test set used: False")


# ------------------------------------------------------------
# 3. Confirm GPU
# ------------------------------------------------------------

catboost_gpu_count = get_gpu_device_count()

print("\nCatBoost version:", catboost.__version__)
print("Detected CatBoost GPUs:", catboost_gpu_count)

if catboost_gpu_count < 1:
    raise RuntimeError(
        "CatBoost cannot detect the Kaggle GPU. "
        "Confirm that the P100 accelerator is enabled."
    )


# ------------------------------------------------------------
# 4. Existing output directories
# ------------------------------------------------------------

RUN_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
SEARCH_DIR = RUN_ROOT / "search_logs"
PARAMS_DIR = RUN_ROOT / "best_parameters"
METADATA_DIR = RUN_ROOT / "metadata"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    SEARCH_DIR,
    PARAMS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR /
    "boosting_tuned_validation_partial.csv"
)


# ------------------------------------------------------------
# 5. Base estimator
# ------------------------------------------------------------

catboost_estimator = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="F1",
    task_type="GPU",
    devices="0",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)


# ------------------------------------------------------------
# 6. Search space
# ------------------------------------------------------------

catboost_parameter_space = {
    "iterations": [
        300,
        500,
        700,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
    ],
    "depth": [
        6,
        8,
        10,
    ],
    "l2_leaf_reg": [
        3,
        5,
        7,
    ],
    "random_strength": [
        0.5,
        1.0,
        2.0,
    ],
    "border_count": [
        64,
        128,
        254,
    ],
}


# ------------------------------------------------------------
# 7. Cross-validation search
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = RandomizedSearchCV(
    estimator=catboost_estimator,
    param_distributions=catboost_parameter_space,
    n_iter=N_ITERATIONS,
    scoring="f1",
    cv=cv_strategy,
    refit=True,
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=False,
    error_score="raise",
    pre_dispatch=1,
)

print("\n" + "=" * 76)
print("TUNING CATBOOST")
print("Device: GPU")
print("Candidates:", N_ITERATIONS)
print("CV folds:", CV_FOLDS)
print("Total CV fits:", N_ITERATIONS * CV_FOLDS)
print("=" * 76)

search_start = time.perf_counter()

search.fit(
    X_train_cat,
    y_train_cat,
)

search_time = (
    time.perf_counter() - search_start
)

best_model = search.best_estimator_

best_parameters = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value in search.best_params_.items()
}

print("\nBest CV F1-score:")
print(f"{search.best_score_:.6f}")

print("\nBest parameters:")
print(
    json.dumps(
        best_parameters,
        indent=2,
    )
)


# ------------------------------------------------------------
# 8. Save complete search log
# ------------------------------------------------------------

search_results_df = pd.DataFrame(
    search.cv_results_
).sort_values(
    "rank_test_score",
    ascending=True,
).reset_index(drop=True)

search_results_path = (
    SEARCH_DIR /
    "catboost_random_search_results.csv"
)

search_results_df.to_csv(
    search_results_path,
    index=False,
)


# ------------------------------------------------------------
# 9. Save best parameters
# ------------------------------------------------------------

parameter_record = {
    "model": "CatBoost Tuned",
    "selection_metric": "Mean 3-fold CV F1-score",
    "best_cv_f1": float(search.best_score_),
    "number_of_random_candidates": N_ITERATIONS,
    "cv_folds": CV_FOLDS,
    "training_records": int(len(y_train_cat)),
    "feature_count": int(X_train_cat.shape[1]),
    "training_device": "GPU",
    "best_parameters": best_parameters,
}

parameters_path = (
    PARAMS_DIR /
    "catboost_best_parameters.json"
)

with open(
    parameters_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        parameter_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Validation probabilities and metrics
# ------------------------------------------------------------

prediction_start = time.perf_counter()

validation_probabilities = (
    best_model.predict_proba(
        X_val_cat
    )[:, 1]
)

prediction_time = (
    time.perf_counter() - prediction_start
)

validation_predictions = (
    validation_probabilities >= 0.50
).astype(np.int32)

tn, fp, fn, tp = confusion_matrix(
    y_val_cat,
    validation_predictions,
    labels=[0, 1],
).ravel()

fpr = (
    fp / (fp + tn)
    if fp + tn > 0
    else 0.0
)

fnr = (
    fn / (fn + tp)
    if fn + tp > 0
    else 0.0
)

validation_result = {
    "Model": "CatBoost Tuned",
    "Evaluation Split": "Validation",
    "Training Device": "GPU",
    "Threshold": 0.50,
    "Best CV F1-score": float(
        search.best_score_
    ),
    "Accuracy": accuracy_score(
        y_val_cat,
        validation_predictions,
    ),
    "Precision": precision_score(
        y_val_cat,
        validation_predictions,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val_cat,
        validation_predictions,
        zero_division=0,
    ),
    "F1-score": f1_score(
        y_val_cat,
        validation_predictions,
        zero_division=0,
    ),
    "FPR": fpr,
    "FNR": fnr,
    "TP": int(tp),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "Search Time (sec)": search_time,
    "Prediction Time (sec)": prediction_time,
}


# ------------------------------------------------------------
# 11. Save validation probabilities
# ------------------------------------------------------------

probabilities_path = (
    RESULTS_DIR /
    "catboost_validation_probabilities.csv"
)

pd.DataFrame({
    "y_true": y_val_cat,
    "attack_probability": validation_probabilities,
}).to_csv(
    probabilities_path,
    index=False,
)


# ------------------------------------------------------------
# 12. Save fitted model
# ------------------------------------------------------------

joblib.dump(
    best_model,
    MODELS_DIR / "catboost_tuned.pkl",
)

best_model.save_model(
    str(
        MODELS_DIR /
        "catboost_tuned.cbm"
    )
)


# ------------------------------------------------------------
# 13. Append to existing comparison checkpoint
# ------------------------------------------------------------

if PARTIAL_RESULTS_PATH.exists():
    partial_df = pd.read_csv(
        PARTIAL_RESULTS_PATH
    )
else:
    partial_df = pd.DataFrame()

# Remove an older CatBoost row before appending
if not partial_df.empty and "Model" in partial_df.columns:
    partial_df = partial_df[
        partial_df["Model"] != "CatBoost Tuned"
    ]

partial_df = pd.concat(
    [
        partial_df,
        pd.DataFrame([validation_result]),
    ],
    ignore_index=True,
)

partial_df = partial_df.sort_values(
    by=["F1-score", "Recall"],
    ascending=[False, False],
).reset_index(drop=True)

partial_df.to_csv(
    PARTIAL_RESULTS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 14. Display result
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("CATBOOST TUNING COMPLETED")
print("=" * 76)

display(
    pd.DataFrame(
        [validation_result]
    ).round(4)
)

print("\nSaved:")
print(search_results_path)
print(parameters_path)
print(probabilities_path)
print(MODELS_DIR / "catboost_tuned.pkl")
print(MODELS_DIR / "catboost_tuned.cbm")
print(PARTIAL_RESULTS_PATH)

print("\nThe test set was not used.")

In [ ]:
from pathlib import Path
import pandas as pd

RUN_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

partial_path = (
    RUN_ROOT / "results" /
    "boosting_tuned_validation_partial.csv"
)

final_path = (
    RUN_ROOT / "results" /
    "boosting_tuned_validation_results.csv"
)

df = pd.read_csv(partial_path)

expected_models = {
    "XGBoost Tuned",
    "LightGBM Tuned",
    "CatBoost Tuned",
}

found_models = set(df["Model"])

missing = expected_models - found_models

if missing:
    raise ValueError(
        f"Missing tuned models: {sorted(missing)}"
    )

final_df = (
    df[df["Model"].isin(expected_models)]
    .drop_duplicates(subset=["Model"], keep="last")
    .sort_values(
        ["F1-score", "Recall"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

final_df.to_csv(final_path, index=False)

display(
    final_df[
        [
            "Model",
            "Best CV F1-score",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
        ]
    ].round(6)
)

print("\nSaved:")
print(final_path)
print("\nTest set used: False")

In [ ]:
# ============================================================
# VALIDATION-SAFE MLP HYPERPARAMETER TUNING
#
# Training data:
#   - 85% of X_train_scaled for fitting
#   - 15% of X_train_scaled for early stopping
#
# External validation:
#   - X_val_scaled used only for candidate comparison
#
# Test set:
#   - Not used
# ============================================================

import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5
INTERNAL_VALIDATION_SIZE = 0.15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "TensorFlow cannot detect the Kaggle GPU. "
        "Confirm that the P100 accelerator is enabled."
    )


# ------------------------------------------------------------
# 2. Verify required arrays
# ------------------------------------------------------------

required_variables = [
    "X_train_scaled",
    "X_val_scaled",
    "y_train",
    "y_val",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun the data preparation and scaling cells first. "
        f"Missing variables: {missing_variables}"
    )

X_train_mlp = np.asarray(
    X_train_scaled,
    dtype=np.float32,
)

X_val_mlp = np.asarray(
    X_val_scaled,
    dtype=np.float32,
)

y_train_mlp = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_mlp = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_mlp.shape == (192593, 78)
assert X_val_mlp.shape == (48149, 78)
assert len(y_train_mlp) == 192593
assert len(y_val_mlp) == 48149

print("\nTraining shape:", X_train_mlp.shape)
print("External validation shape:", X_val_mlp.shape)
print("Test set used: False")


# ------------------------------------------------------------
# 3. Create training-only early-stopping split
# ------------------------------------------------------------

(
    X_fit,
    X_early_stop,
    y_fit,
    y_early_stop,
) = train_test_split(
    X_train_mlp,
    y_train_mlp,
    test_size=INTERNAL_VALIDATION_SIZE,
    stratify=y_train_mlp,
    random_state=RANDOM_STATE,
)

print("\nInternal fitting split:", X_fit.shape)
print("Internal early-stop split:", X_early_stop.shape)
print("External validation split:", X_val_mlp.shape)


# ------------------------------------------------------------
# 4. Output directories
# ------------------------------------------------------------

RUN_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "neural_tuning_validation_clean_v1"
)

RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
PARAMS_DIR = RUN_ROOT / "best_parameters"
HISTORY_DIR = RUN_ROOT / "histories"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    PARAMS_DIR,
    HISTORY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CANDIDATE_RESULTS_PATH = (
    RESULTS_DIR /
    "mlp_candidate_validation_results.csv"
)

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR /
    "neural_tuned_validation_partial.csv"
)

PROBABILITIES_PATH = (
    RESULTS_DIR /
    "mlp_validation_probabilities.csv"
)

BEST_MODEL_PATH = (
    MODELS_DIR /
    "mlp_tuned_validation_safe.keras"
)

BEST_PARAMETERS_PATH = (
    PARAMS_DIR /
    "mlp_best_parameters.json"
)

BEST_HISTORY_PATH = (
    HISTORY_DIR /
    "mlp_best_training_history.csv"
)


# ------------------------------------------------------------
# 5. Candidate configurations
# ------------------------------------------------------------

mlp_candidates = [
    {
        "candidate": "MLP-01",
        "hidden_layers": [128, 64],
        "dropout": 0.20,
        "learning_rate": 0.001,
        "batch_size": 1024,
    },
    {
        "candidate": "MLP-02",
        "hidden_layers": [256, 128],
        "dropout": 0.20,
        "learning_rate": 0.0005,
        "batch_size": 1024,
    },
    {
        "candidate": "MLP-03",
        "hidden_layers": [256, 128, 64],
        "dropout": 0.20,
        "learning_rate": 0.0005,
        "batch_size": 1024,
    },
    {
        "candidate": "MLP-04",
        "hidden_layers": [256, 128, 64],
        "dropout": 0.30,
        "learning_rate": 0.001,
        "batch_size": 512,
    },
    {
        "candidate": "MLP-05",
        "hidden_layers": [512, 256, 128],
        "dropout": 0.30,
        "learning_rate": 0.0005,
        "batch_size": 1024,
    },
    {
        "candidate": "MLP-06",
        "hidden_layers": [512, 256, 128, 64],
        "dropout": 0.30,
        "learning_rate": 0.0003,
        "batch_size": 1024,
    },
]

print("\nMLP candidates:", len(mlp_candidates))


# ------------------------------------------------------------
# 6. Model builder
# ------------------------------------------------------------

def build_mlp(
    input_features,
    hidden_layers,
    dropout,
    learning_rate,
):
    model = tf.keras.Sequential(
        name="validation_safe_mlp"
    )

    model.add(
        tf.keras.layers.Input(
            shape=(input_features,)
        )
    )

    for units in hidden_layers:
        model.add(
            tf.keras.layers.Dense(
                units,
                activation="relu",
                kernel_initializer="he_normal",
            )
        )

        model.add(
            tf.keras.layers.BatchNormalization()
        )

        model.add(
            tf.keras.layers.Dropout(
                dropout
            )
        )

    model.add(
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
        )
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# 7. Metric function
# ------------------------------------------------------------

def calculate_binary_metrics(
    y_true,
    probabilities,
    threshold=0.50,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 8. Candidate search
# ------------------------------------------------------------

candidate_results = []

best_f1 = -1.0
best_recall = -1.0
best_candidate = None
best_history = None

for candidate_number, config in enumerate(
    mlp_candidates,
    start=1,
):
    print("\n" + "=" * 76)
    print(
        f"TRAINING CANDIDATE "
        f"{candidate_number}/{len(mlp_candidates)}"
    )
    print("=" * 76)

    print(json.dumps(config, indent=2))

    tf.keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(
        RANDOM_STATE + candidate_number
    )

    model = build_mlp(
        input_features=X_fit.shape[1],
        hidden_layers=config["hidden_layers"],
        dropout=config["dropout"],
        learning_rate=config["learning_rate"],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            min_delta=1e-4,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    training_start = time.perf_counter()

    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(
            X_early_stop,
            y_early_stop,
        ),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    training_time = (
        time.perf_counter() - training_start
    )

    prediction_start = time.perf_counter()

    validation_probabilities = (
        model.predict(
            X_val_mlp,
            batch_size=4096,
            verbose=0,
        )
        .reshape(-1)
    )

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    metrics = calculate_binary_metrics(
        y_true=y_val_mlp,
        probabilities=validation_probabilities,
        threshold=0.50,
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    result = {
        "Model": "MLP Tuned",
        "Candidate": config["candidate"],
        "Hidden Layers": str(
            config["hidden_layers"]
        ),
        "Dropout": config["dropout"],
        "Learning Rate": config[
            "learning_rate"
        ],
        "Batch Size": config["batch_size"],
        "Best Epoch": best_epoch,
        "Epochs Completed": len(
            history.history["loss"]
        ),
        "Threshold": 0.50,
        "Evaluation Split": "Validation",
        "Training Device": "GPU",
        **metrics,
        "Training Time (sec)": training_time,
        "Prediction Time (sec)": prediction_time,
    }

    candidate_results.append(result)

    candidate_df = (
        pd.DataFrame(candidate_results)
        .sort_values(
            ["F1-score", "Recall"],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )

    candidate_df.to_csv(
        CANDIDATE_RESULTS_PATH,
        index=False,
    )

    print("\nValidation result:")

    display(
        pd.DataFrame([result])[
            [
                "Candidate",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-score",
                "FPR",
                "FNR",
                "Best Epoch",
                "Training Time (sec)",
            ]
        ].round(6)
    )

    candidate_is_better = (
        metrics["F1-score"] > best_f1
        or (
            np.isclose(
                metrics["F1-score"],
                best_f1,
            )
            and metrics["Recall"] > best_recall
        )
    )

    if candidate_is_better:
        best_f1 = metrics["F1-score"]
        best_recall = metrics["Recall"]
        best_candidate = result.copy()

        model.save(
            BEST_MODEL_PATH,
            overwrite=True,
        )

        pd.DataFrame(
            history.history
        ).to_csv(
            BEST_HISTORY_PATH,
            index=False,
        )

        pd.DataFrame({
            "y_true": y_val_mlp,
            "attack_probability":
                validation_probabilities,
        }).to_csv(
            PROBABILITIES_PATH,
            index=False,
        )

        best_history = history.history

        print(
            "\nNew best MLP saved:",
            BEST_MODEL_PATH,
        )

    del model
    del history
    del validation_probabilities

    tf.keras.backend.clear_session()
    gc.collect()


# ------------------------------------------------------------
# 9. Save best parameter record
# ------------------------------------------------------------

best_config_record = {
    "model": "MLP Tuned",
    "selection_metric": (
        "External validation F1-score "
        "at threshold 0.50"
    ),
    "early_stopping_data": (
        "15% stratified subset of training set"
    ),
    "training_records_total": int(
        len(y_train_mlp)
    ),
    "fitting_records": int(len(y_fit)),
    "early_stopping_records": int(
        len(y_early_stop)
    ),
    "external_validation_records": int(
        len(y_val_mlp)
    ),
    "feature_count": int(
        X_train_mlp.shape[1]
    ),
    "test_set_used": False,
    "best_candidate": best_candidate,
}

with open(
    BEST_PARAMETERS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_config_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Add MLP to neural partial results
# ------------------------------------------------------------

best_result_row = {
    key: value
    for key, value in best_candidate.items()
    if key not in {
        "Candidate",
        "Hidden Layers",
        "Dropout",
        "Learning Rate",
        "Batch Size",
        "Best Epoch",
        "Epochs Completed",
    }
}

best_result_row.update({
    "Candidate": best_candidate["Candidate"],
    "Architecture":
        best_candidate["Hidden Layers"],
    "Dropout": best_candidate["Dropout"],
    "Learning Rate":
        best_candidate["Learning Rate"],
    "Batch Size":
        best_candidate["Batch Size"],
    "Best Epoch":
        best_candidate["Best Epoch"],
})

if PARTIAL_RESULTS_PATH.exists():
    neural_partial_df = pd.read_csv(
        PARTIAL_RESULTS_PATH
    )
else:
    neural_partial_df = pd.DataFrame()

if (
    not neural_partial_df.empty
    and "Model" in neural_partial_df.columns
):
    neural_partial_df = neural_partial_df[
        neural_partial_df["Model"]
        != "MLP Tuned"
    ]

neural_partial_df = pd.concat(
    [
        neural_partial_df,
        pd.DataFrame(
            [best_result_row]
        ),
    ],
    ignore_index=True,
)

neural_partial_df = (
    neural_partial_df
    .sort_values(
        ["F1-score", "Recall"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

neural_partial_df.to_csv(
    PARTIAL_RESULTS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. Final display
# ------------------------------------------------------------

candidate_ranking = (
    pd.DataFrame(candidate_results)
    .sort_values(
        ["F1-score", "Recall"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 76)
print("MLP VALIDATION-SAFE TUNING COMPLETED")
print("=" * 76)

display(
    candidate_ranking[
        [
            "Candidate",
            "Hidden Layers",
            "Dropout",
            "Learning Rate",
            "Batch Size",
            "Best Epoch",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
        ]
    ].round(6)
)

print("\nBest MLP candidate:")

display(
    pd.DataFrame(
        [best_candidate]
    ).round(6)
)

print("\nSaved files:")
print(CANDIDATE_RESULTS_PATH)
print(BEST_MODEL_PATH)
print(BEST_PARAMETERS_PATH)
print(BEST_HISTORY_PATH)
print(PROBABILITIES_PATH)
print(PARTIAL_RESULTS_PATH)

print("\nTest set used: False")

In [ ]:
# ============================================================
# VALIDATION-SAFE 1D-CNN HYPERPARAMETER TUNING
#
# Training:
#   - 85% of training set for fitting
#   - 15% of training set for early stopping
#
# Model selection:
#   - External validation F1-score
#
# Test set:
#   - Not used
# ============================================================

import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5
INTERNAL_VALIDATION_SIZE = 0.15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "TensorFlow cannot detect the Kaggle GPU. "
        "Confirm that the P100 accelerator is enabled."
    )


# ------------------------------------------------------------
# 2. Verify and prepare arrays
# ------------------------------------------------------------

required_variables = [
    "X_train_scaled",
    "X_val_scaled",
    "y_train",
    "y_val",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun the data preparation and scaling cells first. "
        f"Missing variables: {missing_variables}"
    )

X_train_cnn = np.asarray(
    X_train_scaled,
    dtype=np.float32,
)

X_val_cnn = np.asarray(
    X_val_scaled,
    dtype=np.float32,
)

y_train_cnn = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_cnn = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_cnn.shape == (192593, 78)
assert X_val_cnn.shape == (48149, 78)
assert len(y_train_cnn) == 192593
assert len(y_val_cnn) == 48149

# Conv1D input: samples × features × channels
X_train_cnn = np.expand_dims(
    X_train_cnn,
    axis=-1,
)

X_val_cnn = np.expand_dims(
    X_val_cnn,
    axis=-1,
)

print("\nTraining shape:", X_train_cnn.shape)
print("External validation shape:", X_val_cnn.shape)
print("Test set used: False")


# ------------------------------------------------------------
# 3. Training-only early-stopping split
# ------------------------------------------------------------

(
    X_fit,
    X_early_stop,
    y_fit,
    y_early_stop,
) = train_test_split(
    X_train_cnn,
    y_train_cnn,
    test_size=INTERNAL_VALIDATION_SIZE,
    stratify=y_train_cnn,
    random_state=RANDOM_STATE,
)

print("\nInternal fitting split:", X_fit.shape)
print("Internal early-stop split:", X_early_stop.shape)
print("External validation split:", X_val_cnn.shape)


# ------------------------------------------------------------
# 4. Output directories
# ------------------------------------------------------------

RUN_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "neural_tuning_validation_clean_v1"
)

RESULTS_DIR = RUN_ROOT / "results"
MODELS_DIR = RUN_ROOT / "models"
PARAMS_DIR = RUN_ROOT / "best_parameters"
HISTORY_DIR = RUN_ROOT / "histories"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    PARAMS_DIR,
    HISTORY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CANDIDATE_RESULTS_PATH = (
    RESULTS_DIR /
    "cnn_candidate_validation_results.csv"
)

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR /
    "neural_tuned_validation_partial.csv"
)

PROBABILITIES_PATH = (
    RESULTS_DIR /
    "cnn_validation_probabilities.csv"
)

BEST_MODEL_PATH = (
    MODELS_DIR /
    "cnn_tuned_validation_safe.keras"
)

BEST_PARAMETERS_PATH = (
    PARAMS_DIR /
    "cnn_best_parameters.json"
)

BEST_HISTORY_PATH = (
    HISTORY_DIR /
    "cnn_best_training_history.csv"
)


# ------------------------------------------------------------
# 5. Candidate configurations
# ------------------------------------------------------------

cnn_candidates = [
    {
        "candidate": "CNN-01",
        "filters": [64, 128],
        "kernel_size": 3,
        "dropout": 0.20,
        "dense_units": 64,
        "learning_rate": 0.001,
        "batch_size": 1024,
    },
    {
        "candidate": "CNN-02",
        "filters": [128, 256],
        "kernel_size": 3,
        "dropout": 0.30,
        "dense_units": 128,
        "learning_rate": 0.0005,
        "batch_size": 1024,
    },
    {
        "candidate": "CNN-03",
        "filters": [128, 256],
        "kernel_size": 5,
        "dropout": 0.30,
        "dense_units": 128,
        "learning_rate": 0.0005,
        "batch_size": 512,
    },
    {
        "candidate": "CNN-04",
        "filters": [64, 128, 256],
        "kernel_size": 3,
        "dropout": 0.30,
        "dense_units": 128,
        "learning_rate": 0.0005,
        "batch_size": 1024,
    },
    {
        "candidate": "CNN-05",
        "filters": [128, 256, 256],
        "kernel_size": 5,
        "dropout": 0.40,
        "dense_units": 128,
        "learning_rate": 0.0003,
        "batch_size": 512,
    },
    {
        "candidate": "CNN-06",
        "filters": [256, 256],
        "kernel_size": 5,
        "dropout": 0.30,
        "dense_units": 256,
        "learning_rate": 0.0003,
        "batch_size": 1024,
    },
]

print("\nCNN candidates:", len(cnn_candidates))


# ------------------------------------------------------------
# 6. CNN builder
# ------------------------------------------------------------

def build_cnn(
    input_shape,
    filters,
    kernel_size,
    dropout,
    dense_units,
    learning_rate,
):
    inputs = tf.keras.layers.Input(
        shape=input_shape,
        name="traffic_features",
    )

    x = inputs

    for block_number, filter_count in enumerate(
        filters,
        start=1,
    ):
        x = tf.keras.layers.Conv1D(
            filters=filter_count,
            kernel_size=kernel_size,
            padding="same",
            activation=None,
            kernel_initializer="he_normal",
            name=f"conv_{block_number}",
        )(x)

        x = tf.keras.layers.BatchNormalization(
            name=f"batch_norm_{block_number}",
        )(x)

        x = tf.keras.layers.Activation(
            "relu",
            name=f"relu_{block_number}",
        )(x)

        # Pool only when enough sequence positions remain
        if block_number < len(filters):
            x = tf.keras.layers.MaxPooling1D(
                pool_size=2,
                name=f"pool_{block_number}",
            )(x)

        x = tf.keras.layers.Dropout(
            dropout,
            name=f"conv_dropout_{block_number}",
        )(x)

    x = tf.keras.layers.GlobalAveragePooling1D(
        name="global_average_pooling",
    )(x)

    x = tf.keras.layers.Dense(
        dense_units,
        activation="relu",
        kernel_initializer="he_normal",
        name="dense_hidden",
    )(x)

    x = tf.keras.layers.BatchNormalization(
        name="dense_batch_norm",
    )(x)

    x = tf.keras.layers.Dropout(
        dropout,
        name="dense_dropout",
    )(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        name="attack_probability",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="validation_safe_1d_cnn",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# 7. Metric function
# ------------------------------------------------------------

def calculate_binary_metrics(
    y_true,
    probabilities,
    threshold=0.50,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 8. Candidate search
# ------------------------------------------------------------

candidate_results = []

best_f1 = -1.0
best_recall = -1.0
best_candidate = None

for candidate_number, config in enumerate(
    cnn_candidates,
    start=1,
):
    print("\n" + "=" * 76)
    print(
        f"TRAINING CNN CANDIDATE "
        f"{candidate_number}/{len(cnn_candidates)}"
    )
    print("=" * 76)

    print(json.dumps(config, indent=2))

    tf.keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(
        RANDOM_STATE + candidate_number
    )

    model = build_cnn(
        input_shape=X_fit.shape[1:],
        filters=config["filters"],
        kernel_size=config["kernel_size"],
        dropout=config["dropout"],
        dense_units=config["dense_units"],
        learning_rate=config["learning_rate"],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            min_delta=1e-4,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    training_start = time.perf_counter()

    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(
            X_early_stop,
            y_early_stop,
        ),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    training_time = (
        time.perf_counter() - training_start
    )

    prediction_start = time.perf_counter()

    validation_probabilities = (
        model.predict(
            X_val_cnn,
            batch_size=4096,
            verbose=0,
        )
        .reshape(-1)
    )

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    metrics = calculate_binary_metrics(
        y_true=y_val_cnn,
        probabilities=validation_probabilities,
        threshold=0.50,
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    result = {
        "Model": "1D-CNN Tuned",
        "Candidate": config["candidate"],
        "Filters": str(config["filters"]),
        "Kernel Size": config["kernel_size"],
        "Dropout": config["dropout"],
        "Dense Units": config["dense_units"],
        "Learning Rate": config[
            "learning_rate"
        ],
        "Batch Size": config["batch_size"],
        "Best Epoch": best_epoch,
        "Epochs Completed": len(
            history.history["loss"]
        ),
        "Threshold": 0.50,
        "Evaluation Split": "Validation",
        "Training Device": "GPU",
        **metrics,
        "Training Time (sec)": training_time,
        "Prediction Time (sec)": prediction_time,
    }

    candidate_results.append(result)

    candidate_df = (
        pd.DataFrame(candidate_results)
        .sort_values(
            ["F1-score", "Recall"],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )

    candidate_df.to_csv(
        CANDIDATE_RESULTS_PATH,
        index=False,
    )

    print("\nValidation result:")

    display(
        pd.DataFrame([result])[
            [
                "Candidate",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-score",
                "FPR",
                "FNR",
                "Best Epoch",
                "Training Time (sec)",
            ]
        ].round(6)
    )

    candidate_is_better = (
        metrics["F1-score"] > best_f1
        or (
            np.isclose(
                metrics["F1-score"],
                best_f1,
            )
            and metrics["Recall"] > best_recall
        )
    )

    if candidate_is_better:
        best_f1 = metrics["F1-score"]
        best_recall = metrics["Recall"]
        best_candidate = result.copy()

        model.save(
            BEST_MODEL_PATH,
            overwrite=True,
        )

        pd.DataFrame(
            history.history
        ).to_csv(
            BEST_HISTORY_PATH,
            index=False,
        )

        pd.DataFrame({
            "y_true": y_val_cnn,
            "attack_probability":
                validation_probabilities,
        }).to_csv(
            PROBABILITIES_PATH,
            index=False,
        )

        print(
            "\nNew best CNN saved:",
            BEST_MODEL_PATH,
        )

    del model
    del history
    del validation_probabilities

    tf.keras.backend.clear_session()
    gc.collect()


# ------------------------------------------------------------
# 9. Save best parameter record
# ------------------------------------------------------------

best_config_record = {
    "model": "1D-CNN Tuned",
    "selection_metric": (
        "External validation F1-score "
        "at threshold 0.50"
    ),
    "early_stopping_data": (
        "15% stratified subset of training set"
    ),
    "training_records_total": int(
        len(y_train_cnn)
    ),
    "fitting_records": int(len(y_fit)),
    "early_stopping_records": int(
        len(y_early_stop)
    ),
    "external_validation_records": int(
        len(y_val_cnn)
    ),
    "feature_count": int(
        X_train_cnn.shape[1]
    ),
    "input_shape": list(
        X_train_cnn.shape[1:]
    ),
    "test_set_used": False,
    "best_candidate": best_candidate,
}

with open(
    BEST_PARAMETERS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_config_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Append best CNN to neural checkpoint
# ------------------------------------------------------------

best_result_row = {
    key: value
    for key, value in best_candidate.items()
    if key not in {
        "Candidate",
        "Filters",
        "Kernel Size",
        "Dropout",
        "Dense Units",
        "Learning Rate",
        "Batch Size",
        "Best Epoch",
        "Epochs Completed",
    }
}

best_result_row.update({
    "Candidate":
        best_candidate["Candidate"],
    "Architecture":
        best_candidate["Filters"],
    "Kernel Size":
        best_candidate["Kernel Size"],
    "Dropout":
        best_candidate["Dropout"],
    "Dense Units":
        best_candidate["Dense Units"],
    "Learning Rate":
        best_candidate["Learning Rate"],
    "Batch Size":
        best_candidate["Batch Size"],
    "Best Epoch":
        best_candidate["Best Epoch"],
})

if PARTIAL_RESULTS_PATH.exists():
    neural_partial_df = pd.read_csv(
        PARTIAL_RESULTS_PATH
    )
else:
    neural_partial_df = pd.DataFrame()

if (
    not neural_partial_df.empty
    and "Model" in neural_partial_df.columns
):
    neural_partial_df = neural_partial_df[
        neural_partial_df["Model"]
        != "1D-CNN Tuned"
    ]

neural_partial_df = pd.concat(
    [
        neural_partial_df,
        pd.DataFrame(
            [best_result_row]
        ),
    ],
    ignore_index=True,
)

neural_partial_df = (
    neural_partial_df
    .sort_values(
        ["F1-score", "Recall"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

neural_partial_df.to_csv(
    PARTIAL_RESULTS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. Final ranking
# ------------------------------------------------------------

candidate_ranking = (
    pd.DataFrame(candidate_results)
    .sort_values(
        ["F1-score", "Recall"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 76)
print("1D-CNN VALIDATION-SAFE TUNING COMPLETED")
print("=" * 76)

display(
    candidate_ranking[
        [
            "Candidate",
            "Filters",
            "Kernel Size",
            "Dropout",
            "Dense Units",
            "Learning Rate",
            "Batch Size",
            "Best Epoch",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
        ]
    ].round(6)
)

print("\nBest CNN candidate:")

display(
    pd.DataFrame(
        [best_candidate]
    ).round(6)
)

print("\nSaved files:")
print(CANDIDATE_RESULTS_PATH)
print(BEST_MODEL_PATH)
print(BEST_PARAMETERS_PATH)
print(BEST_HISTORY_PATH)
print(PROBABILITIES_PATH)
print(PARTIAL_RESULTS_PATH)

print("\nTest set used: False")

In [ ]:
# ============================================================
# MERGE THE FIVE VALIDATION-SAFE TUNED MODELS
#
# Models:
#   - XGBoost
#   - LightGBM
#   - CatBoost
#   - MLP
#   - 1D-CNN
#
# Test set:
#   - Not used
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. Input paths
# ------------------------------------------------------------

BOOSTING_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

NEURAL_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "neural_tuning_validation_clean_v1"
)

boosting_path = (
    BOOSTING_ROOT
    / "results"
    / "boosting_tuned_validation_results.csv"
)

# Fall back to the checkpoint file if needed
if not boosting_path.exists():
    boosting_path = (
        BOOSTING_ROOT
        / "results"
        / "boosting_tuned_validation_partial.csv"
    )

neural_path = (
    NEURAL_ROOT
    / "results"
    / "neural_tuned_validation_partial.csv"
)


# ------------------------------------------------------------
# 2. Verify files
# ------------------------------------------------------------

required_files = {
    "Boosting results": boosting_path,
    "Neural results": neural_path,
}

for label, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )

    print(f"FOUND: {label}")
    print(path)


# ------------------------------------------------------------
# 3. Load result tables
# ------------------------------------------------------------

boosting_df = pd.read_csv(boosting_path)
neural_df = pd.read_csv(neural_path)

print("\nBoosting rows:", len(boosting_df))
print("Neural rows:", len(neural_df))


# ------------------------------------------------------------
# 4. Expected models
# ------------------------------------------------------------

expected_models = [
    "XGBoost Tuned",
    "LightGBM Tuned",
    "CatBoost Tuned",
    "MLP Tuned",
    "1D-CNN Tuned",
]

combined_df = pd.concat(
    [
        boosting_df,
        neural_df,
    ],
    ignore_index=True,
    sort=False,
)

combined_df = combined_df[
    combined_df["Model"].isin(expected_models)
].copy()

# Keep the most recently stored row for each model
combined_df = combined_df.drop_duplicates(
    subset=["Model"],
    keep="last",
)

found_models = set(combined_df["Model"])
missing_models = set(expected_models) - found_models

if missing_models:
    raise ValueError(
        "The following tuned models are missing: "
        f"{sorted(missing_models)}"
    )


# ------------------------------------------------------------
# 5. Standardize essential columns
# ------------------------------------------------------------

essential_columns = [
    "Model",
    "Evaluation Split",
    "Training Device",
    "Threshold",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "FPR",
    "FNR",
    "TP",
    "TN",
    "FP",
    "FN",
    "Training Time (sec)",
    "Prediction Time (sec)",
]

for column in essential_columns:
    if column not in combined_df.columns:
        combined_df[column] = np.nan

combined_df["Evaluation Split"] = "Validation"
combined_df["Threshold"] = 0.50

final_df = (
    combined_df[essential_columns]
    .sort_values(
        by=[
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

final_df.insert(
    0,
    "Rank",
    range(1, len(final_df) + 1),
)


# ------------------------------------------------------------
# 6. Output directory
# ------------------------------------------------------------

OUTPUT_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "top5_tuned_validation_clean_v1"
)

RESULTS_DIR = OUTPUT_ROOT / "results"
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

final_path = (
    RESULTS_DIR
    / "top5_tuned_validation_results.csv"
)

final_df.to_csv(
    final_path,
    index=False,
)


# ------------------------------------------------------------
# 7. Verify saved validation probability files
# ------------------------------------------------------------

probability_files = {
    "XGBoost Tuned": (
        BOOSTING_ROOT
        / "results"
        / "xgboost_validation_probabilities.csv"
    ),
    "LightGBM Tuned": (
        BOOSTING_ROOT
        / "results"
        / "lightgbm_validation_probabilities.csv"
    ),
    "CatBoost Tuned": (
        BOOSTING_ROOT
        / "results"
        / "catboost_validation_probabilities.csv"
    ),
    "MLP Tuned": (
        NEURAL_ROOT
        / "results"
        / "mlp_validation_probabilities.csv"
    ),
    "1D-CNN Tuned": (
        NEURAL_ROOT
        / "results"
        / "cnn_validation_probabilities.csv"
    ),
}

probability_manifest = pd.DataFrame([
    {
        "Model": model_name,
        "Probability File": str(path),
        "Available": path.exists(),
    }
    for model_name, path in probability_files.items()
])

manifest_path = (
    RESULTS_DIR
    / "validation_probability_manifest.csv"
)

probability_manifest.to_csv(
    manifest_path,
    index=False,
)


# ------------------------------------------------------------
# 8. Display final comparison
# ------------------------------------------------------------

print("\n" + "=" * 82)
print("FINAL FIVE-MODEL TUNED VALIDATION COMPARISON")
print("=" * 82)

display(
    final_df[
        [
            "Rank",
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
        ]
    ].round(6)
)

print("\nValidation-probability files:")

display(probability_manifest)

print("\nSaved:")
print(final_path)
print(manifest_path)

print("\nHighest validation F1 model:")
print(final_df.loc[0, "Model"])

print("\nTest set used: False")

In [ ]:
# ============================================================
# LIGHTGBM-ONLY VALIDATION THRESHOLD SELECTION
#
# Operating points:
#   1. Standard threshold = 0.50
#   2. Best validation F1 threshold
#   3. Security-oriented best validation F2 threshold
#
# Test set:
#   - Not used
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BOOSTING_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

PROBABILITY_PATH = (
    BOOSTING_ROOT
    / "results"
    / "lightgbm_validation_probabilities.csv"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/revised_experiments/"
    "lightgbm_threshold_validation_clean_v1"
)

RESULTS_DIR = OUTPUT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SWEEP_PATH = (
    RESULTS_DIR
    / "lightgbm_validation_threshold_sweep.csv"
)

SELECTED_PATH = (
    RESULTS_DIR
    / "lightgbm_selected_validation_operating_points.csv"
)


# ------------------------------------------------------------
# 2. Load saved validation probabilities
# ------------------------------------------------------------

if not PROBABILITY_PATH.exists():
    raise FileNotFoundError(
        f"LightGBM probability file not found:\n{PROBABILITY_PATH}"
    )

probability_df = pd.read_csv(PROBABILITY_PATH)

required_columns = {
    "y_true",
    "attack_probability",
}

missing_columns = (
    required_columns
    - set(probability_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing columns: {sorted(missing_columns)}"
    )

y_true = np.asarray(
    probability_df["y_true"],
    dtype=np.int32,
).reshape(-1)

probabilities = np.asarray(
    probability_df["attack_probability"],
    dtype=np.float64,
).reshape(-1)

assert len(y_true) == 48149
assert len(probabilities) == 48149

if np.isnan(probabilities).any():
    raise ValueError("Probabilities contain NaN values.")

if (
    (probabilities < 0).any()
    or (probabilities > 1).any()
):
    raise ValueError(
        "Probabilities must remain between 0 and 1."
    )

print("Validation records:", len(y_true))
print("Benign records:", int((y_true == 0).sum()))
print("Attack records:", int((y_true == 1).sum()))
print("Test set used: False")


# ------------------------------------------------------------
# 3. Threshold metric function
# ------------------------------------------------------------

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F2-score": fbeta_score(
            y_true,
            predictions,
            beta=2,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 4. Threshold sweep
# ------------------------------------------------------------

thresholds = np.round(
    np.arange(
        0.05,
        0.951,
        0.01,
    ),
    2,
)

results = []

for threshold in thresholds:
    results.append(
        calculate_metrics(
            y_true=y_true,
            probabilities=probabilities,
            threshold=threshold,
        )
    )

sweep_df = pd.DataFrame(results)

sweep_df.to_csv(
    SWEEP_PATH,
    index=False,
)


# ------------------------------------------------------------
# 5. Standard threshold 0.50
# ------------------------------------------------------------

standard_row = (
    sweep_df[
        np.isclose(
            sweep_df["Threshold"],
            0.50,
        )
    ]
    .iloc[0]
    .copy()
)


# ------------------------------------------------------------
# 6. Highest-F1 threshold
# ------------------------------------------------------------

best_f1_row = (
    sweep_df
    .sort_values(
        by=[
            "F1-score",
            "Recall",
            "FPR",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .iloc[0]
    .copy()
)


# ------------------------------------------------------------
# 7. Security-oriented highest-F2 threshold
# ------------------------------------------------------------

best_f2_row = (
    sweep_df
    .sort_values(
        by=[
            "F2-score",
            "F1-score",
            "Recall",
            "FPR",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            False,
        ],
    )
    .iloc[0]
    .copy()
)


# ------------------------------------------------------------
# 8. Save selected operating points
# ------------------------------------------------------------

selected_df = pd.DataFrame([
    {
        "Operating Point": "Standard",
        **standard_row.to_dict(),
    },
    {
        "Operating Point": "Maximum Validation F1",
        **best_f1_row.to_dict(),
    },
    {
        "Operating Point": "Security-Oriented Maximum F2",
        **best_f2_row.to_dict(),
    },
])

selected_df.to_csv(
    SELECTED_PATH,
    index=False,
)


# ------------------------------------------------------------
# 9. Calculate changes relative to threshold 0.50
# ------------------------------------------------------------

standard_fn = int(standard_row["FN"])
standard_fp = int(standard_row["FP"])

selected_df[
    "False Negatives Reduced vs 0.50"
] = (
    standard_fn
    - selected_df["FN"].astype(int)
)

selected_df[
    "Additional False Positives vs 0.50"
] = (
    selected_df["FP"].astype(int)
    - standard_fp
)

selected_df.to_csv(
    SELECTED_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Display final selection table
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("LIGHTGBM VALIDATION THRESHOLD SELECTION")
print("=" * 92)

display(
    selected_df[
        [
            "Operating Point",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "False Negatives Reduced vs 0.50",
            "Additional False Positives vs 0.50",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 11. Show nearby thresholds for interpretation
# ------------------------------------------------------------

selected_thresholds = {
    float(standard_row["Threshold"]),
    float(best_f1_row["Threshold"]),
    float(best_f2_row["Threshold"]),
}

nearby_thresholds = sorted({
    round(max(0.05, value - 0.02), 2)
    for value in selected_thresholds
} | selected_thresholds | {
    round(min(0.95, value + 0.02), 2)
    for value in selected_thresholds
})

nearby_df = sweep_df[
    sweep_df["Threshold"].isin(
        nearby_thresholds
    )
].copy()

print("\nRelevant neighboring thresholds:")

display(
    nearby_df[
        [
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "FP",
            "FN",
        ]
    ].round(6)
)


print("\nSaved:")
print(SWEEP_PATH)
print(SELECTED_PATH)

print("\nTest set used: False")

In [ ]:
from pathlib import Path

search_names = [
    "lightgbm_validation_probabilities.csv",
    "boosting_tuned_validation_partial.csv",
    "boosting_tuned_validation_results.csv",
    "neural_tuned_validation_partial.csv",
    "top5_tuned_validation_results.csv",
    "mlp_validation_probabilities.csv",
    "cnn_validation_probabilities.csv",
    "catboost_validation_probabilities.csv",
    "xgboost_validation_probabilities.csv",
    "lightgbm_tuned.pkl",
    "lightgbm_tuned.txt",
]

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

found = []

for root in roots:
    if not root.exists():
        continue

    for name in search_names:
        for path in root.rglob(name):
            found.append(path)
            print("FOUND:", path)

if not found:
    print("No revised experiment files found in mounted storage.")

In [ ]:
from pathlib import Path

BACKUP_ROOT = Path("/kaggle/working/FINAL_REVISED_LIGHTGBM")
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

print(BACKUP_ROOT)

In [ ]:
import shutil
from pathlib import Path

source = Path(
    "/kaggle/working/revised_experiments/"
    "boosting_tuning_validation_clean_v1"
)

backup = Path(
    "/kaggle/working/FINAL_REVISED_LIGHTGBM/"
    "boosting_tuning_validation_clean_v1"
)

if source.exists():
    if backup.exists():
        shutil.rmtree(backup)

    shutil.copytree(source, backup)
    print("Backup created:", backup)
else:
    print("Source directory not found.")

In [ ]:
X_train
X_val
X_test

y_train
y_val
y_test

In [ ]:
# ============================================================
# STAGE 1 — DATA PREPARATION AND 64/16/20 SPLIT
#
# Training:   64%
# Validation: 16%
# Test:       20%
#
# The scaler is fitted on training data only.
# The test set is not used for model selection.
# ============================================================

import json
import hashlib
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from IPython.display import display


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = ROOT / "stage01_data"

folders = [
    STAGE_DIR,
    ROOT / "stage02_baselines",
    ROOT / "stage03_top5_tuning" / "xgboost",
    ROOT / "stage03_top5_tuning" / "lightgbm",
    ROOT / "stage03_top5_tuning" / "catboost",
    ROOT / "stage03_top5_tuning" / "mlp",
    ROOT / "stage03_top5_tuning" / "cnn",
    ROOT / "stage04_threshold",
    ROOT / "stage05_final_test",
    ROOT / "stage06_shap",
    ROOT / "models",
    ROOT / "figures",
]

for folder in folders:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Experiment root:")
print(ROOT)


# ------------------------------------------------------------
# 2. Locate dataset
# ------------------------------------------------------------

dataset_matches = list(
    Path("/kaggle/input").rglob(
        "merged_balanced_ids2018_safe.csv"
    )
)

if not dataset_matches:
    raise FileNotFoundError(
        "merged_balanced_ids2018_safe.csv "
        "was not found under /kaggle/input."
    )

print("\nMatching dataset files:")

for index, path in enumerate(
    dataset_matches,
    start=1,
):
    print(f"{index}. {path}")

# Use the first match.
# Change this manually when multiple datasets are mounted
# and the first one is not the intended file.
DATASET_PATH = dataset_matches[0]

print("\nSelected dataset:")
print(DATASET_PATH)


# ------------------------------------------------------------
# 3. Load processed dataset
# ------------------------------------------------------------

df = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

print("\nDataset shape:", df.shape)

if df.shape[0] != 300_928:
    raise ValueError(
        f"Expected 300,928 rows, "
        f"found {df.shape[0]:,}."
    )

if "binary_label" not in df.columns:
    raise ValueError(
        "Target column 'binary_label' "
        "was not found."
    )


# ------------------------------------------------------------
# 4. Define features and target
# ------------------------------------------------------------

excluded_columns = [
    column
    for column in [
        "Label",
        "binary_label",
    ]
    if column in df.columns
]

feature_columns = [
    column
    for column in df.columns
    if column not in excluded_columns
]

if len(feature_columns) != 78:
    raise ValueError(
        f"Expected 78 predictor features, "
        f"found {len(feature_columns)}."
    )

print("\nExcluded columns:")
print(excluded_columns)

print("\nNumber of predictor features:")
print(len(feature_columns))


# ------------------------------------------------------------
# 5. Validate target
# ------------------------------------------------------------

y = pd.to_numeric(
    df["binary_label"],
    errors="raise",
).to_numpy(
    dtype=np.int32,
)

unique_labels = np.unique(y)

if not np.array_equal(
    unique_labels,
    np.array([0, 1]),
):
    raise ValueError(
        f"Expected labels [0, 1], "
        f"found {unique_labels.tolist()}."
    )


# ------------------------------------------------------------
# 6. Validate predictors
# ------------------------------------------------------------

non_numeric_columns = [
    column
    for column in feature_columns
    if not pd.api.types.is_numeric_dtype(
        df[column]
    )
]

if non_numeric_columns:
    raise TypeError(
        "Non-numeric predictor columns found:\n"
        f"{non_numeric_columns}"
    )

X = df[
    feature_columns
].to_numpy(
    dtype=np.float32,
)

if np.isnan(X).any():
    raise ValueError(
        "Predictor matrix contains NaN values."
    )

if np.isinf(X).any():
    raise ValueError(
        "Predictor matrix contains infinite values."
    )

print("\nPredictor matrix:", X.shape)
print("Target vector:", y.shape)


# ------------------------------------------------------------
# 7. Create deterministic split indices
# ------------------------------------------------------------

all_indices = np.arange(
    len(y),
    dtype=np.int64,
)

# First split:
# 80% train-validation, 20% final test
train_val_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Second split:
# 80% of train-validation becomes training
# 20% becomes validation
#
# Overall:
# training   = 0.80 × 0.80 = 0.64
# validation = 0.80 × 0.20 = 0.16
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.20,
    stratify=y[train_val_indices],
    random_state=RANDOM_STATE,
)


# ------------------------------------------------------------
# 8. Verify split integrity
# ------------------------------------------------------------

assert len(
    np.intersect1d(
        train_indices,
        val_indices,
    )
) == 0

assert len(
    np.intersect1d(
        train_indices,
        test_indices,
    )
) == 0

assert len(
    np.intersect1d(
        val_indices,
        test_indices,
    )
) == 0

all_split_indices = np.concatenate([
    train_indices,
    val_indices,
    test_indices,
])

assert len(
    np.unique(all_split_indices)
) == len(y)


# ------------------------------------------------------------
# 9. Create unscaled arrays
# ------------------------------------------------------------

X_train = X[train_indices]
X_val = X[val_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_val = y[val_indices]
y_test = y[test_indices]

expected_shapes = {
    "X_train": (192_593, 78),
    "X_val": (48_149, 78),
    "X_test": (60_186, 78),
}

actual_shapes = {
    "X_train": X_train.shape,
    "X_val": X_val.shape,
    "X_test": X_test.shape,
}

for name, expected in expected_shapes.items():
    actual = actual_shapes[name]

    if actual != expected:
        raise ValueError(
            f"{name}: expected {expected}, "
            f"found {actual}."
        )


# ------------------------------------------------------------
# 10. Scale using training data only
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
).astype(np.float32)

X_val_scaled = scaler.transform(
    X_val
).astype(np.float32)

X_test_scaled = scaler.transform(
    X_test
).astype(np.float32)


# ------------------------------------------------------------
# 11. Deep-learning shaped arrays
# ------------------------------------------------------------

X_train_deep = np.expand_dims(
    X_train_scaled,
    axis=-1,
)

X_val_deep = np.expand_dims(
    X_val_scaled,
    axis=-1,
)

X_test_deep = np.expand_dims(
    X_test_scaled,
    axis=-1,
)


# ------------------------------------------------------------
# 12. Class-distribution report
# ------------------------------------------------------------

def summarize_split(
    split_name,
    labels,
):
    benign = int(
        np.sum(labels == 0)
    )

    attack = int(
        np.sum(labels == 1)
    )

    return {
        "Split": split_name,
        "Records": int(len(labels)),
        "Benign": benign,
        "Attack": attack,
        "Benign Ratio": benign / len(labels),
        "Attack Ratio": attack / len(labels),
    }


split_summary = pd.DataFrame([
    summarize_split(
        "Complete dataset",
        y,
    ),
    summarize_split(
        "Training",
        y_train,
    ),
    summarize_split(
        "Validation",
        y_val,
    ),
    summarize_split(
        "Test",
        y_test,
    ),
])


# ------------------------------------------------------------
# 13. Save reproducibility artifacts
# ------------------------------------------------------------

np.savez_compressed(
    STAGE_DIR / "split_indices.npz",
    train_indices=train_indices,
    validation_indices=val_indices,
    test_indices=test_indices,
)

joblib.dump(
    scaler,
    STAGE_DIR / "standard_scaler.joblib",
)

with open(
    STAGE_DIR / "feature_names.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_columns,
        file,
        indent=2,
    )

feature_signature = hashlib.sha256(
    "\n".join(
        feature_columns
    ).encode("utf-8")
).hexdigest()

metadata = {
    "experiment_name":
        "IDS2018 Clean Validation V2",
    "dataset_path":
        str(DATASET_PATH),
    "dataset_rows":
        int(df.shape[0]),
    "dataset_columns":
        int(df.shape[1]),
    "predictor_features":
        int(len(feature_columns)),
    "target_column":
        "binary_label",
    "excluded_columns":
        excluded_columns,
    "random_state":
        RANDOM_STATE,
    "training_ratio":
        0.64,
    "validation_ratio":
        0.16,
    "test_ratio":
        0.20,
    "training_records":
        int(len(y_train)),
    "validation_records":
        int(len(y_val)),
    "test_records":
        int(len(y_test)),
    "training_benign":
        int(np.sum(y_train == 0)),
    "training_attack":
        int(np.sum(y_train == 1)),
    "validation_benign":
        int(np.sum(y_val == 0)),
    "validation_attack":
        int(np.sum(y_val == 1)),
    "test_benign":
        int(np.sum(y_test == 0)),
    "test_attack":
        int(np.sum(y_test == 1)),
    "scaler_fit_split":
        "Training only",
    "feature_signature_sha256":
        feature_signature,
    "test_used_for_selection":
        False,
}

with open(
    STAGE_DIR / "split_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

split_summary.to_csv(
    STAGE_DIR / "split_summary.csv",
    index=False,
)


# ------------------------------------------------------------
# 14. Verification display
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STAGE 1 COMPLETED")
print("=" * 80)

display(
    split_summary.round(6)
)

print("\nUnscaled arrays:")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\nScaled arrays:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:  ", X_val_scaled.shape)
print("X_test_scaled: ", X_test_scaled.shape)

print("\nDeep-learning arrays:")
print("X_train_deep:", X_train_deep.shape)
print("X_val_deep:  ", X_val_deep.shape)
print("X_test_deep: ", X_test_deep.shape)

print("\nFeature count:")
print(len(feature_columns))

print("\nTest set used for model selection:")
print(False)


# ------------------------------------------------------------
# 15. Create archive
# ------------------------------------------------------------

archive_path = shutil.make_archive(
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage01",
    "zip",
    STAGE_DIR,
)

print("\nSaved Stage 1 files:")

for path in sorted(
    STAGE_DIR.iterdir()
):
    print(path)

print("\nArchive created:")
print(archive_path)

In [ ]:
# ============================================================
# STAGE 2A — 12 CLASSICAL AND BOOSTING BASELINE MODELS
#
# Training:
#   X_train / X_train_scaled
#
# Evaluation:
#   X_val / X_val_scaled
#
# Test set:
#   Not used
#
# Models:
#   1. Logistic Regression
#   2. Naive Bayes
#   3. KNN
#   4. Linear SVM
#   5. Decision Tree
#   6. Random Forest
#   7. Extra Trees
#   8. AdaBoost
#   9. Gradient Boosting
#  10. XGBoost
#  11. LightGBM
#  12. CatBoost
# ============================================================

import gc
import json
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

import xgboost
import lightgbm
import catboost

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Verify Stage 1 arrays
# ------------------------------------------------------------

required_variables = [
    "X_train",
    "X_val",
    "X_train_scaled",
    "X_val_scaled",
    "y_train",
    "y_val",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Stage 1 arrays are missing. Rerun Stage 1 first.\n"
        f"Missing: {missing_variables}"
    )

X_train_raw = np.asarray(
    X_train,
    dtype=np.float32,
)

X_val_raw = np.asarray(
    X_val,
    dtype=np.float32,
)

X_train_std = np.asarray(
    X_train_scaled,
    dtype=np.float32,
)

X_val_std = np.asarray(
    X_val_scaled,
    dtype=np.float32,
)

y_train_baseline = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_baseline = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_raw.shape == (192_593, 78)
assert X_val_raw.shape == (48_149, 78)
assert X_train_std.shape == (192_593, 78)
assert X_val_std.shape == (48_149, 78)

print("Training:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test set used:", False)


# ------------------------------------------------------------
# 3. Output directories
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = ROOT / "stage02_baselines"
RESULTS_DIR = STAGE_DIR / "results"
MODELS_DIR = STAGE_DIR / "models"
CONFIG_DIR = STAGE_DIR / "configurations"

for directory in [
    STAGE_DIR,
    RESULTS_DIR,
    MODELS_DIR,
    CONFIG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR /
    "baseline12_validation_partial.csv"
)

FINAL_RESULTS_PATH = (
    RESULTS_DIR /
    "baseline12_validation_results.csv"
)

ENVIRONMENT_PATH = (
    CONFIG_DIR /
    "environment_versions.json"
)

MODEL_CONFIG_PATH = (
    CONFIG_DIR /
    "baseline_model_configurations.json"
)


# ------------------------------------------------------------
# 4. Record environment
# ------------------------------------------------------------

environment = {
    "python_hash_seed": RANDOM_STATE,
    "random_state": RANDOM_STATE,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "lightgbm": lightgbm.__version__,
    "catboost": catboost.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "training_records": int(
        len(y_train_baseline)
    ),
    "validation_records": int(
        len(y_val_baseline)
    ),
    "feature_count": int(
        X_train_raw.shape[1]
    ),
    "test_set_used": False,
}

with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 5. Detect XGBoost GPU support
# ------------------------------------------------------------

xgb_device_label = "CPU"
xgb_device_parameters = {
    "tree_method": "hist",
}

try:
    major_version = int(
        xgboost.__version__.split(".")[0]
    )

    if major_version >= 2:
        candidate_parameters = {
            "tree_method": "hist",
            "device": "cuda",
        }
    else:
        candidate_parameters = {
            "tree_method": "gpu_hist",
            "predictor": "gpu_predictor",
        }

    probe = XGBClassifier(
        n_estimators=5,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        **candidate_parameters,
    )

    probe.fit(
        X_train_raw[:5_000],
        y_train_baseline[:5_000],
    )

    xgb_device_parameters = candidate_parameters
    xgb_device_label = "GPU"

    del probe
    gc.collect()

except Exception as error:
    print(
        "XGBoost GPU probe failed. "
        "Using CPU histogram training."
    )
    print(str(error)[:250])


# ------------------------------------------------------------
# 6. Detect LightGBM backend
# ------------------------------------------------------------

lightgbm_backend = "cpu"
lightgbm_device_label = "CPU"

for candidate_backend in [
    "gpu",
    "cuda",
]:
    try:
        probe = LGBMClassifier(
            n_estimators=5,
            device_type=candidate_backend,
            random_state=RANDOM_STATE,
            verbosity=-1,
        )

        probe.fit(
            X_train_raw[:5_000],
            y_train_baseline[:5_000],
        )

        lightgbm_backend = candidate_backend
        lightgbm_device_label = "GPU"

        del probe
        gc.collect()
        break

    except Exception:
        continue


# ------------------------------------------------------------
# 7. Detect CatBoost GPU support
# ------------------------------------------------------------

catboost_gpu_count = get_gpu_device_count()

if catboost_gpu_count > 0:
    catboost_task_type = "GPU"
    catboost_device_label = "GPU"
else:
    catboost_task_type = "CPU"
    catboost_device_label = "CPU"

print("\nDetected acceleration:")
print("XGBoost:", xgb_device_label)
print(
    "LightGBM:",
    lightgbm_device_label,
    f"({lightgbm_backend})",
)
print("CatBoost:", catboost_device_label)


# ------------------------------------------------------------
# 8. Define fixed untuned baseline models
# ------------------------------------------------------------

models = [
    {
        "name": "Logistic Regression",
        "model": LogisticRegression(
            solver="lbfgs",
            max_iter=1_000,
            random_state=RANDOM_STATE,
        ),
        "input": "scaled",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "Naive Bayes",
        "model": GaussianNB(),
        "input": "scaled",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "KNN",
        "model": KNeighborsClassifier(
            n_neighbors=5,
            weights="uniform",
            n_jobs=-1,
        ),
        "input": "scaled",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "Linear SVM",
        "model": LinearSVC(
            C=1.0,
            max_iter=10_000,
            random_state=RANDOM_STATE,
        ),
        "input": "scaled",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "Decision Tree",
        "model": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
        ),
        "input": "raw",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "Random Forest",
        "model": RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input": "raw",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "Extra Trees",
        "model": ExtraTreesClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "input": "raw",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "AdaBoost",
        "model": AdaBoostClassifier(
            n_estimators=50,
            learning_rate=1.0,
            random_state=RANDOM_STATE,
        ),
        "input": "raw",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "Gradient Boosting",
        "model": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
        ),
        "input": "raw",
        "device": "CPU",
        "prediction_method": "predict",
    },
    {
        "name": "XGBoost",
        "model": XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.3,
            subsample=1.0,
            colsample_bytree=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **xgb_device_parameters,
        ),
        "input": "raw",
        "device": xgb_device_label,
        "prediction_method": "predict",
    },
    {
        "name": "LightGBM",
        "model": LGBMClassifier(
            n_estimators=100,
            learning_rate=0.1,
            num_leaves=31,
            objective="binary",
            device_type=lightgbm_backend,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        ),
        "input": "raw",
        "device": lightgbm_device_label,
        "prediction_method": "predict",
    },
    {
        "name": "CatBoost",
        "model": CatBoostClassifier(
            iterations=300,
            depth=6,
            learning_rate=0.1,
            loss_function="Logloss",
            eval_metric="F1",
            task_type=catboost_task_type,
            devices="0"
            if catboost_task_type == "GPU"
            else None,
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        ),
        "input": "raw",
        "device": catboost_device_label,
        "prediction_method": "predict",
    },
]


# ------------------------------------------------------------
# 9. Save model configurations
# ------------------------------------------------------------

configuration_records = []

for specification in models:
    configuration_records.append({
        "Model": specification["name"],
        "Input Representation":
            specification["input"],
        "Training Device":
            specification["device"],
        "Parameters":
            specification["model"].get_params(),
    })

with open(
    MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        configuration_records,
        file,
        indent=2,
        default=str,
    )


# ------------------------------------------------------------
# 10. Metric function
# ------------------------------------------------------------

def calculate_metrics(
    y_true,
    predictions,
):
    predictions = np.asarray(
        predictions,
        dtype=np.int32,
    ).reshape(-1)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 11. Load checkpoint when present
# ------------------------------------------------------------

if PARTIAL_RESULTS_PATH.exists():
    results_df = pd.read_csv(
        PARTIAL_RESULTS_PATH
    )
    completed_models = set(
        results_df["Model"].tolist()
    )

    print("\nResuming existing Stage 2 run.")
    print(
        "Already completed:",
        sorted(completed_models),
    )
else:
    results_df = pd.DataFrame()
    completed_models = set()


# ------------------------------------------------------------
# 12. Train and evaluate each model
# ------------------------------------------------------------

for model_number, specification in enumerate(
    models,
    start=1,
):
    model_name = specification["name"]

    if model_name in completed_models:
        print(
            f"\nSKIPPING {model_name}: "
            "result already checkpointed."
        )
        continue

    print("\n" + "=" * 80)
    print(
        f"MODEL {model_number}/{len(models)}: "
        f"{model_name}"
    )
    print("=" * 80)

    model = specification["model"]

    if specification["input"] == "scaled":
        training_features = X_train_std
        validation_features = X_val_std
    else:
        training_features = X_train_raw
        validation_features = X_val_raw

    print(
        "Input representation:",
        specification["input"],
    )
    print(
        "Training device:",
        specification["device"],
    )

    training_start = time.perf_counter()

    model.fit(
        training_features,
        y_train_baseline,
    )

    training_time = (
        time.perf_counter()
        - training_start
    )

    prediction_start = time.perf_counter()

    predictions = model.predict(
        validation_features
    )

    predictions = np.asarray(
        predictions
    ).reshape(-1)

    # CatBoost may return float labels
    predictions = (
        predictions >= 0.5
    ).astype(np.int32)

    prediction_time = (
        time.perf_counter()
        - prediction_start
    )

    metrics = calculate_metrics(
        y_true=y_val_baseline,
        predictions=predictions,
    )

    result = {
        "Model": model_name,
        "Evaluation Split": "Validation",
        "Training Device":
            specification["device"],
        "Input Representation":
            specification["input"],
        "Threshold": 0.50,
        **metrics,
        "Training Time (sec)":
            training_time,
        "Prediction Time (sec)":
            prediction_time,
    }

    # Save model immediately
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_path = (
        MODELS_DIR /
        f"{safe_name}_baseline.joblib"
    )

    joblib.dump(
        model,
        model_path,
    )

    # Save native boosting formats too
    if model_name == "XGBoost":
        model.save_model(
            MODELS_DIR /
            "xgboost_baseline.json"
        )

    elif model_name == "LightGBM":
        model.booster_.save_model(
            str(
                MODELS_DIR /
                "lightgbm_baseline.txt"
            )
        )

    elif model_name == "CatBoost":
        model.save_model(
            str(
                MODELS_DIR /
                "catboost_baseline.cbm"
            )
        )

    # Update checkpoint immediately
    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([result]),
        ],
        ignore_index=True,
    )

    results_df = (
        results_df
        .drop_duplicates(
            subset=["Model"],
            keep="last",
        )
        .sort_values(
            by=[
                "F1-score",
                "Recall",
                "Precision",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    results_df.to_csv(
        PARTIAL_RESULTS_PATH,
        index=False,
    )

    print("\nValidation result:")

    display(
        pd.DataFrame([result])[
            [
                "Model",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-score",
                "FPR",
                "FNR",
                "TP",
                "TN",
                "FP",
                "FN",
                "Training Time (sec)",
            ]
        ].round(6)
    )

    print("\nCheckpoint saved:")
    print(PARTIAL_RESULTS_PATH)
    print(model_path)

    del model
    del predictions

    gc.collect()


# ------------------------------------------------------------
# 13. Verify all 12 models completed
# ------------------------------------------------------------

expected_models = {
    specification["name"]
    for specification in models
}

found_models = set(
    results_df["Model"].tolist()
)

missing_models = (
    expected_models
    - found_models
)

if missing_models:
    raise RuntimeError(
        "Stage 2 is incomplete. Missing models:\n"
        f"{sorted(missing_models)}"
    )


# ------------------------------------------------------------
# 14. Create final ranked table
# ------------------------------------------------------------

final_df = (
    results_df
    .sort_values(
        by=[
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

if "Rank" in final_df.columns:
    final_df = final_df.drop(
        columns=["Rank"]
    )

final_df.insert(
    0,
    "Rank",
    range(
        1,
        len(final_df) + 1,
    ),
)

final_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 15. Final display
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("STAGE 2A COMPLETED — 12 BASELINE MODELS")
print("=" * 92)

display(
    final_df[
        [
            "Rank",
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "Training Device",
            "Training Time (sec)",
        ]
    ].round(6)
)

print("\nSaved result table:")
print(FINAL_RESULTS_PATH)

print("\nTest set used:")
print(False)


# ------------------------------------------------------------
# 16. Create Stage 2A archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage02a"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nArchive created:")
print(archive_path)

In [ ]:
# ============================================================
# STAGE 2B — FOUR NEURAL BASELINE MODELS
#
# Models:
#   1. MLP
#   2. 1D-CNN
#   3. LSTM
#   4. Transformer Encoder
#
# Training:
#   85% of X_train_scaled
#
# Early stopping:
#   15% of X_train_scaled
#
# Evaluation:
#   X_val_scaled
#
# Test set:
#   Not used
# ============================================================

import gc
import json
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
MAX_EPOCHS = 30
PATIENCE = 4
INTERNAL_VALIDATION_SIZE = 0.15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

gpus = tf.config.list_physical_devices("GPU")

print("TensorFlow:", tf.__version__)
print("Detected GPUs:", gpus)

if not gpus:
    raise RuntimeError(
        "TensorFlow cannot detect the Kaggle GPU. "
        "Enable the P100 accelerator before continuing."
    )

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass


# ------------------------------------------------------------
# 2. Verify Stage 1 arrays
# ------------------------------------------------------------

required_variables = [
    "X_train_scaled",
    "X_val_scaled",
    "y_train",
    "y_val",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Stage 1 arrays are missing. "
        f"Missing: {missing_variables}"
    )

X_train_neural = np.asarray(
    X_train_scaled,
    dtype=np.float32,
)

X_val_neural = np.asarray(
    X_val_scaled,
    dtype=np.float32,
)

y_train_neural = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_neural = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_neural.shape == (192_593, 78)
assert X_val_neural.shape == (48_149, 78)

print("\nTraining:", X_train_neural.shape)
print("Validation:", X_val_neural.shape)
print("Test set used:", False)


# ------------------------------------------------------------
# 3. Training-only early-stopping split
# ------------------------------------------------------------

train_positions = np.arange(
    len(y_train_neural)
)

fit_positions, early_stop_positions = train_test_split(
    train_positions,
    test_size=INTERNAL_VALIDATION_SIZE,
    stratify=y_train_neural,
    random_state=RANDOM_STATE,
)

X_fit_2d = X_train_neural[fit_positions]
X_early_stop_2d = X_train_neural[
    early_stop_positions
]

y_fit = y_train_neural[fit_positions]
y_early_stop = y_train_neural[
    early_stop_positions
]

X_fit_3d = np.expand_dims(
    X_fit_2d,
    axis=-1,
)

X_early_stop_3d = np.expand_dims(
    X_early_stop_2d,
    axis=-1,
)

X_val_3d = np.expand_dims(
    X_val_neural,
    axis=-1,
)

print("\nInternal fitting split:", X_fit_2d.shape)
print(
    "Internal early-stop split:",
    X_early_stop_2d.shape,
)
print("External validation:", X_val_neural.shape)


# ------------------------------------------------------------
# 4. Output directories
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = ROOT / "stage02_baselines"
RESULTS_DIR = STAGE_DIR / "results"
MODELS_DIR = STAGE_DIR / "models"
HISTORY_DIR = STAGE_DIR / "histories"
CONFIG_DIR = STAGE_DIR / "configurations"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    HISTORY_DIR,
    CONFIG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

PARTIAL_RESULTS_PATH = (
    RESULTS_DIR /
    "baseline4_neural_validation_partial.csv"
)

FINAL_NEURAL_PATH = (
    RESULTS_DIR /
    "baseline4_neural_validation_results.csv"
)

BASELINE12_PATH = (
    RESULTS_DIR /
    "baseline12_validation_results.csv"
)

FINAL16_PATH = (
    RESULTS_DIR /
    "final_16_model_validation_ablation.csv"
)

CONFIG_PATH = (
    CONFIG_DIR /
    "neural_baseline_configurations.json"
)


# ------------------------------------------------------------
# 5. Registered trainable positional embedding
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="IDS2018"
)
class TrainablePositionEmbedding(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        sequence_length,
        embedding_dimension,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.sequence_length = sequence_length
        self.embedding_dimension = (
            embedding_dimension
        )

        self.position_embedding = (
            tf.keras.layers.Embedding(
                input_dim=sequence_length,
                output_dim=embedding_dimension,
            )
        )

    def call(self, inputs):
        positions = tf.range(
            start=0,
            limit=self.sequence_length,
            delta=1,
        )

        position_vectors = (
            self.position_embedding(
                positions
            )
        )

        return inputs + position_vectors

    def get_config(self):
        config = super().get_config()

        config.update({
            "sequence_length":
                self.sequence_length,
            "embedding_dimension":
                self.embedding_dimension,
        })

        return config


# ------------------------------------------------------------
# 6. Model builders
# ------------------------------------------------------------

def build_mlp():
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(
                shape=(78,)
            ),
            tf.keras.layers.Dense(
                256,
                activation="relu",
                kernel_initializer="he_normal",
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.20),

            tf.keras.layers.Dense(
                128,
                activation="relu",
                kernel_initializer="he_normal",
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.20),

            tf.keras.layers.Dense(
                64,
                activation="relu",
                kernel_initializer="he_normal",
            ),
            tf.keras.layers.Dropout(0.20),

            tf.keras.layers.Dense(
                1,
                activation="sigmoid",
            ),
        ],
        name="mlp_baseline",
    )

    return model


def build_cnn():
    inputs = tf.keras.layers.Input(
        shape=(78, 1)
    )

    x = tf.keras.layers.Conv1D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
    )(inputs)

    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.MaxPooling1D(
        pool_size=2
    )(x)

    x = tf.keras.layers.Dropout(0.25)(x)

    x = tf.keras.layers.Conv1D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(
        128,
        activation="relu",
    )(x)

    x = tf.keras.layers.Dropout(0.30)(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="cnn_baseline",
    )


def build_lstm():
    inputs = tf.keras.layers.Input(
        shape=(78, 1)
    )

    x = tf.keras.layers.LSTM(
        64,
        return_sequences=False,
        dropout=0.20,
    )(inputs)

    x = tf.keras.layers.Dense(
        64,
        activation="relu",
    )(x)

    x = tf.keras.layers.Dropout(0.25)(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="lstm_baseline",
    )


def build_transformer():
    sequence_length = 78
    embedding_dimension = 64

    inputs = tf.keras.layers.Input(
        shape=(sequence_length, 1)
    )

    x = tf.keras.layers.Dense(
        embedding_dimension
    )(inputs)

    x = TrainablePositionEmbedding(
        sequence_length=sequence_length,
        embedding_dimension=embedding_dimension,
        name="position_embedding",
    )(x)

    attention_output = (
        tf.keras.layers.MultiHeadAttention(
            num_heads=4,
            key_dim=16,
            dropout=0.10,
        )(
            query=x,
            value=x,
            key=x,
        )
    )

    x = tf.keras.layers.Add()([
        x,
        attention_output,
    ])

    x = tf.keras.layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    feed_forward = tf.keras.layers.Dense(
        128,
        activation="relu",
    )(x)

    feed_forward = tf.keras.layers.Dropout(
        0.10
    )(feed_forward)

    feed_forward = tf.keras.layers.Dense(
        embedding_dimension
    )(feed_forward)

    x = tf.keras.layers.Add()([
        x,
        feed_forward,
    ])

    x = tf.keras.layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(
        64,
        activation="relu",
    )(x)

    x = tf.keras.layers.Dropout(0.20)(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="transformer_baseline",
    )


# ------------------------------------------------------------
# 7. Fixed baseline specifications
# ------------------------------------------------------------

model_specs = [
    {
        "name": "MLP",
        "builder": build_mlp,
        "input_type": "2d",
        "batch_size": 1024,
        "learning_rate": 0.001,
    },
    {
        "name": "1D-CNN",
        "builder": build_cnn,
        "input_type": "3d",
        "batch_size": 1024,
        "learning_rate": 0.001,
    },
    {
        "name": "LSTM",
        "builder": build_lstm,
        "input_type": "3d",
        "batch_size": 1024,
        "learning_rate": 0.001,
    },
    {
        "name": "Transformer Encoder",
        "builder": build_transformer,
        "input_type": "3d",
        "batch_size": 1024,
        "learning_rate": 0.0005,
    },
]

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        [
            {
                "Model": spec["name"],
                "Input Type":
                    spec["input_type"],
                "Batch Size":
                    spec["batch_size"],
                "Learning Rate":
                    spec["learning_rate"],
                "Maximum Epochs":
                    MAX_EPOCHS,
                "Early-Stopping Patience":
                    PATIENCE,
            }
            for spec in model_specs
        ],
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 8. Metric function
# ------------------------------------------------------------

def calculate_metrics(
    y_true,
    probabilities,
    threshold=0.50,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 9. Resume checkpoint
# ------------------------------------------------------------

if PARTIAL_RESULTS_PATH.exists():
    results_df = pd.read_csv(
        PARTIAL_RESULTS_PATH
    )

    completed_models = set(
        results_df["Model"].tolist()
    )

    print(
        "\nResuming completed neural models:",
        sorted(completed_models),
    )
else:
    results_df = pd.DataFrame()
    completed_models = set()


# ------------------------------------------------------------
# 10. Train models
# ------------------------------------------------------------

for model_number, spec in enumerate(
    model_specs,
    start=1,
):
    model_name = spec["name"]

    if model_name in completed_models:
        print(
            f"\nSKIPPING {model_name}: "
            "already checkpointed."
        )
        continue

    print("\n" + "=" * 82)
    print(
        f"NEURAL MODEL "
        f"{model_number}/{len(model_specs)}: "
        f"{model_name}"
    )
    print("=" * 82)

    tf.keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(
        RANDOM_STATE + model_number
    )

    model = spec["builder"]()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=
                spec["learning_rate"]
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    if spec["input_type"] == "2d":
        X_fit_model = X_fit_2d
        X_stop_model = X_early_stop_2d
        X_val_model = X_val_neural
    else:
        X_fit_model = X_fit_3d
        X_stop_model = X_early_stop_3d
        X_val_model = X_val_3d

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    print("Training device: GPU")
    print("Batch size:", spec["batch_size"])

    training_start = time.perf_counter()

    history = model.fit(
        X_fit_model,
        y_fit,
        validation_data=(
            X_stop_model,
            y_early_stop,
        ),
        epochs=MAX_EPOCHS,
        batch_size=spec["batch_size"],
        callbacks=callbacks,
        shuffle=True,
        verbose=2,
    )

    training_time = (
        time.perf_counter()
        - training_start
    )

    prediction_start = time.perf_counter()

    probabilities = model.predict(
        X_val_model,
        batch_size=4096,
        verbose=0,
    ).reshape(-1)

    prediction_time = (
        time.perf_counter()
        - prediction_start
    )

    metrics = calculate_metrics(
        y_true=y_val_neural,
        probabilities=probabilities,
        threshold=0.50,
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    result = {
        "Model": model_name,
        "Evaluation Split": "Validation",
        "Training Device": "GPU",
        "Input Representation":
            spec["input_type"],
        "Threshold": 0.50,
        **metrics,
        "Best Epoch": best_epoch,
        "Epochs Completed": len(
            history.history["loss"]
        ),
        "Training Time (sec)":
            training_time,
        "Prediction Time (sec)":
            prediction_time,
    }

    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_path = (
        MODELS_DIR /
        f"{safe_name}_baseline.keras"
    )

    history_path = (
        HISTORY_DIR /
        f"{safe_name}_baseline_history.csv"
    )

    probability_path = (
        RESULTS_DIR /
        f"{safe_name}_baseline_validation_probabilities.csv"
    )

    model.save(
        model_path,
        overwrite=True,
    )

    pd.DataFrame(
        history.history
    ).to_csv(
        history_path,
        index=False,
    )

    pd.DataFrame({
        "y_true": y_val_neural,
        "attack_probability": probabilities,
    }).to_csv(
        probability_path,
        index=False,
    )

    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([result]),
        ],
        ignore_index=True,
    )

    results_df = (
        results_df
        .drop_duplicates(
            subset=["Model"],
            keep="last",
        )
        .sort_values(
            [
                "F1-score",
                "Recall",
                "Precision",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    results_df.to_csv(
        PARTIAL_RESULTS_PATH,
        index=False,
    )

    print("\nValidation result:")

    display(
        pd.DataFrame([result])[
            [
                "Model",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-score",
                "FPR",
                "FNR",
                "TP",
                "TN",
                "FP",
                "FN",
                "Best Epoch",
                "Training Time (sec)",
            ]
        ].round(6)
    )

    print("\nCheckpoint saved:")
    print(PARTIAL_RESULTS_PATH)
    print(model_path)
    print(probability_path)

    del model
    del history
    del probabilities

    tf.keras.backend.clear_session()
    gc.collect()


# ------------------------------------------------------------
# 11. Verify all four completed
# ------------------------------------------------------------

expected_neural_models = {
    spec["name"]
    for spec in model_specs
}

found_neural_models = set(
    results_df["Model"].tolist()
)

missing_neural_models = (
    expected_neural_models
    - found_neural_models
)

if missing_neural_models:
    raise RuntimeError(
        "Neural baseline stage incomplete. "
        f"Missing: {sorted(missing_neural_models)}"
    )


# ------------------------------------------------------------
# 12. Save ranked neural table
# ------------------------------------------------------------

neural_final_df = (
    results_df
    .sort_values(
        [
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

if "Rank" in neural_final_df.columns:
    neural_final_df = (
        neural_final_df.drop(
            columns=["Rank"]
        )
    )

neural_final_df.insert(
    0,
    "Rank",
    range(
        1,
        len(neural_final_df) + 1,
    ),
)

neural_final_df.to_csv(
    FINAL_NEURAL_PATH,
    index=False,
)


# ------------------------------------------------------------
# 13. Merge all 16 baseline models
# ------------------------------------------------------------

if not BASELINE12_PATH.exists():
    raise FileNotFoundError(
        "The 12-model baseline file is missing:\n"
        f"{BASELINE12_PATH}"
    )

baseline12_df = pd.read_csv(
    BASELINE12_PATH
)

combined_df = pd.concat(
    [
        baseline12_df,
        neural_final_df,
    ],
    ignore_index=True,
    sort=False,
)

combined_df = (
    combined_df
    .drop_duplicates(
        subset=["Model"],
        keep="last",
    )
    .sort_values(
        [
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

if "Rank" in combined_df.columns:
    combined_df = combined_df.drop(
        columns=["Rank"]
    )

combined_df.insert(
    0,
    "Rank",
    range(
        1,
        len(combined_df) + 1,
    ),
)

if len(combined_df) != 16:
    raise ValueError(
        f"Expected 16 models, "
        f"found {len(combined_df)}."
    )

combined_df.to_csv(
    FINAL16_PATH,
    index=False,
)


# ------------------------------------------------------------
# 14. Final displays
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("FOUR NEURAL BASELINES")
print("=" * 92)

display(
    neural_final_df[
        [
            "Rank",
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "Best Epoch",
            "Training Time (sec)",
        ]
    ].round(6)
)

print("\n" + "=" * 92)
print("FINAL 16-MODEL VALIDATION ABLATION")
print("=" * 92)

display(
    combined_df[
        [
            "Rank",
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "Training Device",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 15. Select top five baseline models
# ------------------------------------------------------------

top5_df = combined_df.head(5).copy()

top5_path = (
    RESULTS_DIR /
    "validation_selected_top5_models.csv"
)

top5_df.to_csv(
    top5_path,
    index=False,
)

print("\nValidation-selected top five:")

display(
    top5_df[
        [
            "Rank",
            "Model",
            "F1-score",
            "Recall",
            "Precision",
            "FPR",
            "FNR",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 16. Archive complete Stage 2
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage02_complete"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(FINAL_NEURAL_PATH)
print(FINAL16_PATH)
print(top5_path)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

In [ ]:
# ============================================================
# STAGE 3A — VALIDATION-SAFE XGBOOST TUNING
#
# Hyperparameter selection:
#   3-fold stratified CV on training data only
#
# External evaluation:
#   Validation set at threshold 0.50
#
# Test set:
#   Not used
# ============================================================

import gc
import json
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
CV_FOLDS = 3
N_ITERATIONS = 15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Verify arrays
# ------------------------------------------------------------

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Stage 1 arrays are missing. "
        f"Missing: {missing_variables}"
    )

X_train_xgb = np.asarray(
    X_train,
    dtype=np.float32,
)

X_val_xgb = np.asarray(
    X_val,
    dtype=np.float32,
)

y_train_xgb = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_xgb = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_xgb.shape == (192_593, 78)
assert X_val_xgb.shape == (48_149, 78)

print("Training:", X_train_xgb.shape)
print("Validation:", X_val_xgb.shape)
print("Test set used:", False)


# ------------------------------------------------------------
# 3. Output directories
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = (
    ROOT
    / "stage03_top5_tuning"
    / "xgboost"
)

RESULTS_DIR = STAGE_DIR / "results"
MODELS_DIR = STAGE_DIR / "models"
LOGS_DIR = STAGE_DIR / "search_logs"
PARAMS_DIR = STAGE_DIR / "best_parameters"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    LOGS_DIR,
    PARAMS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

RESULT_PATH = (
    RESULTS_DIR
    / "xgboost_tuned_validation_result.csv"
)

PROBABILITY_PATH = (
    RESULTS_DIR
    / "xgboost_validation_probabilities.csv"
)

SEARCH_LOG_PATH = (
    LOGS_DIR
    / "xgboost_random_search_results.csv"
)

PARAMETERS_PATH = (
    PARAMS_DIR
    / "xgboost_best_parameters.json"
)

JOBLIB_MODEL_PATH = (
    MODELS_DIR
    / "xgboost_tuned.joblib"
)

NATIVE_MODEL_PATH = (
    MODELS_DIR
    / "xgboost_tuned.json"
)


# ------------------------------------------------------------
# 4. Detect GPU support
# ------------------------------------------------------------

xgb_device_label = "CPU"
xgb_device_parameters = {
    "tree_method": "hist",
}

major_version = int(
    xgboost.__version__.split(".")[0]
)

try:
    if major_version >= 2:
        candidate_parameters = {
            "tree_method": "hist",
            "device": "cuda",
        }
    else:
        candidate_parameters = {
            "tree_method": "gpu_hist",
            "predictor": "gpu_predictor",
        }

    probe = XGBClassifier(
        n_estimators=5,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        **candidate_parameters,
    )

    probe.fit(
        X_train_xgb[:5_000],
        y_train_xgb[:5_000],
    )

    xgb_device_parameters = candidate_parameters
    xgb_device_label = "GPU"

    del probe
    gc.collect()

except Exception as error:
    print("XGBoost GPU unavailable; using CPU.")
    print(str(error)[:250])

print("XGBoost version:", xgboost.__version__)
print("Training device:", xgb_device_label)


# ------------------------------------------------------------
# 5. Base estimator
# ------------------------------------------------------------

estimator = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0,
    **xgb_device_parameters,
)


# ------------------------------------------------------------
# 6. Hyperparameter space
# ------------------------------------------------------------

parameter_space = {
    "n_estimators": [
        200,
        300,
        500,
        700,
    ],
    "max_depth": [
        4,
        6,
        8,
        10,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
    ],
    "min_child_weight": [
        1,
        3,
        5,
    ],
    "subsample": [
        0.80,
        0.90,
        1.00,
    ],
    "colsample_bytree": [
        0.80,
        0.90,
        1.00,
    ],
    "gamma": [
        0.0,
        0.1,
        0.3,
    ],
    "reg_alpha": [
        0.0,
        0.1,
        1.0,
    ],
    "reg_lambda": [
        1.0,
        5.0,
        10.0,
    ],
}


# ------------------------------------------------------------
# 7. Randomized search
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=parameter_space,
    n_iter=N_ITERATIONS,
    scoring="f1",
    cv=cv_strategy,
    refit=True,
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=False,
    error_score="raise",
    pre_dispatch=1,
)

print("\n" + "=" * 80)
print("TUNING XGBOOST")
print("=" * 80)
print("Random candidates:", N_ITERATIONS)
print("CV folds:", CV_FOLDS)
print("Total fits:", N_ITERATIONS * CV_FOLDS)
print("Device:", xgb_device_label)

search_start = time.perf_counter()

search.fit(
    X_train_xgb,
    y_train_xgb,
)

search_time = (
    time.perf_counter()
    - search_start
)

best_model = search.best_estimator_


# ------------------------------------------------------------
# 8. Save search log and parameters
# ------------------------------------------------------------

search_results_df = (
    pd.DataFrame(search.cv_results_)
    .sort_values(
        "rank_test_score",
        ascending=True,
    )
    .reset_index(drop=True)
)

search_results_df.to_csv(
    SEARCH_LOG_PATH,
    index=False,
)

best_parameters = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value
    in search.best_params_.items()
}

parameter_record = {
    "Model": "XGBoost Tuned",
    "Selection Metric":
        "Mean 3-fold training CV F1-score",
    "Best CV F1-score":
        float(search.best_score_),
    "Random Candidates":
        N_ITERATIONS,
    "CV Folds":
        CV_FOLDS,
    "Training Device":
        xgb_device_label,
    "Training Records":
        int(len(y_train_xgb)),
    "Validation Records":
        int(len(y_val_xgb)),
    "Feature Count":
        int(X_train_xgb.shape[1]),
    "Test Set Used":
        False,
    "Best Parameters":
        best_parameters,
}

with open(
    PARAMETERS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        parameter_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Validation evaluation
# ------------------------------------------------------------

prediction_start = time.perf_counter()

validation_probabilities = (
    best_model.predict_proba(
        X_val_xgb
    )[:, 1]
)

prediction_time = (
    time.perf_counter()
    - prediction_start
)

validation_predictions = (
    validation_probabilities >= 0.50
).astype(np.int32)

tn, fp, fn, tp = confusion_matrix(
    y_val_xgb,
    validation_predictions,
    labels=[0, 1],
).ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

result = {
    "Model": "XGBoost Tuned",
    "Evaluation Split": "Validation",
    "Training Device": xgb_device_label,
    "Threshold": 0.50,
    "Best CV F1-score":
        float(search.best_score_),
    "Accuracy": accuracy_score(
        y_val_xgb,
        validation_predictions,
    ),
    "Precision": precision_score(
        y_val_xgb,
        validation_predictions,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val_xgb,
        validation_predictions,
        zero_division=0,
    ),
    "F1-score": f1_score(
        y_val_xgb,
        validation_predictions,
        zero_division=0,
    ),
    "FPR": fpr,
    "FNR": fnr,
    "TP": int(tp),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "Search Time (sec)": search_time,
    "Prediction Time (sec)": prediction_time,
}

pd.DataFrame([result]).to_csv(
    RESULT_PATH,
    index=False,
)

pd.DataFrame({
    "y_true": y_val_xgb,
    "attack_probability":
        validation_probabilities,
}).to_csv(
    PROBABILITY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Save model
# ------------------------------------------------------------

joblib.dump(
    best_model,
    JOBLIB_MODEL_PATH,
)

best_model.save_model(
    NATIVE_MODEL_PATH,
)


# ------------------------------------------------------------
# 11. Display result
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("XGBOOST TUNING COMPLETED")
print("=" * 84)

print("\nBest CV F1-score:")
print(round(search.best_score_, 6))

print("\nBest parameters:")
print(json.dumps(best_parameters, indent=2))

print("\nValidation result:")

display(
    pd.DataFrame([result])[
        [
            "Model",
            "Best CV F1-score",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "Search Time (sec)",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 12. Create archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage03a_xgboost"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(RESULT_PATH)
print(PROBABILITY_PATH)
print(SEARCH_LOG_PATH)
print(PARAMETERS_PATH)
print(JOBLIB_MODEL_PATH)
print(NATIVE_MODEL_PATH)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

In [ ]:
# ============================================================
# STAGE 3B — VALIDATION-SAFE LIGHTGBM TUNING
#
# Hyperparameter selection:
#   3-fold stratified CV on training data only
#
# External evaluation:
#   Validation set at threshold 0.50
#
# Test set:
#   Not used
# ============================================================

import gc
import json
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import lightgbm

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from lightgbm import LGBMClassifier


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
CV_FOLDS = 3
N_ITERATIONS = 15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Verify arrays
# ------------------------------------------------------------

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Stage 1 arrays are missing. "
        f"Missing: {missing_variables}"
    )

X_train_lgbm = np.asarray(
    X_train,
    dtype=np.float32,
)

X_val_lgbm = np.asarray(
    X_val,
    dtype=np.float32,
)

y_train_lgbm = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_lgbm = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_lgbm.shape == (192_593, 78)
assert X_val_lgbm.shape == (48_149, 78)

print("Training:", X_train_lgbm.shape)
print("Validation:", X_val_lgbm.shape)
print("Test set used:", False)


# ------------------------------------------------------------
# 3. Output directories
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = (
    ROOT
    / "stage03_top5_tuning"
    / "lightgbm"
)

RESULTS_DIR = STAGE_DIR / "results"
MODELS_DIR = STAGE_DIR / "models"
LOGS_DIR = STAGE_DIR / "search_logs"
PARAMS_DIR = STAGE_DIR / "best_parameters"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    LOGS_DIR,
    PARAMS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

RESULT_PATH = (
    RESULTS_DIR
    / "lightgbm_tuned_validation_result.csv"
)

PROBABILITY_PATH = (
    RESULTS_DIR
    / "lightgbm_validation_probabilities.csv"
)

SEARCH_LOG_PATH = (
    LOGS_DIR
    / "lightgbm_random_search_results.csv"
)

PARAMETERS_PATH = (
    PARAMS_DIR
    / "lightgbm_best_parameters.json"
)

JOBLIB_MODEL_PATH = (
    MODELS_DIR
    / "lightgbm_tuned.joblib"
)

NATIVE_MODEL_PATH = (
    MODELS_DIR
    / "lightgbm_tuned.txt"
)


# ------------------------------------------------------------
# 4. Detect LightGBM acceleration backend
# ------------------------------------------------------------

lightgbm_backend = "cpu"
lightgbm_device_label = "CPU"

for candidate_backend in [
    "gpu",
    "cuda",
]:
    try:
        probe = LGBMClassifier(
            n_estimators=5,
            objective="binary",
            device_type=candidate_backend,
            random_state=RANDOM_STATE,
            verbosity=-1,
        )

        probe.fit(
            X_train_lgbm[:5_000],
            y_train_lgbm[:5_000],
        )

        lightgbm_backend = candidate_backend
        lightgbm_device_label = "GPU"

        print(
            "Selected LightGBM backend:",
            candidate_backend,
        )

        del probe
        gc.collect()
        break

    except Exception as error:
        print(
            f"Backend '{candidate_backend}' unavailable:"
        )
        print(str(error)[:200])

if lightgbm_backend == "cpu":
    print("LightGBM will use CPU fallback.")

print("\nLightGBM version:", lightgbm.__version__)
print("Training device:", lightgbm_device_label)
print("Backend:", lightgbm_backend)


# ------------------------------------------------------------
# 5. Base estimator
# ------------------------------------------------------------

estimator = LGBMClassifier(
    objective="binary",
    device_type=lightgbm_backend,
    subsample_freq=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)


# ------------------------------------------------------------
# 6. Hyperparameter space
# ------------------------------------------------------------

parameter_space = {
    "n_estimators": [
        200,
        400,
        700,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
    ],
    "num_leaves": [
        31,
        63,
        127,
    ],
    "max_depth": [
        -1,
        15,
        30,
    ],
    "min_child_samples": [
        10,
        20,
        40,
    ],
    "subsample": [
        0.80,
        0.90,
        1.00,
    ],
    "colsample_bytree": [
        0.80,
        0.90,
        1.00,
    ],
    "reg_alpha": [
        0.0,
        0.1,
        1.0,
    ],
    "reg_lambda": [
        0.0,
        1.0,
        5.0,
    ],
}


# ------------------------------------------------------------
# 7. Randomized search
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=parameter_space,
    n_iter=N_ITERATIONS,
    scoring="f1",
    cv=cv_strategy,
    refit=True,
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=False,
    error_score="raise",
    pre_dispatch=1,
)

print("\n" + "=" * 80)
print("TUNING LIGHTGBM")
print("=" * 80)
print("Random candidates:", N_ITERATIONS)
print("CV folds:", CV_FOLDS)
print("Total fits:", N_ITERATIONS * CV_FOLDS)
print("Device:", lightgbm_device_label)
print("Backend:", lightgbm_backend)

search_start = time.perf_counter()

search.fit(
    X_train_lgbm,
    y_train_lgbm,
)

search_time = (
    time.perf_counter()
    - search_start
)

best_model = search.best_estimator_


# ------------------------------------------------------------
# 8. Save search log
# ------------------------------------------------------------

search_results_df = (
    pd.DataFrame(search.cv_results_)
    .sort_values(
        "rank_test_score",
        ascending=True,
    )
    .reset_index(drop=True)
)

search_results_df.to_csv(
    SEARCH_LOG_PATH,
    index=False,
)


# ------------------------------------------------------------
# 9. Save best parameters
# ------------------------------------------------------------

best_parameters = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value
    in search.best_params_.items()
}

parameter_record = {
    "Model": "LightGBM Tuned",
    "Selection Metric":
        "Mean 3-fold training CV F1-score",
    "Best CV F1-score":
        float(search.best_score_),
    "Random Candidates":
        N_ITERATIONS,
    "CV Folds":
        CV_FOLDS,
    "Training Device":
        lightgbm_device_label,
    "LightGBM Backend":
        lightgbm_backend,
    "Training Records":
        int(len(y_train_lgbm)),
    "Validation Records":
        int(len(y_val_lgbm)),
    "Feature Count":
        int(X_train_lgbm.shape[1]),
    "Test Set Used":
        False,
    "Best Parameters":
        best_parameters,
}

with open(
    PARAMETERS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        parameter_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Validation evaluation
# ------------------------------------------------------------

prediction_start = time.perf_counter()

validation_probabilities = (
    best_model.predict_proba(
        X_val_lgbm
    )[:, 1]
)

prediction_time = (
    time.perf_counter()
    - prediction_start
)

validation_predictions = (
    validation_probabilities >= 0.50
).astype(np.int32)

tn, fp, fn, tp = confusion_matrix(
    y_val_lgbm,
    validation_predictions,
    labels=[0, 1],
).ravel()

fpr = (
    fp / (fp + tn)
    if fp + tn > 0
    else 0.0
)

fnr = (
    fn / (fn + tp)
    if fn + tp > 0
    else 0.0
)

result = {
    "Model": "LightGBM Tuned",
    "Evaluation Split": "Validation",
    "Training Device":
        lightgbm_device_label,
    "LightGBM Backend":
        lightgbm_backend,
    "Threshold": 0.50,
    "Best CV F1-score":
        float(search.best_score_),
    "Accuracy": accuracy_score(
        y_val_lgbm,
        validation_predictions,
    ),
    "Precision": precision_score(
        y_val_lgbm,
        validation_predictions,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val_lgbm,
        validation_predictions,
        zero_division=0,
    ),
    "F1-score": f1_score(
        y_val_lgbm,
        validation_predictions,
        zero_division=0,
    ),
    "FPR": fpr,
    "FNR": fnr,
    "TP": int(tp),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "Search Time (sec)": search_time,
    "Prediction Time (sec)": prediction_time,
}

pd.DataFrame([result]).to_csv(
    RESULT_PATH,
    index=False,
)

pd.DataFrame({
    "y_true": y_val_lgbm,
    "attack_probability":
        validation_probabilities,
}).to_csv(
    PROBABILITY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. Save model
# ------------------------------------------------------------

joblib.dump(
    best_model,
    JOBLIB_MODEL_PATH,
)

best_model.booster_.save_model(
    str(NATIVE_MODEL_PATH)
)


# ------------------------------------------------------------
# 12. Display result
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("LIGHTGBM TUNING COMPLETED")
print("=" * 84)

print("\nBest CV F1-score:")
print(round(search.best_score_, 6))

print("\nBest parameters:")
print(
    json.dumps(
        best_parameters,
        indent=2,
    )
)

print("\nValidation result:")

display(
    pd.DataFrame([result])[
        [
            "Model",
            "Best CV F1-score",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "Search Time (sec)",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 13. Create archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage03b_lightgbm"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(RESULT_PATH)
print(PROBABILITY_PATH)
print(SEARCH_LOG_PATH)
print(PARAMETERS_PATH)
print(JOBLIB_MODEL_PATH)
print(NATIVE_MODEL_PATH)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

In [ ]:
# ============================================================
# STAGE 3C — VALIDATION-SAFE CATBOOST TUNING
#
# Hyperparameter selection:
#   3-fold stratified CV on training data only
#
# External evaluation:
#   Validation set at threshold 0.50
#
# Test set:
#   Not used
# ============================================================

import gc
import json
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import catboost

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
CV_FOLDS = 3
N_ITERATIONS = 15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Verify Stage 1 arrays
# ------------------------------------------------------------

required_variables = [
    "X_train",
    "X_val",
    "y_train",
    "y_val",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Stage 1 arrays are missing. "
        f"Missing: {missing_variables}"
    )

X_train_cat = np.asarray(
    X_train,
    dtype=np.float32,
)

X_val_cat = np.asarray(
    X_val,
    dtype=np.float32,
)

y_train_cat = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_cat = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_cat.shape == (192_593, 78)
assert X_val_cat.shape == (48_149, 78)
assert len(y_train_cat) == 192_593
assert len(y_val_cat) == 48_149

print("Training:", X_train_cat.shape)
print("Validation:", X_val_cat.shape)
print("Test set used:", False)


# ------------------------------------------------------------
# 3. Detect CatBoost GPU support
# ------------------------------------------------------------

gpu_count = get_gpu_device_count()

print("\nCatBoost version:", catboost.__version__)
print("Detected CatBoost GPUs:", gpu_count)

if gpu_count > 0:
    task_type = "GPU"
    device_label = "GPU"
else:
    task_type = "CPU"
    device_label = "CPU"

print("Training device:", device_label)


# ------------------------------------------------------------
# 4. Output directories
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = (
    ROOT
    / "stage03_top5_tuning"
    / "catboost"
)

RESULTS_DIR = STAGE_DIR / "results"
MODELS_DIR = STAGE_DIR / "models"
LOGS_DIR = STAGE_DIR / "search_logs"
PARAMS_DIR = STAGE_DIR / "best_parameters"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    LOGS_DIR,
    PARAMS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

RESULT_PATH = (
    RESULTS_DIR
    / "catboost_tuned_validation_result.csv"
)

PROBABILITY_PATH = (
    RESULTS_DIR
    / "catboost_validation_probabilities.csv"
)

SEARCH_LOG_PATH = (
    LOGS_DIR
    / "catboost_random_search_results.csv"
)

PARAMETERS_PATH = (
    PARAMS_DIR
    / "catboost_best_parameters.json"
)

JOBLIB_MODEL_PATH = (
    MODELS_DIR
    / "catboost_tuned.joblib"
)

NATIVE_MODEL_PATH = (
    MODELS_DIR
    / "catboost_tuned.cbm"
)


# ------------------------------------------------------------
# 5. Base estimator
# ------------------------------------------------------------

estimator_parameters = {
    "loss_function": "Logloss",
    "eval_metric": "F1",
    "task_type": task_type,
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
}

if task_type == "GPU":
    estimator_parameters["devices"] = "0"

estimator = CatBoostClassifier(
    **estimator_parameters
)


# ------------------------------------------------------------
# 6. Hyperparameter search space
# ------------------------------------------------------------

parameter_space = {
    "iterations": [
        300,
        500,
        700,
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10,
    ],
    "depth": [
        6,
        8,
        10,
    ],
    "l2_leaf_reg": [
        3,
        5,
        7,
    ],
    "random_strength": [
        0.5,
        1.0,
        2.0,
    ],
    "border_count": [
        64,
        128,
        254,
    ],
}


# ------------------------------------------------------------
# 7. Randomized search
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=parameter_space,
    n_iter=N_ITERATIONS,
    scoring="f1",
    cv=cv_strategy,
    refit=True,
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=False,
    error_score="raise",
    pre_dispatch=1,
)

print("\n" + "=" * 80)
print("TUNING CATBOOST")
print("=" * 80)
print("Random candidates:", N_ITERATIONS)
print("CV folds:", CV_FOLDS)
print("Total fits:", N_ITERATIONS * CV_FOLDS)
print("Device:", device_label)

search_start = time.perf_counter()

search.fit(
    X_train_cat,
    y_train_cat,
)

search_time = (
    time.perf_counter()
    - search_start
)

best_model = search.best_estimator_


# ------------------------------------------------------------
# 8. Save complete search log
# ------------------------------------------------------------

search_results_df = (
    pd.DataFrame(search.cv_results_)
    .sort_values(
        "rank_test_score",
        ascending=True,
    )
    .reset_index(drop=True)
)

search_results_df.to_csv(
    SEARCH_LOG_PATH,
    index=False,
)


# ------------------------------------------------------------
# 9. Save best parameters
# ------------------------------------------------------------

best_parameters = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value
    in search.best_params_.items()
}

parameter_record = {
    "Model": "CatBoost Tuned",
    "Selection Metric":
        "Mean 3-fold training CV F1-score",
    "Best CV F1-score":
        float(search.best_score_),
    "Random Candidates":
        N_ITERATIONS,
    "CV Folds":
        CV_FOLDS,
    "Training Device":
        device_label,
    "Training Records":
        int(len(y_train_cat)),
    "Validation Records":
        int(len(y_val_cat)),
    "Feature Count":
        int(X_train_cat.shape[1]),
    "Test Set Used":
        False,
    "Best Parameters":
        best_parameters,
}

with open(
    PARAMETERS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        parameter_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Validation evaluation
# ------------------------------------------------------------

prediction_start = time.perf_counter()

validation_probabilities = (
    best_model.predict_proba(
        X_val_cat
    )[:, 1]
)

prediction_time = (
    time.perf_counter()
    - prediction_start
)

validation_predictions = (
    validation_probabilities >= 0.50
).astype(np.int32)

tn, fp, fn, tp = confusion_matrix(
    y_val_cat,
    validation_predictions,
    labels=[0, 1],
).ravel()

fpr = (
    fp / (fp + tn)
    if fp + tn > 0
    else 0.0
)

fnr = (
    fn / (fn + tp)
    if fn + tp > 0
    else 0.0
)

result = {
    "Model": "CatBoost Tuned",
    "Evaluation Split": "Validation",
    "Training Device": device_label,
    "Threshold": 0.50,
    "Best CV F1-score":
        float(search.best_score_),
    "Accuracy": accuracy_score(
        y_val_cat,
        validation_predictions,
    ),
    "Precision": precision_score(
        y_val_cat,
        validation_predictions,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_val_cat,
        validation_predictions,
        zero_division=0,
    ),
    "F1-score": f1_score(
        y_val_cat,
        validation_predictions,
        zero_division=0,
    ),
    "FPR": fpr,
    "FNR": fnr,
    "TP": int(tp),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "Search Time (sec)": search_time,
    "Prediction Time (sec)": prediction_time,
}

pd.DataFrame([result]).to_csv(
    RESULT_PATH,
    index=False,
)

pd.DataFrame({
    "y_true": y_val_cat,
    "attack_probability":
        validation_probabilities,
}).to_csv(
    PROBABILITY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. Save model
# ------------------------------------------------------------

joblib.dump(
    best_model,
    JOBLIB_MODEL_PATH,
)

best_model.save_model(
    str(NATIVE_MODEL_PATH)
)


# ------------------------------------------------------------
# 12. Display result
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("CATBOOST TUNING COMPLETED")
print("=" * 84)

print("\nBest CV F1-score:")
print(round(search.best_score_, 6))

print("\nBest parameters:")
print(
    json.dumps(
        best_parameters,
        indent=2,
    )
)

print("\nValidation result:")

display(
    pd.DataFrame([result])[
        [
            "Model",
            "Best CV F1-score",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "Search Time (sec)",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 13. Create archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage03c_catboost"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(RESULT_PATH)
print(PROBABILITY_PATH)
print(SEARCH_LOG_PATH)
print(PARAMETERS_PATH)
print(JOBLIB_MODEL_PATH)
print(NATIVE_MODEL_PATH)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

In [ ]:
# ============================================================
# STAGE 3D — VALIDATION-SAFE MLP TUNING
#
# Candidate training:
#   Training-only fitting subset
#
# Early stopping:
#   Separate 15% subset of the training set
#
# Candidate selection:
#   External validation F1-score at threshold 0.50
#
# Test set:
#   Not used
# ============================================================

import gc
import json
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
MAX_EPOCHS = 50
PATIENCE = 5
INTERNAL_VALIDATION_SIZE = 0.15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Confirm GPU
# ------------------------------------------------------------

gpus = tf.config.list_physical_devices("GPU")

print("TensorFlow version:", tf.__version__)
print("Detected GPUs:", gpus)

if not gpus:
    raise RuntimeError(
        "TensorFlow cannot detect the Kaggle GPU. "
        "Enable the P100 accelerator."
    )

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass


# ------------------------------------------------------------
# 3. Verify Stage 1 arrays
# ------------------------------------------------------------

required_variables = [
    "X_train_scaled",
    "X_val_scaled",
    "y_train",
    "y_val",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun the Stage 1 split cell first. "
        f"Missing variables: {missing_variables}"
    )

X_train_mlp = np.asarray(
    X_train_scaled,
    dtype=np.float32,
)

X_val_mlp = np.asarray(
    X_val_scaled,
    dtype=np.float32,
)

y_train_mlp = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_mlp = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_mlp.shape == (192_593, 78)
assert X_val_mlp.shape == (48_149, 78)
assert y_train_mlp.shape == (192_593,)
assert y_val_mlp.shape == (48_149,)

print("\nTraining:", X_train_mlp.shape)
print("Validation:", X_val_mlp.shape)
print("Test set used:", False)


# ------------------------------------------------------------
# 4. Training-only early-stopping split
# ------------------------------------------------------------

(
    X_fit,
    X_early_stop,
    y_fit,
    y_early_stop,
) = train_test_split(
    X_train_mlp,
    y_train_mlp,
    test_size=INTERNAL_VALIDATION_SIZE,
    stratify=y_train_mlp,
    random_state=RANDOM_STATE,
)

print("\nFitting subset:", X_fit.shape)
print("Early-stopping subset:", X_early_stop.shape)
print("External validation:", X_val_mlp.shape)


# ------------------------------------------------------------
# 5. Output directories
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = (
    ROOT
    / "stage03_top5_tuning"
    / "mlp"
)

RESULTS_DIR = STAGE_DIR / "results"
MODELS_DIR = STAGE_DIR / "models"
HISTORY_DIR = STAGE_DIR / "histories"
PARAMS_DIR = STAGE_DIR / "best_parameters"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    HISTORY_DIR,
    PARAMS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

CANDIDATE_RESULTS_PATH = (
    RESULTS_DIR
    / "mlp_candidate_validation_results.csv"
)

BEST_RESULT_PATH = (
    RESULTS_DIR
    / "mlp_tuned_validation_result.csv"
)

PROBABILITY_PATH = (
    RESULTS_DIR
    / "mlp_validation_probabilities.csv"
)

BEST_MODEL_PATH = (
    MODELS_DIR
    / "mlp_tuned.keras"
)

BEST_HISTORY_PATH = (
    HISTORY_DIR
    / "mlp_best_training_history.csv"
)

BEST_PARAMETERS_PATH = (
    PARAMS_DIR
    / "mlp_best_parameters.json"
)


# ------------------------------------------------------------
# 6. Candidate configurations
# ------------------------------------------------------------

candidates = [
    {
        "Candidate": "MLP-01",
        "Hidden Layers": [128, 64],
        "Dropout": 0.20,
        "Learning Rate": 0.001,
        "Batch Size": 1024,
    },
    {
        "Candidate": "MLP-02",
        "Hidden Layers": [256, 128],
        "Dropout": 0.20,
        "Learning Rate": 0.0005,
        "Batch Size": 1024,
    },
    {
        "Candidate": "MLP-03",
        "Hidden Layers": [256, 128, 64],
        "Dropout": 0.20,
        "Learning Rate": 0.0005,
        "Batch Size": 1024,
    },
    {
        "Candidate": "MLP-04",
        "Hidden Layers": [256, 128, 64],
        "Dropout": 0.30,
        "Learning Rate": 0.001,
        "Batch Size": 512,
    },
    {
        "Candidate": "MLP-05",
        "Hidden Layers": [512, 256, 128],
        "Dropout": 0.30,
        "Learning Rate": 0.0005,
        "Batch Size": 1024,
    },
    {
        "Candidate": "MLP-06",
        "Hidden Layers": [512, 256, 128, 64],
        "Dropout": 0.30,
        "Learning Rate": 0.0003,
        "Batch Size": 1024,
    },
]

print("\nNumber of candidates:", len(candidates))


# ------------------------------------------------------------
# 7. Model builder
# ------------------------------------------------------------

def build_mlp(
    hidden_layers,
    dropout,
    learning_rate,
):
    inputs = tf.keras.layers.Input(
        shape=(78,),
        name="traffic_features",
    )

    x = inputs

    for layer_number, units in enumerate(
        hidden_layers,
        start=1,
    ):
        x = tf.keras.layers.Dense(
            units,
            activation=None,
            kernel_initializer="he_normal",
            name=f"dense_{layer_number}",
        )(x)

        x = tf.keras.layers.BatchNormalization(
            name=f"batch_norm_{layer_number}",
        )(x)

        x = tf.keras.layers.Activation(
            "relu",
            name=f"relu_{layer_number}",
        )(x)

        x = tf.keras.layers.Dropout(
            dropout,
            name=f"dropout_{layer_number}",
        )(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        name="attack_probability",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="validation_safe_mlp",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# 8. Metric function
# ------------------------------------------------------------

def calculate_metrics(
    y_true,
    probabilities,
    threshold=0.50,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 9. Resume checkpoint if present
# ------------------------------------------------------------

if CANDIDATE_RESULTS_PATH.exists():
    candidate_results_df = pd.read_csv(
        CANDIDATE_RESULTS_PATH
    )

    completed_candidates = set(
        candidate_results_df[
            "Candidate"
        ].tolist()
    )

    candidate_results = (
        candidate_results_df
        .to_dict("records")
    )

    ranked_existing = (
        candidate_results_df
        .sort_values(
            [
                "F1-score",
                "Recall",
                "Precision",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    best_f1 = float(
        ranked_existing.loc[0, "F1-score"]
    )

    best_recall = float(
        ranked_existing.loc[0, "Recall"]
    )

    best_candidate_record = (
        ranked_existing.iloc[0].to_dict()
    )

    print(
        "\nResuming completed candidates:",
        sorted(completed_candidates),
    )
else:
    candidate_results = []
    completed_candidates = set()

    best_f1 = -1.0
    best_recall = -1.0
    best_candidate_record = None


# ------------------------------------------------------------
# 10. Train candidate configurations
# ------------------------------------------------------------

for candidate_number, config in enumerate(
    candidates,
    start=1,
):
    candidate_name = config["Candidate"]

    if candidate_name in completed_candidates:
        print(
            f"\nSKIPPING {candidate_name}: "
            "already checkpointed."
        )
        continue

    print("\n" + "=" * 82)
    print(
        f"MLP CANDIDATE "
        f"{candidate_number}/{len(candidates)}: "
        f"{candidate_name}"
    )
    print("=" * 82)

    print(json.dumps(config, indent=2))

    tf.keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(
        RANDOM_STATE + candidate_number
    )

    model = build_mlp(
        hidden_layers=config["Hidden Layers"],
        dropout=config["Dropout"],
        learning_rate=config[
            "Learning Rate"
        ],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    training_start = time.perf_counter()

    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(
            X_early_stop,
            y_early_stop,
        ),
        epochs=MAX_EPOCHS,
        batch_size=config["Batch Size"],
        callbacks=callbacks,
        shuffle=True,
        verbose=2,
    )

    training_time = (
        time.perf_counter()
        - training_start
    )

    prediction_start = time.perf_counter()

    validation_probabilities = (
        model.predict(
            X_val_mlp,
            batch_size=4096,
            verbose=0,
        )
        .reshape(-1)
    )

    prediction_time = (
        time.perf_counter()
        - prediction_start
    )

    metrics = calculate_metrics(
        y_true=y_val_mlp,
        probabilities=validation_probabilities,
        threshold=0.50,
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    result = {
        "Model": "MLP Tuned",
        "Candidate": candidate_name,
        "Hidden Layers": str(
            config["Hidden Layers"]
        ),
        "Dropout": config["Dropout"],
        "Learning Rate":
            config["Learning Rate"],
        "Batch Size":
            config["Batch Size"],
        "Best Epoch": best_epoch,
        "Epochs Completed": len(
            history.history["loss"]
        ),
        "Evaluation Split": "Validation",
        "Training Device": "GPU",
        "Threshold": 0.50,
        **metrics,
        "Training Time (sec)":
            training_time,
        "Prediction Time (sec)":
            prediction_time,
    }

    candidate_results.append(result)

    candidate_results_df = (
        pd.DataFrame(candidate_results)
        .drop_duplicates(
            subset=["Candidate"],
            keep="last",
        )
        .sort_values(
            [
                "F1-score",
                "Recall",
                "Precision",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    candidate_results_df.to_csv(
        CANDIDATE_RESULTS_PATH,
        index=False,
    )

    print("\nValidation result:")

    display(
        pd.DataFrame([result])[
            [
                "Candidate",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-score",
                "FPR",
                "FNR",
                "TP",
                "TN",
                "FP",
                "FN",
                "Best Epoch",
                "Training Time (sec)",
            ]
        ].round(6)
    )

    candidate_is_better = (
        metrics["F1-score"] > best_f1
        or (
            np.isclose(
                metrics["F1-score"],
                best_f1,
            )
            and metrics["Recall"]
            > best_recall
        )
    )

    if candidate_is_better:
        best_f1 = metrics["F1-score"]
        best_recall = metrics["Recall"]
        best_candidate_record = result.copy()

        model.save(
            BEST_MODEL_PATH,
            overwrite=True,
        )

        pd.DataFrame(
            history.history
        ).to_csv(
            BEST_HISTORY_PATH,
            index=False,
        )

        pd.DataFrame({
            "y_true": y_val_mlp,
            "attack_probability":
                validation_probabilities,
        }).to_csv(
            PROBABILITY_PATH,
            index=False,
        )

        print("\nNew best MLP model saved:")
        print(BEST_MODEL_PATH)

    del model
    del history
    del validation_probabilities

    tf.keras.backend.clear_session()
    gc.collect()


# ------------------------------------------------------------
# 11. Final ranked candidate table
# ------------------------------------------------------------

candidate_results_df = (
    pd.DataFrame(candidate_results)
    .drop_duplicates(
        subset=["Candidate"],
        keep="last",
    )
    .sort_values(
        [
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

if len(candidate_results_df) != len(candidates):
    raise RuntimeError(
        f"Expected {len(candidates)} completed candidates, "
        f"found {len(candidate_results_df)}."
    )

best_candidate_record = (
    candidate_results_df.iloc[0].to_dict()
)

pd.DataFrame([
    best_candidate_record
]).to_csv(
    BEST_RESULT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. Save best parameters
# ------------------------------------------------------------

parameter_record = {
    "Model": "MLP Tuned",
    "Selection Metric":
        "External validation F1-score at threshold 0.50",
    "Early Stopping Data":
        "15% stratified subset of training set",
    "Training Records Total":
        int(len(y_train_mlp)),
    "Fitting Records":
        int(len(y_fit)),
    "Early-Stopping Records":
        int(len(y_early_stop)),
    "External Validation Records":
        int(len(y_val_mlp)),
    "Feature Count":
        int(X_train_mlp.shape[1]),
    "Training Device":
        "GPU",
    "Test Set Used":
        False,
    "Best Candidate":
        best_candidate_record,
}

with open(
    BEST_PARAMETERS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        parameter_record,
        file,
        indent=2,
        default=str,
    )


# ------------------------------------------------------------
# 13. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("MLP TUNING COMPLETED")
print("=" * 90)

display(
    candidate_results_df[
        [
            "Candidate",
            "Hidden Layers",
            "Dropout",
            "Learning Rate",
            "Batch Size",
            "Best Epoch",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
        ]
    ].round(6)
)

print("\nBest MLP candidate:")

display(
    pd.DataFrame([
        best_candidate_record
    ]).round(6)
)


# ------------------------------------------------------------
# 14. Create archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage03d_mlp"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(CANDIDATE_RESULTS_PATH)
print(BEST_RESULT_PATH)
print(PROBABILITY_PATH)
print(BEST_MODEL_PATH)
print(BEST_HISTORY_PATH)
print(BEST_PARAMETERS_PATH)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

In [ ]:
# ============================================================
# STAGE 3E — VALIDATION-SAFE 1D-CNN TUNING
#
# Fitting:
#   85% training subset
#
# Early stopping:
#   Separate 15% training subset
#
# Candidate selection:
#   External validation F1 at threshold 0.50
#
# Test set:
#   Not used
# ============================================================

import gc
import json
import os
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42
MAX_EPOCHS = 50
PATIENCE = 5
INTERNAL_VALIDATION_SIZE = 0.15

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Confirm GPU
# ------------------------------------------------------------

gpus = tf.config.list_physical_devices("GPU")

print("TensorFlow version:", tf.__version__)
print("Detected GPUs:", gpus)

if not gpus:
    raise RuntimeError(
        "TensorFlow cannot detect a GPU. "
        "Enable the Kaggle P100 accelerator."
    )

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass


# ------------------------------------------------------------
# 3. Verify Stage 1 arrays
# ------------------------------------------------------------

required_variables = [
    "X_train_scaled",
    "X_val_scaled",
    "y_train",
    "y_val",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun Stage 1 first. "
        f"Missing variables: {missing_variables}"
    )

X_train_cnn_2d = np.asarray(
    X_train_scaled,
    dtype=np.float32,
)

X_val_cnn_2d = np.asarray(
    X_val_scaled,
    dtype=np.float32,
)

y_train_cnn = np.asarray(
    y_train,
    dtype=np.int32,
).reshape(-1)

y_val_cnn = np.asarray(
    y_val,
    dtype=np.int32,
).reshape(-1)

assert X_train_cnn_2d.shape == (192_593, 78)
assert X_val_cnn_2d.shape == (48_149, 78)
assert y_train_cnn.shape == (192_593,)
assert y_val_cnn.shape == (48_149,)

X_train_cnn = np.expand_dims(
    X_train_cnn_2d,
    axis=-1,
)

X_val_cnn = np.expand_dims(
    X_val_cnn_2d,
    axis=-1,
)

print("\nTraining:", X_train_cnn.shape)
print("Validation:", X_val_cnn.shape)
print("Test set used:", False)


# ------------------------------------------------------------
# 4. Training-only early-stopping split
# ------------------------------------------------------------

(
    X_fit,
    X_early_stop,
    y_fit,
    y_early_stop,
) = train_test_split(
    X_train_cnn,
    y_train_cnn,
    test_size=INTERNAL_VALIDATION_SIZE,
    stratify=y_train_cnn,
    random_state=RANDOM_STATE,
)

print("\nFitting subset:", X_fit.shape)
print("Early-stopping subset:", X_early_stop.shape)
print("External validation:", X_val_cnn.shape)


# ------------------------------------------------------------
# 5. Output directories
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = (
    ROOT
    / "stage03_top5_tuning"
    / "cnn"
)

RESULTS_DIR = STAGE_DIR / "results"
MODELS_DIR = STAGE_DIR / "models"
HISTORY_DIR = STAGE_DIR / "histories"
PARAMS_DIR = STAGE_DIR / "best_parameters"

for directory in [
    RESULTS_DIR,
    MODELS_DIR,
    HISTORY_DIR,
    PARAMS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

CANDIDATE_RESULTS_PATH = (
    RESULTS_DIR
    / "cnn_candidate_validation_results.csv"
)

BEST_RESULT_PATH = (
    RESULTS_DIR
    / "cnn_tuned_validation_result.csv"
)

PROBABILITY_PATH = (
    RESULTS_DIR
    / "cnn_validation_probabilities.csv"
)

BEST_MODEL_PATH = (
    MODELS_DIR
    / "cnn_tuned.keras"
)

BEST_HISTORY_PATH = (
    HISTORY_DIR
    / "cnn_best_training_history.csv"
)

BEST_PARAMETERS_PATH = (
    PARAMS_DIR
    / "cnn_best_parameters.json"
)


# ------------------------------------------------------------
# 6. Candidate configurations
# ------------------------------------------------------------

candidates = [
    {
        "Candidate": "CNN-01",
        "Filters": [64, 128],
        "Kernel Size": 3,
        "Dropout": 0.20,
        "Dense Units": 64,
        "Learning Rate": 0.001,
        "Batch Size": 1024,
    },
    {
        "Candidate": "CNN-02",
        "Filters": [128, 256],
        "Kernel Size": 3,
        "Dropout": 0.30,
        "Dense Units": 128,
        "Learning Rate": 0.0005,
        "Batch Size": 1024,
    },
    {
        "Candidate": "CNN-03",
        "Filters": [128, 256],
        "Kernel Size": 5,
        "Dropout": 0.30,
        "Dense Units": 128,
        "Learning Rate": 0.0005,
        "Batch Size": 512,
    },
    {
        "Candidate": "CNN-04",
        "Filters": [64, 128, 256],
        "Kernel Size": 3,
        "Dropout": 0.30,
        "Dense Units": 128,
        "Learning Rate": 0.0005,
        "Batch Size": 1024,
    },
    {
        "Candidate": "CNN-05",
        "Filters": [128, 256, 256],
        "Kernel Size": 5,
        "Dropout": 0.40,
        "Dense Units": 128,
        "Learning Rate": 0.0003,
        "Batch Size": 512,
    },
    {
        "Candidate": "CNN-06",
        "Filters": [256, 256],
        "Kernel Size": 5,
        "Dropout": 0.30,
        "Dense Units": 256,
        "Learning Rate": 0.0003,
        "Batch Size": 1024,
    },
]

print("\nNumber of CNN candidates:", len(candidates))


# ------------------------------------------------------------
# 7. Model builder
# ------------------------------------------------------------

def build_cnn(
    filters,
    kernel_size,
    dropout,
    dense_units,
    learning_rate,
):
    inputs = tf.keras.layers.Input(
        shape=(78, 1),
        name="traffic_features",
    )

    x = inputs

    for block_number, filter_count in enumerate(
        filters,
        start=1,
    ):
        x = tf.keras.layers.Conv1D(
            filters=filter_count,
            kernel_size=kernel_size,
            padding="same",
            activation=None,
            kernel_initializer="he_normal",
            name=f"conv_{block_number}",
        )(x)

        x = tf.keras.layers.BatchNormalization(
            name=f"batch_norm_{block_number}",
        )(x)

        x = tf.keras.layers.Activation(
            "relu",
            name=f"relu_{block_number}",
        )(x)

        if block_number < len(filters):
            x = tf.keras.layers.MaxPooling1D(
                pool_size=2,
                name=f"pool_{block_number}",
            )(x)

        x = tf.keras.layers.Dropout(
            dropout,
            name=f"conv_dropout_{block_number}",
        )(x)

    x = tf.keras.layers.GlobalAveragePooling1D(
        name="global_average_pooling",
    )(x)

    x = tf.keras.layers.Dense(
        dense_units,
        activation="relu",
        kernel_initializer="he_normal",
        name="dense_hidden",
    )(x)

    x = tf.keras.layers.BatchNormalization(
        name="dense_batch_norm",
    )(x)

    x = tf.keras.layers.Dropout(
        dropout,
        name="dense_dropout",
    )(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        name="attack_probability",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="validation_safe_1d_cnn",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# 8. Metric function
# ------------------------------------------------------------

def calculate_metrics(
    y_true,
    probabilities,
    threshold=0.50,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 9. Resume checkpoint
# ------------------------------------------------------------

if CANDIDATE_RESULTS_PATH.exists():
    candidate_results_df = pd.read_csv(
        CANDIDATE_RESULTS_PATH
    )

    completed_candidates = set(
        candidate_results_df[
            "Candidate"
        ].tolist()
    )

    candidate_results = (
        candidate_results_df
        .to_dict("records")
    )

    ranked_existing = (
        candidate_results_df
        .sort_values(
            [
                "F1-score",
                "Recall",
                "Precision",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    best_f1 = float(
        ranked_existing.loc[0, "F1-score"]
    )

    best_recall = float(
        ranked_existing.loc[0, "Recall"]
    )

    print(
        "\nResuming completed candidates:",
        sorted(completed_candidates),
    )
else:
    candidate_results = []
    completed_candidates = set()
    best_f1 = -1.0
    best_recall = -1.0


# ------------------------------------------------------------
# 10. Train candidates
# ------------------------------------------------------------

for candidate_number, config in enumerate(
    candidates,
    start=1,
):
    candidate_name = config["Candidate"]

    if candidate_name in completed_candidates:
        print(
            f"\nSKIPPING {candidate_name}: "
            "already checkpointed."
        )
        continue

    print("\n" + "=" * 84)
    print(
        f"CNN CANDIDATE "
        f"{candidate_number}/{len(candidates)}: "
        f"{candidate_name}"
    )
    print("=" * 84)

    print(json.dumps(config, indent=2))

    tf.keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(
        RANDOM_STATE + candidate_number
    )

    model = build_cnn(
        filters=config["Filters"],
        kernel_size=config["Kernel Size"],
        dropout=config["Dropout"],
        dense_units=config["Dense Units"],
        learning_rate=config["Learning Rate"],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    training_start = time.perf_counter()

    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(
            X_early_stop,
            y_early_stop,
        ),
        epochs=MAX_EPOCHS,
        batch_size=config["Batch Size"],
        callbacks=callbacks,
        shuffle=True,
        verbose=2,
    )

    training_time = (
        time.perf_counter()
        - training_start
    )

    prediction_start = time.perf_counter()

    validation_probabilities = (
        model.predict(
            X_val_cnn,
            batch_size=4096,
            verbose=0,
        )
        .reshape(-1)
    )

    prediction_time = (
        time.perf_counter()
        - prediction_start
    )

    metrics = calculate_metrics(
        y_true=y_val_cnn,
        probabilities=validation_probabilities,
        threshold=0.50,
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    result = {
        "Model": "1D-CNN Tuned",
        "Candidate": candidate_name,
        "Filters": str(config["Filters"]),
        "Kernel Size": config["Kernel Size"],
        "Dropout": config["Dropout"],
        "Dense Units": config["Dense Units"],
        "Learning Rate":
            config["Learning Rate"],
        "Batch Size": config["Batch Size"],
        "Best Epoch": best_epoch,
        "Epochs Completed": len(
            history.history["loss"]
        ),
        "Evaluation Split": "Validation",
        "Training Device": "GPU",
        "Threshold": 0.50,
        **metrics,
        "Training Time (sec)":
            training_time,
        "Prediction Time (sec)":
            prediction_time,
    }

    candidate_results.append(result)

    candidate_results_df = (
        pd.DataFrame(candidate_results)
        .drop_duplicates(
            subset=["Candidate"],
            keep="last",
        )
        .sort_values(
            [
                "F1-score",
                "Recall",
                "Precision",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    candidate_results_df.to_csv(
        CANDIDATE_RESULTS_PATH,
        index=False,
    )

    print("\nValidation result:")

    display(
        pd.DataFrame([result])[
            [
                "Candidate",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-score",
                "FPR",
                "FNR",
                "TP",
                "TN",
                "FP",
                "FN",
                "Best Epoch",
                "Training Time (sec)",
            ]
        ].round(6)
    )

    candidate_is_better = (
        metrics["F1-score"] > best_f1
        or (
            np.isclose(
                metrics["F1-score"],
                best_f1,
            )
            and metrics["Recall"] > best_recall
        )
    )

    if candidate_is_better:
        best_f1 = metrics["F1-score"]
        best_recall = metrics["Recall"]

        model.save(
            BEST_MODEL_PATH,
            overwrite=True,
        )

        pd.DataFrame(
            history.history
        ).to_csv(
            BEST_HISTORY_PATH,
            index=False,
        )

        pd.DataFrame({
            "y_true": y_val_cnn,
            "attack_probability":
                validation_probabilities,
        }).to_csv(
            PROBABILITY_PATH,
            index=False,
        )

        print("\nNew best CNN model saved:")
        print(BEST_MODEL_PATH)

    del model
    del history
    del validation_probabilities

    tf.keras.backend.clear_session()
    gc.collect()


# ------------------------------------------------------------
# 11. Final ranking and best result
# ------------------------------------------------------------

candidate_results_df = (
    pd.DataFrame(candidate_results)
    .drop_duplicates(
        subset=["Candidate"],
        keep="last",
    )
    .sort_values(
        [
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

if len(candidate_results_df) != len(candidates):
    raise RuntimeError(
        f"Expected {len(candidates)} completed candidates, "
        f"found {len(candidate_results_df)}."
    )

best_candidate_record = (
    candidate_results_df.iloc[0].to_dict()
)

pd.DataFrame([
    best_candidate_record
]).to_csv(
    BEST_RESULT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. Save parameter record
# ------------------------------------------------------------

parameter_record = {
    "Model": "1D-CNN Tuned",
    "Selection Metric":
        "External validation F1-score at threshold 0.50",
    "Early Stopping Data":
        "15% stratified subset of training set",
    "Training Records Total":
        int(len(y_train_cnn)),
    "Fitting Records":
        int(len(y_fit)),
    "Early-Stopping Records":
        int(len(y_early_stop)),
    "External Validation Records":
        int(len(y_val_cnn)),
    "Feature Count":
        int(X_train_cnn.shape[1]),
    "Input Shape":
        list(X_train_cnn.shape[1:]),
    "Training Device":
        "GPU",
    "Test Set Used":
        False,
    "Best Candidate":
        best_candidate_record,
}

with open(
    BEST_PARAMETERS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        parameter_record,
        file,
        indent=2,
        default=str,
    )


# ------------------------------------------------------------
# 13. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("1D-CNN TUNING COMPLETED")
print("=" * 92)

display(
    candidate_results_df[
        [
            "Candidate",
            "Filters",
            "Kernel Size",
            "Dropout",
            "Dense Units",
            "Learning Rate",
            "Batch Size",
            "Best Epoch",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
        ]
    ].round(6)
)

print("\nBest CNN candidate:")

display(
    pd.DataFrame([
        best_candidate_record
    ]).round(6)
)


# ------------------------------------------------------------
# 14. Create archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage03e_cnn"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(CANDIDATE_RESULTS_PATH)
print(BEST_RESULT_PATH)
print(PROBABILITY_PATH)
print(BEST_MODEL_PATH)
print(BEST_HISTORY_PATH)
print(BEST_PARAMETERS_PATH)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

In [16]:
# ============================================================
# STAGE 3F — MERGE EXISTING TOP-FIVE TUNING OUTPUTS
#
# No upload
# No extraction
# No training
# No test-set use
# ============================================================

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

TUNING_ROOT = (
    ROOT
    / "stage03_top5_tuning"
)

OUTPUT_DIR = (
    TUNING_ROOT
    / "combined_results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


artifact_paths = {
    "XGBoost Tuned": {
        "result": (
            TUNING_ROOT
            / "xgboost"
            / "results"
            / "xgboost_tuned_validation_result.csv"
        ),
        "probabilities": (
            TUNING_ROOT
            / "xgboost"
            / "results"
            / "xgboost_validation_probabilities.csv"
        ),
    },
    "LightGBM Tuned": {
        "result": (
            TUNING_ROOT
            / "lightgbm"
            / "results"
            / "lightgbm_tuned_validation_result.csv"
        ),
        "probabilities": (
            TUNING_ROOT
            / "lightgbm"
            / "results"
            / "lightgbm_validation_probabilities.csv"
        ),
    },
    "CatBoost Tuned": {
        "result": (
            TUNING_ROOT
            / "catboost"
            / "results"
            / "catboost_tuned_validation_result.csv"
        ),
        "probabilities": (
            TUNING_ROOT
            / "catboost"
            / "results"
            / "catboost_validation_probabilities.csv"
        ),
    },
    "MLP Tuned": {
        "result": (
            TUNING_ROOT
            / "mlp"
            / "results"
            / "mlp_tuned_validation_result.csv"
        ),
        "probabilities": (
            TUNING_ROOT
            / "mlp"
            / "results"
            / "mlp_validation_probabilities.csv"
        ),
    },
    "1D-CNN Tuned": {
        "result": (
            TUNING_ROOT
            / "cnn"
            / "results"
            / "cnn_tuned_validation_result.csv"
        ),
        "probabilities": (
            TUNING_ROOT
            / "cnn"
            / "results"
            / "cnn_validation_probabilities.csv"
        ),
    },
}


# ------------------------------------------------------------
# 2. Verify existing artifacts
# ------------------------------------------------------------

verification_rows = []

for model_name, paths in artifact_paths.items():

    verification_rows.append({
        "Model": model_name,
        "Result Available":
            paths["result"].exists(),
        "Probability Available":
            paths["probabilities"].exists(),
        "Result Path":
            str(paths["result"]),
        "Probability Path":
            str(paths["probabilities"]),
    })

verification_df = pd.DataFrame(
    verification_rows
)

display(verification_df)

if not verification_df[
    "Result Available"
].all():
    missing = verification_df.loc[
        ~verification_df["Result Available"],
        ["Model", "Result Path"],
    ]

    raise FileNotFoundError(
        "Missing tuned result files:\n"
        + missing.to_string(index=False)
    )

if not verification_df[
    "Probability Available"
].all():
    missing = verification_df.loc[
        ~verification_df[
            "Probability Available"
        ],
        ["Model", "Probability Path"],
    ]

    raise FileNotFoundError(
        "Missing validation probability files:\n"
        + missing.to_string(index=False)
    )


# ------------------------------------------------------------
# 3. Load selected tuned result rows
# ------------------------------------------------------------

result_frames = []

for model_name, paths in artifact_paths.items():

    result_df = pd.read_csv(
        paths["result"]
    )

    if len(result_df) != 1:
        raise ValueError(
            f"{model_name} result file contains "
            f"{len(result_df)} rows; expected exactly one."
        )

    result_frames.append(
        result_df
    )

combined_df = pd.concat(
    result_frames,
    ignore_index=True,
    sort=False,
)


# ------------------------------------------------------------
# 4. Standardize columns
# ------------------------------------------------------------

comparison_columns = [
    "Model",
    "Evaluation Split",
    "Training Device",
    "Threshold",
    "Best CV F1-score",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "FPR",
    "FNR",
    "TP",
    "TN",
    "FP",
    "FN",
    "Search Time (sec)",
    "Training Time (sec)",
    "Prediction Time (sec)",
]

for column in comparison_columns:
    if column not in combined_df.columns:
        combined_df[column] = np.nan

combined_df["Evaluation Split"] = "Validation"
combined_df["Threshold"] = 0.50

final_df = (
    combined_df[
        comparison_columns
    ]
    .drop_duplicates(
        subset=["Model"],
        keep="last",
    )
    .sort_values(
        by=[
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

if len(final_df) != 5:
    raise ValueError(
        f"Expected five tuned models, "
        f"found {len(final_df)}."
    )

final_df.insert(
    0,
    "Rank",
    range(1, 6),
)


# ------------------------------------------------------------
# 5. Save combined outputs
# ------------------------------------------------------------

FINAL_RESULT_PATH = (
    OUTPUT_DIR
    / "top5_tuned_validation_results.csv"
)

MANIFEST_PATH = (
    OUTPUT_DIR
    / "stage03_artifact_manifest.csv"
)

METADATA_PATH = (
    OUTPUT_DIR
    / "stage03_combined_metadata.json"
)

final_df.to_csv(
    FINAL_RESULT_PATH,
    index=False,
)

verification_df.to_csv(
    MANIFEST_PATH,
    index=False,
)

metadata = {
    "Experiment":
        "IDS2018 Clean Validation V2",
    "Stage":
        "Top-five tuned validation comparison",
    "Ranking Metric":
        "Validation F1-score at threshold 0.50",
    "Tie Breakers": [
        "Validation recall",
        "Validation precision",
    ],
    "Models":
        final_df["Model"].tolist(),
    "Test Set Used":
        False,
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 6. Display comparison
# ------------------------------------------------------------

print("\n" + "=" * 94)
print("FINAL TOP-FIVE TUNED VALIDATION COMPARISON")
print("=" * 94)

display(
    final_df[
        [
            "Rank",
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "Training Device",
        ]
    ].round(6)
)

print("\nValidation winner:")
print(final_df.loc[0, "Model"])


# ------------------------------------------------------------
# 7. Archive complete Stage 3
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage03_complete"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    TUNING_ROOT,
)

print("\nSaved:")
print(FINAL_RESULT_PATH)
print(MANIFEST_PATH)
print(METADATA_PATH)

print("\nComplete Stage 3 archive:")
print(archive_path)

print("\nTest set used:")
print(False)

,Model,Result Available,Probability Available,Result Path,Probability Path
0,XGBoost Tuned,True,True,/kaggle/working/ids2018_clean_validation_v2/st...,/kaggle/working/ids2018_clean_validation_v2/st...
1,LightGBM Tuned,True,True,/kaggle/working/ids2018_clean_validation_v2/st...,/kaggle/working/ids2018_clean_validation_v2/st...
2,CatBoost Tuned,True,True,/kaggle/working/ids2018_clean_validation_v2/st...,/kaggle/working/ids2018_clean_validation_v2/st...
3,MLP Tuned,True,True,/kaggle/working/ids2018_clean_validation_v2/st...,/kaggle/working/ids2018_clean_validation_v2/st...
4,1D-CNN Tuned,True,True,/kaggle/working/ids2018_clean_validation_v2/st...,/kaggle/working/ids2018_clean_validation_v2/st...



FINAL TOP-FIVE TUNED VALIDATION COMPARISON


,Rank,Model,Accuracy,Precision,Recall,F1-score,FPR,FNR,TP,TN,FP,FN,Training Device
0,1,XGBoost Tuned,0.944651,0.986925,0.873844,0.926948,0.007778,0.126156,16908,28576,224,2441,GPU
1,2,LightGBM Tuned,0.944568,0.988062,0.872603,0.926750,0.007083,0.127397,16884,28596,204,2465,GPU
2,3,CatBoost Tuned,0.943966,0.987813,0.871311,0.925912,0.007222,0.128689,16859,28592,208,2490,GPU
3,4,MLP Tuned,0.938171,0.993727,0.851517,0.917142,0.003611,0.148483,16476,28696,104,2873,GPU
4,5,1D-CNN Tuned,0.935762,0.987992,0.850483,0.914095,0.006944,0.149517,16456,28600,200,2893,GPU



Validation winner:
XGBoost Tuned

Saved:
/kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/combined_results/top5_tuned_validation_results.csv
/kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/combined_results/stage03_artifact_manifest.csv
/kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/combined_results/stage03_combined_metadata.json

Complete Stage 3 archive:
/kaggle/working/ids2018_clean_validation_v2_stage03_complete.zip

Test set used:
False


In [17]:
# ============================================================
# STAGE 4 — VALIDATION-ONLY THRESHOLD ANALYSIS
#
# Automatically:
#   1. Reads the tuned top-five comparison
#   2. Identifies the validation winner
#   3. Loads its validation probabilities
#   4. Evaluates thresholds from 0.05 to 0.95
#   5. Selects:
#        - Standard threshold 0.50
#        - Maximum validation F1 threshold
#        - Maximum validation F2 threshold
#
# Test set:
#   Not used
# ============================================================

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

TUNING_ROOT = (
    ROOT
    / "stage03_top5_tuning"
)

TOP5_RESULT_PATH = (
    TUNING_ROOT
    / "combined_results"
    / "top5_tuned_validation_results.csv"
)

STAGE_DIR = (
    ROOT
    / "stage04_threshold"
)

RESULTS_DIR = STAGE_DIR / "results"
METADATA_DIR = STAGE_DIR / "metadata"

for directory in [
    RESULTS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 2. Verify tuned comparison
# ------------------------------------------------------------

if not TOP5_RESULT_PATH.exists():
    raise FileNotFoundError(
        "The tuned top-five comparison was not found:\n"
        f"{TOP5_RESULT_PATH}"
    )

top5_df = pd.read_csv(
    TOP5_RESULT_PATH
)

required_result_columns = {
    "Model",
    "F1-score",
    "Recall",
    "Precision",
}

missing_result_columns = (
    required_result_columns
    - set(top5_df.columns)
)

if missing_result_columns:
    raise ValueError(
        "The tuned comparison is missing columns: "
        f"{sorted(missing_result_columns)}"
    )


# ------------------------------------------------------------
# 3. Identify validation winner
# ------------------------------------------------------------

ranked_top5 = (
    top5_df
    .sort_values(
        by=[
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

winning_model = str(
    ranked_top5.loc[0, "Model"]
)

print("Validation winner:", winning_model)
print(
    "Validation F1:",
    round(
        float(
            ranked_top5.loc[
                0,
                "F1-score",
            ]
        ),
        6,
    ),
)


# ------------------------------------------------------------
# 4. Probability file map
# ------------------------------------------------------------

probability_paths = {
    "XGBoost Tuned": (
        TUNING_ROOT
        / "xgboost"
        / "results"
        / "xgboost_validation_probabilities.csv"
    ),
    "LightGBM Tuned": (
        TUNING_ROOT
        / "lightgbm"
        / "results"
        / "lightgbm_validation_probabilities.csv"
    ),
    "CatBoost Tuned": (
        TUNING_ROOT
        / "catboost"
        / "results"
        / "catboost_validation_probabilities.csv"
    ),
    "MLP Tuned": (
        TUNING_ROOT
        / "mlp"
        / "results"
        / "mlp_validation_probabilities.csv"
    ),
    "1D-CNN Tuned": (
        TUNING_ROOT
        / "cnn"
        / "results"
        / "cnn_validation_probabilities.csv"
    ),
}

if winning_model not in probability_paths:
    raise ValueError(
        f"No probability-file mapping exists for "
        f"{winning_model}."
    )

PROBABILITY_PATH = probability_paths[
    winning_model
]

if not PROBABILITY_PATH.exists():
    raise FileNotFoundError(
        f"Validation probabilities for {winning_model} "
        f"were not found:\n{PROBABILITY_PATH}"
    )

print("Probability file:")
print(PROBABILITY_PATH)


# ------------------------------------------------------------
# 5. Load validation probabilities
# ------------------------------------------------------------

probability_df = pd.read_csv(
    PROBABILITY_PATH
)

required_probability_columns = {
    "y_true",
    "attack_probability",
}

missing_probability_columns = (
    required_probability_columns
    - set(probability_df.columns)
)

if missing_probability_columns:
    raise ValueError(
        "Probability file is missing columns: "
        f"{sorted(missing_probability_columns)}"
    )

y_true = np.asarray(
    probability_df["y_true"],
    dtype=np.int32,
).reshape(-1)

probabilities = np.asarray(
    probability_df[
        "attack_probability"
    ],
    dtype=np.float64,
).reshape(-1)

if len(y_true) != 48_149:
    raise ValueError(
        f"Expected 48,149 validation records, "
        f"found {len(y_true):,}."
    )

if len(probabilities) != len(y_true):
    raise ValueError(
        "The label and probability lengths differ."
    )

if np.isnan(probabilities).any():
    raise ValueError(
        "The probability vector contains NaN values."
    )

if (
    (probabilities < 0).any()
    or (probabilities > 1).any()
):
    raise ValueError(
        "Probabilities must be within [0, 1]."
    )

print("\nValidation records:", len(y_true))
print("Benign:", int(np.sum(y_true == 0)))
print("Attack:", int(np.sum(y_true == 1)))
print("Test set used:", False)


# ------------------------------------------------------------
# 6. Metric function
# ------------------------------------------------------------

def calculate_threshold_metrics(
    labels,
    scores,
    threshold,
):
    predictions = (
        scores >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Model": winning_model,
        "Evaluation Split": "Validation",
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "Precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "F2-score": fbeta_score(
            labels,
            predictions,
            beta=2,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 7. Threshold sweep
# ------------------------------------------------------------

thresholds = np.round(
    np.arange(
        0.05,
        0.951,
        0.01,
    ),
    2,
)

threshold_results = [
    calculate_threshold_metrics(
        labels=y_true,
        scores=probabilities,
        threshold=threshold,
    )
    for threshold in thresholds
]

sweep_df = pd.DataFrame(
    threshold_results
)

SWEEP_PATH = (
    RESULTS_DIR
    / "winning_model_validation_threshold_sweep.csv"
)

sweep_df.to_csv(
    SWEEP_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Standard threshold 0.50
# ------------------------------------------------------------

standard_row = (
    sweep_df[
        np.isclose(
            sweep_df["Threshold"],
            0.50,
        )
    ]
    .iloc[0]
    .copy()
)


# ------------------------------------------------------------
# 9. Maximum validation F1 threshold
# ------------------------------------------------------------

best_f1_row = (
    sweep_df
    .sort_values(
        by=[
            "F1-score",
            "Recall",
            "FPR",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .iloc[0]
    .copy()
)


# ------------------------------------------------------------
# 10. Maximum validation F2 threshold
# ------------------------------------------------------------

best_f2_row = (
    sweep_df
    .sort_values(
        by=[
            "F2-score",
            "F1-score",
            "Recall",
            "FPR",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            False,
        ],
    )
    .iloc[0]
    .copy()
)


# ------------------------------------------------------------
# 11. Selected operating points
# ------------------------------------------------------------

selected_df = pd.DataFrame([
    {
        "Operating Point":
            "Standard Threshold",
        **standard_row.to_dict(),
    },
    {
        "Operating Point":
            "Maximum Validation F1",
        **best_f1_row.to_dict(),
    },
    {
        "Operating Point":
            "Security-Oriented Maximum F2",
        **best_f2_row.to_dict(),
    },
])

standard_fn = int(
    standard_row["FN"]
)

standard_fp = int(
    standard_row["FP"]
)

selected_df[
    "False Negatives Reduced vs 0.50"
] = (
    standard_fn
    - selected_df["FN"].astype(int)
)

selected_df[
    "Additional False Positives vs 0.50"
] = (
    selected_df["FP"].astype(int)
    - standard_fp
)

SELECTED_PATH = (
    RESULTS_DIR
    / "selected_validation_operating_points.csv"
)

selected_df.to_csv(
    SELECTED_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. Nearby thresholds
# ------------------------------------------------------------

important_thresholds = {
    float(standard_row["Threshold"]),
    float(best_f1_row["Threshold"]),
    float(best_f2_row["Threshold"]),
}

nearby_thresholds = set()

for threshold in important_thresholds:
    for offset in [
        -0.03,
        -0.02,
        -0.01,
        0.00,
        0.01,
        0.02,
        0.03,
    ]:
        value = round(
            threshold + offset,
            2,
        )

        if 0.05 <= value <= 0.95:
            nearby_thresholds.add(
                value
            )

nearby_df = (
    sweep_df[
        sweep_df["Threshold"].isin(
            sorted(nearby_thresholds)
        )
    ]
    .sort_values(
        "Threshold"
    )
    .reset_index(drop=True)
)

NEARBY_PATH = (
    RESULTS_DIR
    / "selected_threshold_neighborhood.csv"
)

nearby_df.to_csv(
    NEARBY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 13. Save metadata
# ------------------------------------------------------------

metadata = {
    "Experiment":
        "IDS2018 Clean Validation V2",
    "Stage":
        "Validation-only threshold selection",
    "Winning Model":
        winning_model,
    "Threshold Grid": {
        "Minimum": 0.05,
        "Maximum": 0.95,
        "Step": 0.01,
        "Count": int(len(thresholds)),
    },
    "Operating Points": {
        "Standard Threshold":
            float(standard_row["Threshold"]),
        "Maximum Validation F1":
            float(best_f1_row["Threshold"]),
        "Maximum Validation F2":
            float(best_f2_row["Threshold"]),
    },
    "Threshold Selection Split":
        "Validation",
    "Test Set Used":
        False,
}

METADATA_PATH = (
    METADATA_DIR
    / "threshold_selection_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 14. Display selected points
# ------------------------------------------------------------

print("\n" + "=" * 104)
print("VALIDATION-ONLY THRESHOLD SELECTION")
print("=" * 104)

display(
    selected_df[
        [
            "Operating Point",
            "Model",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "False Negatives Reduced vs 0.50",
            "Additional False Positives vs 0.50",
        ]
    ].round(6)
)

print("\nNearby thresholds:")

display(
    nearby_df[
        [
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "FP",
            "FN",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 15. Create Stage 4 archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage04_threshold"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(SWEEP_PATH)
print(SELECTED_PATH)
print(NEARBY_PATH)
print(METADATA_PATH)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

Validation winner: XGBoost Tuned
Validation F1: 0.926948
Probability file:
/kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/xgboost/results/xgboost_validation_probabilities.csv

Validation records: 48149
Benign: 28800
Attack: 19349
Test set used: False

VALIDATION-ONLY THRESHOLD SELECTION


,Operating Point,Model,Threshold,Accuracy,Precision,Recall,F1-score,F2-score,FPR,FNR,TP,TN,FP,FN,False Negatives Reduced vs 0.50,Additional False Positives vs 0.50
0,Standard Threshold,XGBoost Tuned,0.50,0.944651,0.986925,0.873844,0.926948,0.894338,0.007778,0.126156,16908,28576,224,2441,0,0
1,Maximum Validation F1,XGBoost Tuned,0.51,0.944796,0.988012,0.873223,0.927078,0.893997,0.007118,0.126777,16896,28595,205,2453,-12,-19
2,Security-Oriented Maximum F2,XGBoost Tuned,0.16,0.888326,0.809691,0.943976,0.871692,0.913670,0.149062,0.056024,18265,24507,4293,1084,1357,4069



Nearby thresholds:


,Threshold,Accuracy,Precision,Recall,F1-score,F2-score,FPR,FNR,FP,FN
0,0.13,0.857401,0.752887,0.960360,0.844061,0.910195,0.211771,0.039640,6099,767
1,0.14,0.868845,0.772154,0.955605,0.854140,0.912258,0.189444,0.044395,5456,859
2,0.15,0.879084,0.791241,0.949661,0.863243,0.913098,0.168333,0.050339,4848,974
3,0.16,0.888326,0.809691,0.943976,0.871692,0.913670,0.149062,0.056024,4293,1084
4,0.17,0.895886,0.826233,0.938240,0.878682,0.913473,0.132569,0.061760,3818,1195
5,0.18,0.902615,0.841407,0.933640,0.885127,0.913610,0.118229,0.066360,3405,1284
6,0.19,0.908264,0.855998,0.927800,0.890454,0.912492,0.104861,0.072200,3020,1397
7,0.47,0.944111,0.983457,0.875652,0.926429,0.895280,0.009896,0.124348,285,2406
8,0.48,0.944423,0.984821,0.875187,0.926773,0.895117,0.009062,0.124813,261,2415
9,0.49,0.944506,0.985615,0.874671,0.926835,0.894815,0.008576,0.125329,247,2425



Saved:
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/results/winning_model_validation_threshold_sweep.csv
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/results/selected_validation_operating_points.csv
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/results/selected_threshold_neighborhood.csv
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/metadata/threshold_selection_metadata.json

Archive created:
/kaggle/working/ids2018_clean_validation_v2_stage04_threshold.zip

Test set used:
False


In [19]:
# ============================================================
# SELECT OPERATIONALLY CONSTRAINED SECURITY THRESHOLD
#
# Objective:
#   Maximum validation F2
#
# Constraint:
#   False-positive rate <= 5%
#
# Test set:
#   Not used
# ============================================================

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


FPR_LIMIT = 0.05

eligible_df = sweep_df[
    sweep_df["FPR"] <= FPR_LIMIT
].copy()

if eligible_df.empty:
    raise RuntimeError(
        "No threshold satisfies the specified FPR limit."
    )

constrained_row = (
    eligible_df
    .sort_values(
        by=[
            "F2-score",
            "Recall",
            "F1-score",
            "FPR",
            "Threshold",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            False,
        ],
    )
    .iloc[0]
    .copy()
)

standard_row = (
    sweep_df[
        sweep_df["Threshold"] == 0.50
    ]
    .iloc[0]
    .copy()
)

constrained_selection_df = pd.DataFrame([
    {
        "Operating Point": "Standard",
        **standard_row.to_dict(),
    },
    {
        "Operating Point":
            "Unconstrained Maximum F2",
        **best_f2_row.to_dict(),
    },
    {
        "Operating Point":
            "Constrained Security Threshold",
        **constrained_row.to_dict(),
    },
])

standard_fn = int(standard_row["FN"])
standard_fp = int(standard_row["FP"])

constrained_selection_df[
    "False Negatives Reduced vs 0.50"
] = (
    standard_fn
    - constrained_selection_df["FN"].astype(int)
)

constrained_selection_df[
    "Additional False Positives vs 0.50"
] = (
    constrained_selection_df["FP"].astype(int)
    - standard_fp
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "final_validation_threshold_selection.csv"
)

constrained_selection_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("=" * 100)
print("FINAL VALIDATION THRESHOLD SELECTION")
print("=" * 100)

display(
    constrained_selection_df[
        [
            "Operating Point",
            "Model",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "False Negatives Reduced vs 0.50",
            "Additional False Positives vs 0.50",
        ]
    ].round(6)
)

print("\nConstrained security threshold:")
print(float(constrained_row["Threshold"]))

print("\nSaved:")
print(OUTPUT_PATH)

print("\nTest set used: False")

FINAL VALIDATION THRESHOLD SELECTION


,Operating Point,Model,Threshold,Accuracy,Precision,Recall,F1-score,F2-score,FPR,FNR,TP,TN,FP,FN,False Negatives Reduced vs 0.50,Additional False Positives vs 0.50
0,Standard,XGBoost Tuned,0.50,0.944651,0.986925,0.873844,0.926948,0.894338,0.007778,0.126156,16908,28576,224,2441,0,0
1,Unconstrained Maximum F2,XGBoost Tuned,0.16,0.888326,0.809691,0.943976,0.871692,0.913670,0.149062,0.056024,18265,24507,4293,1084,1357,4069
2,Constrained Security Threshold,XGBoost Tuned,0.27,0.932480,0.926912,0.903199,0.914902,0.907844,0.047847,0.096801,17476,27422,1378,1873,568,1154



Constrained security threshold:
0.27

Saved:
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/results/final_validation_threshold_selection.csv

Test set used: False


In [20]:
import shutil
from pathlib import Path

stage4_dir = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2/"
    "stage04_threshold"
)

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage04_threshold_final"
)

archive_file = Path(archive_base + ".zip")

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    stage4_dir,
)

print("Updated Stage 4 archive:")
print(archive_path)

Updated Stage 4 archive:
/kaggle/working/ids2018_clean_validation_v2_stage04_threshold_final.zip


In [21]:
# ============================================================
# STAGE 5 — ONE-TIME UNTOUCHED TEST EVALUATION
#
# Frozen validation decisions:
#   Final model: XGBoost Tuned
#   Standard threshold: 0.50
#   Constrained security threshold: 0.27
#
# No fitting
# No hyperparameter selection
# No threshold selection
# ============================================================

import json
import shutil
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)


# ------------------------------------------------------------
# 1. Frozen configuration
# ------------------------------------------------------------

FINAL_MODEL_NAME = "XGBoost Tuned"

STANDARD_THRESHOLD = 0.50
SECURITY_THRESHOLD = 0.27

VALIDATION_FPR_CONSTRAINT = 0.05


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

MODEL_PATH = (
    ROOT
    / "stage03_top5_tuning"
    / "xgboost"
    / "models"
    / "xgboost_tuned.joblib"
)

THRESHOLD_SELECTION_PATH = (
    ROOT
    / "stage04_threshold"
    / "results"
    / "final_validation_threshold_selection.csv"
)

STAGE_DIR = (
    ROOT
    / "stage05_final_test"
)

RESULTS_DIR = STAGE_DIR / "results"
METADATA_DIR = STAGE_DIR / "metadata"
PREDICTIONS_DIR = STAGE_DIR / "predictions"

for directory in [
    RESULTS_DIR,
    METADATA_DIR,
    PREDICTIONS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

RESULT_PATH = (
    RESULTS_DIR
    / "xgboost_final_test_operating_points.csv"
)

PREDICTION_PATH = (
    PREDICTIONS_DIR
    / "xgboost_final_test_probabilities.csv"
)

METADATA_PATH = (
    METADATA_DIR
    / "final_test_evaluation_metadata.json"
)


# ------------------------------------------------------------
# 3. Verify untouched test arrays
# ------------------------------------------------------------

required_variables = [
    "X_test",
    "y_test",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "The Stage 1 test arrays are unavailable. "
        "Rerun Stage 1 only before executing this cell. "
        f"Missing: {missing_variables}"
    )

X_test_final = np.asarray(
    X_test,
    dtype=np.float32,
)

y_test_final = np.asarray(
    y_test,
    dtype=np.int32,
).reshape(-1)

assert X_test_final.shape == (60_186, 78)
assert y_test_final.shape == (60_186,)

print("Test features:", X_test_final.shape)
print("Test labels:", y_test_final.shape)
print("Benign test records:", int(np.sum(y_test_final == 0)))
print("Attack test records:", int(np.sum(y_test_final == 1)))


# ------------------------------------------------------------
# 4. Verify frozen artifacts
# ------------------------------------------------------------

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Saved tuned XGBoost model not found:\n{MODEL_PATH}"
    )

if not THRESHOLD_SELECTION_PATH.exists():
    raise FileNotFoundError(
        "Frozen validation threshold-selection file "
        f"not found:\n{THRESHOLD_SELECTION_PATH}"
    )

threshold_selection_df = pd.read_csv(
    THRESHOLD_SELECTION_PATH
)

security_rows = threshold_selection_df[
    threshold_selection_df["Operating Point"]
    == "Constrained Security Threshold"
]

if len(security_rows) != 1:
    raise ValueError(
        "Could not identify exactly one constrained "
        "security threshold in the validation file."
    )

saved_security_threshold = float(
    security_rows.iloc[0]["Threshold"]
)

if not np.isclose(
    saved_security_threshold,
    SECURITY_THRESHOLD,
):
    raise ValueError(
        "The hard-coded security threshold does not match "
        "the frozen validation result. "
        f"Expected {saved_security_threshold}, "
        f"received {SECURITY_THRESHOLD}."
    )

print("\nFrozen model:", FINAL_MODEL_NAME)
print("Standard threshold:", STANDARD_THRESHOLD)
print("Security threshold:", SECURITY_THRESHOLD)
print("No test-based decisions will be made.")


# ------------------------------------------------------------
# 5. Load saved tuned model
# ------------------------------------------------------------

model = joblib.load(
    MODEL_PATH
)


# ------------------------------------------------------------
# 6. Generate test probabilities once
# ------------------------------------------------------------

prediction_start = time.perf_counter()

test_probabilities = (
    model.predict_proba(
        X_test_final
    )[:, 1]
)

prediction_time = (
    time.perf_counter()
    - prediction_start
)

if len(test_probabilities) != len(y_test_final):
    raise ValueError(
        "Test probability count does not match test labels."
    )

if np.isnan(test_probabilities).any():
    raise ValueError(
        "Test probabilities contain NaN values."
    )

if (
    (test_probabilities < 0).any()
    or (test_probabilities > 1).any()
):
    raise ValueError(
        "Test probabilities fall outside [0, 1]."
    )


# ------------------------------------------------------------
# 7. Threshold metric function
# ------------------------------------------------------------

def calculate_test_metrics(
    labels,
    probabilities,
    threshold,
    operating_point,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Operating Point": operating_point,
        "Model": FINAL_MODEL_NAME,
        "Evaluation Split": "Untouched Test",
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "Precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "F1-score": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "F2-score": fbeta_score(
            labels,
            predictions,
            beta=2,
            zero_division=0,
        ),
        "FPR": fpr,
        "FNR": fnr,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 8. Evaluate only the frozen thresholds
# ------------------------------------------------------------

standard_result = calculate_test_metrics(
    labels=y_test_final,
    probabilities=test_probabilities,
    threshold=STANDARD_THRESHOLD,
    operating_point="Standard Threshold",
)

security_result = calculate_test_metrics(
    labels=y_test_final,
    probabilities=test_probabilities,
    threshold=SECURITY_THRESHOLD,
    operating_point="Constrained Security Threshold",
)

test_results_df = pd.DataFrame([
    standard_result,
    security_result,
])

standard_fn = int(
    standard_result["FN"]
)

standard_fp = int(
    standard_result["FP"]
)

test_results_df[
    "False Negatives Reduced vs 0.50"
] = (
    standard_fn
    - test_results_df["FN"].astype(int)
)

test_results_df[
    "Additional False Positives vs 0.50"
] = (
    test_results_df["FP"].astype(int)
    - standard_fp
)


# ------------------------------------------------------------
# 9. Threshold-independent metrics
# ------------------------------------------------------------

test_roc_auc = roc_auc_score(
    y_test_final,
    test_probabilities,
)

test_pr_auc = average_precision_score(
    y_test_final,
    test_probabilities,
)

test_results_df[
    "ROC-AUC"
] = test_roc_auc

test_results_df[
    "PR-AUC"
] = test_pr_auc

test_results_df[
    "Probability Prediction Time (sec)"
] = prediction_time


# ------------------------------------------------------------
# 10. Save probabilities and results
# ------------------------------------------------------------

pd.DataFrame({
    "y_true": y_test_final,
    "attack_probability":
        test_probabilities,
}).to_csv(
    PREDICTION_PATH,
    index=False,
)

test_results_df.to_csv(
    RESULT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. Save evaluation metadata
# ------------------------------------------------------------

metadata = {
    "Experiment":
        "IDS2018 Clean Validation V2",
    "Stage":
        "One-time untouched test evaluation",
    "Final Model":
        FINAL_MODEL_NAME,
    "Model Artifact":
        str(MODEL_PATH),
    "Frozen Thresholds": {
        "Standard": STANDARD_THRESHOLD,
        "Constrained Security":
            SECURITY_THRESHOLD,
    },
    "Validation FPR Constraint":
        VALIDATION_FPR_CONSTRAINT,
    "Test Records":
        int(len(y_test_final)),
    "Test Benign":
        int(np.sum(y_test_final == 0)),
    "Test Attack":
        int(np.sum(y_test_final == 1)),
    "Feature Count":
        int(X_test_final.shape[1]),
    "Model Retrained During Stage 5":
        False,
    "Threshold Selected Using Test":
        False,
    "Model Selected Using Test":
        False,
    "Test Evaluations Performed":
        1,
    "ROC-AUC":
        float(test_roc_auc),
    "PR-AUC":
        float(test_pr_auc),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Display final test results
# ------------------------------------------------------------

print("\n" + "=" * 112)
print("FINAL ONE-TIME UNTOUCHED TEST EVALUATION")
print("=" * 112)

display(
    test_results_df[
        [
            "Operating Point",
            "Model",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "False Negatives Reduced vs 0.50",
            "Additional False Positives vs 0.50",
            "ROC-AUC",
            "PR-AUC",
        ]
    ].round(6)
)

print("\nThis table is descriptive only.")
print(
    "No model or threshold will be changed "
    "based on these test results."
)


# ------------------------------------------------------------
# 13. Create Stage 5 archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_stage05_final_test"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(RESULT_PATH)
print(PREDICTION_PATH)
print(METADATA_PATH)

print("\nArchive created:")
print(archive_path)

Test features: (60186, 78)
Test labels: (60186,)
Benign test records: 36000
Attack test records: 24186

Frozen model: XGBoost Tuned
Standard threshold: 0.5
Security threshold: 0.27
No test-based decisions will be made.

FINAL ONE-TIME UNTOUCHED TEST EVALUATION


,Operating Point,Model,Threshold,Accuracy,Precision,Recall,F1-score,F2-score,FPR,FNR,TP,TN,FP,FN,False Negatives Reduced vs 0.50,Additional False Positives vs 0.50,ROC-AUC,PR-AUC
0,Standard Threshold,XGBoost Tuned,0.50,0.945901,0.989202,0.874928,0.928562,0.895620,0.006417,0.125072,21161,35769,231,3025,0,0,0.980191,0.977643
1,Constrained Security Threshold,XGBoost Tuned,0.27,0.934520,0.929775,0.905441,0.917447,0.910206,0.045944,0.094559,21899,34346,1654,2287,738,1423,0.980191,0.977643



This table is descriptive only.
No model or threshold will be changed based on these test results.

Saved:
/kaggle/working/ids2018_clean_validation_v2/stage05_final_test/results/xgboost_final_test_operating_points.csv
/kaggle/working/ids2018_clean_validation_v2/stage05_final_test/predictions/xgboost_final_test_probabilities.csv
/kaggle/working/ids2018_clean_validation_v2/stage05_final_test/metadata/final_test_evaluation_metadata.json

Archive created:
/kaggle/working/ids2018_clean_validation_v2_stage05_final_test.zip


In [22]:
# ============================================================
# STAGE 4B — ALL TOP-FIVE VALIDATION THRESHOLD ANALYSIS
#
# Models:
#   XGBoost Tuned
#   LightGBM Tuned
#   CatBoost Tuned
#   MLP Tuned
#   1D-CNN Tuned
#
# Operating points per model:
#   1. Standard threshold 0.50
#   2. Maximum validation F1
#   3. Unconstrained maximum validation F2
#   4. Maximum validation F2 subject to FPR <= 5%
#
# No training
# No test-set access
# ============================================================

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

FPR_LIMIT = 0.05

THRESHOLDS = np.round(
    np.arange(
        0.05,
        0.951,
        0.01,
    ),
    2,
)


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

TUNING_ROOT = (
    ROOT
    / "stage03_top5_tuning"
)

STAGE4_ROOT = (
    ROOT
    / "stage04_threshold"
)

ALL_MODELS_DIR = (
    STAGE4_ROOT
    / "all_models"
)

RESULTS_DIR = (
    ALL_MODELS_DIR
    / "results"
)

METADATA_DIR = (
    ALL_MODELS_DIR
    / "metadata"
)

for directory in [
    RESULTS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


probability_paths = {
    "XGBoost Tuned": (
        TUNING_ROOT
        / "xgboost"
        / "results"
        / "xgboost_validation_probabilities.csv"
    ),
    "LightGBM Tuned": (
        TUNING_ROOT
        / "lightgbm"
        / "results"
        / "lightgbm_validation_probabilities.csv"
    ),
    "CatBoost Tuned": (
        TUNING_ROOT
        / "catboost"
        / "results"
        / "catboost_validation_probabilities.csv"
    ),
    "MLP Tuned": (
        TUNING_ROOT
        / "mlp"
        / "results"
        / "mlp_validation_probabilities.csv"
    ),
    "1D-CNN Tuned": (
        TUNING_ROOT
        / "cnn"
        / "results"
        / "cnn_validation_probabilities.csv"
    ),
}


FULL_SWEEP_PATH = (
    RESULTS_DIR
    / "all_top5_validation_threshold_sweep.csv"
)

SELECTED_POINTS_PATH = (
    RESULTS_DIR
    / "all_top5_selected_validation_operating_points.csv"
)

MODEL_SUMMARY_PATH = (
    RESULTS_DIR
    / "all_top5_threshold_summary.csv"
)

CROSS_MODEL_LEADERS_PATH = (
    RESULTS_DIR
    / "cross_model_threshold_leaders.csv"
)

METADATA_PATH = (
    METADATA_DIR
    / "all_models_threshold_analysis_metadata.json"
)


# ------------------------------------------------------------
# 3. Verify probability files
# ------------------------------------------------------------

for model_name, path in probability_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Missing probability file for "
            f"{model_name}:\n{path}"
        )

    print(
        f"FOUND {model_name:<18}: "
        f"{path}"
    )


# ------------------------------------------------------------
# 4. Metric function
# ------------------------------------------------------------

def calculate_metrics(
    model_name,
    labels,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if fp + tn > 0
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if fn + tp > 0
        else 0.0
    )

    return {
        "Model": model_name,
        "Evaluation Split": "Validation",
        "Threshold": float(threshold),

        "Accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "Precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),

        "Recall": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),

        "F1-score": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),

        "F2-score": fbeta_score(
            labels,
            predictions,
            beta=2,
            zero_division=0,
        ),

        "FPR": fpr,
        "FNR": fnr,

        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ------------------------------------------------------------
# 5. Run all model-threshold combinations
# ------------------------------------------------------------

all_results = []
reference_labels = None

for model_name, probability_path in (
    probability_paths.items()
):

    probability_df = pd.read_csv(
        probability_path
    )

    required_columns = {
        "y_true",
        "attack_probability",
    }

    missing_columns = (
        required_columns
        - set(probability_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{model_name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    labels = np.asarray(
        probability_df["y_true"],
        dtype=np.int32,
    ).reshape(-1)

    probabilities = np.asarray(
        probability_df[
            "attack_probability"
        ],
        dtype=np.float64,
    ).reshape(-1)

    if len(labels) != 48_149:
        raise ValueError(
            f"{model_name}: expected 48,149 rows, "
            f"found {len(labels):,}."
        )

    if len(probabilities) != len(labels):
        raise ValueError(
            f"{model_name}: label and probability "
            "counts differ."
        )

    if np.isnan(probabilities).any():
        raise ValueError(
            f"{model_name}: probabilities contain NaN."
        )

    if (
        (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise ValueError(
            f"{model_name}: probabilities fall "
            "outside [0, 1]."
        )

    if reference_labels is None:
        reference_labels = labels.copy()

    elif not np.array_equal(
        reference_labels,
        labels,
    ):
        raise ValueError(
            f"{model_name}: validation labels or "
            "record ordering differ from other models."
        )

    print(
        f"\nEvaluating {model_name}: "
        f"{len(THRESHOLDS)} thresholds"
    )

    for threshold in THRESHOLDS:

        all_results.append(
            calculate_metrics(
                model_name=model_name,
                labels=labels,
                probabilities=probabilities,
                threshold=threshold,
            )
        )


sweep_df = pd.DataFrame(
    all_results
)

sweep_df = (
    sweep_df
    .sort_values(
        [
            "Model",
            "Threshold",
        ]
    )
    .reset_index(drop=True)
)

sweep_df.to_csv(
    FULL_SWEEP_PATH,
    index=False,
)


# ------------------------------------------------------------
# 6. Select four operating points per model
# ------------------------------------------------------------

selected_rows = []

for model_name in probability_paths:

    model_df = sweep_df[
        sweep_df["Model"] == model_name
    ].copy()

    standard_row = (
        model_df[
            np.isclose(
                model_df["Threshold"],
                0.50,
            )
        ]
        .iloc[0]
        .copy()
    )

    best_f1_row = (
        model_df
        .sort_values(
            by=[
                "F1-score",
                "Recall",
                "FPR",
                "Threshold",
            ],
            ascending=[
                False,
                False,
                True,
                False,
            ],
        )
        .iloc[0]
        .copy()
    )

    best_f2_row = (
        model_df
        .sort_values(
            by=[
                "F2-score",
                "Recall",
                "F1-score",
                "FPR",
                "Threshold",
            ],
            ascending=[
                False,
                False,
                False,
                True,
                False,
            ],
        )
        .iloc[0]
        .copy()
    )

    eligible_df = model_df[
        model_df["FPR"] <= FPR_LIMIT
    ].copy()

    if eligible_df.empty:
        raise RuntimeError(
            f"No threshold for {model_name} "
            f"satisfies FPR <= {FPR_LIMIT:.2f}."
        )

    constrained_row = (
        eligible_df
        .sort_values(
            by=[
                "F2-score",
                "Recall",
                "F1-score",
                "FPR",
                "Threshold",
            ],
            ascending=[
                False,
                False,
                False,
                True,
                False,
            ],
        )
        .iloc[0]
        .copy()
    )

    operating_points = [
        (
            "Standard Threshold",
            standard_row,
        ),
        (
            "Maximum Validation F1",
            best_f1_row,
        ),
        (
            "Unconstrained Maximum F2",
            best_f2_row,
        ),
        (
            "Constrained Maximum F2",
            constrained_row,
        ),
    ]

    standard_fn = int(
        standard_row["FN"]
    )

    standard_fp = int(
        standard_row["FP"]
    )

    for operating_name, row in operating_points:

        record = {
            "Operating Point":
                operating_name,
            **row.to_dict(),
        }

        record[
            "False Negatives Reduced vs 0.50"
        ] = (
            standard_fn
            - int(row["FN"])
        )

        record[
            "Additional False Positives vs 0.50"
        ] = (
            int(row["FP"])
            - standard_fp
        )

        record[
            "FPR Constraint"
        ] = (
            FPR_LIMIT
            if operating_name
            == "Constrained Maximum F2"
            else np.nan
        )

        selected_rows.append(
            record
        )


selected_df = pd.DataFrame(
    selected_rows
)

selected_df.to_csv(
    SELECTED_POINTS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 7. Create one-row summary per model
# ------------------------------------------------------------

summary_rows = []

for model_name in probability_paths:

    model_selected = selected_df[
        selected_df["Model"] == model_name
    ]

    standard = model_selected[
        model_selected["Operating Point"]
        == "Standard Threshold"
    ].iloc[0]

    max_f1 = model_selected[
        model_selected["Operating Point"]
        == "Maximum Validation F1"
    ].iloc[0]

    max_f2 = model_selected[
        model_selected["Operating Point"]
        == "Unconstrained Maximum F2"
    ].iloc[0]

    constrained = model_selected[
        model_selected["Operating Point"]
        == "Constrained Maximum F2"
    ].iloc[0]

    summary_rows.append({
        "Model": model_name,

        "Standard Threshold":
            float(standard["Threshold"]),
        "Standard F1":
            float(standard["F1-score"]),
        "Standard Recall":
            float(standard["Recall"]),
        "Standard FPR":
            float(standard["FPR"]),
        "Standard FNR":
            float(standard["FNR"]),
        "Standard FP":
            int(standard["FP"]),
        "Standard FN":
            int(standard["FN"]),

        "Best F1 Threshold":
            float(max_f1["Threshold"]),
        "Best Validation F1":
            float(max_f1["F1-score"]),
        "Recall at Best F1":
            float(max_f1["Recall"]),
        "FPR at Best F1":
            float(max_f1["FPR"]),

        "Unconstrained F2 Threshold":
            float(max_f2["Threshold"]),
        "Best Validation F2":
            float(max_f2["F2-score"]),
        "Recall at Best F2":
            float(max_f2["Recall"]),
        "FPR at Best F2":
            float(max_f2["FPR"]),

        "Constrained Threshold":
            float(constrained["Threshold"]),
        "Constrained F2":
            float(constrained["F2-score"]),
        "Constrained F1":
            float(constrained["F1-score"]),
        "Constrained Recall":
            float(constrained["Recall"]),
        "Constrained Precision":
            float(constrained["Precision"]),
        "Constrained FPR":
            float(constrained["FPR"]),
        "Constrained FNR":
            float(constrained["FNR"]),
        "Constrained FP":
            int(constrained["FP"]),
        "Constrained FN":
            int(constrained["FN"]),
        "Constrained FN Reduction":
            int(
                constrained[
                    "False Negatives Reduced vs 0.50"
                ]
            ),
        "Constrained Additional FP":
            int(
                constrained[
                    "Additional False Positives vs 0.50"
                ]
            ),
    })


summary_df = pd.DataFrame(
    summary_rows
)

summary_df = (
    summary_df
    .sort_values(
        [
            "Constrained F2",
            "Constrained Recall",
            "Constrained F1",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

summary_df.to_csv(
    MODEL_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Identify cross-model leaders
# ------------------------------------------------------------

standard_all = selected_df[
    selected_df["Operating Point"]
    == "Standard Threshold"
].copy()

best_f1_all = selected_df[
    selected_df["Operating Point"]
    == "Maximum Validation F1"
].copy()

best_f2_all = selected_df[
    selected_df["Operating Point"]
    == "Unconstrained Maximum F2"
].copy()

constrained_all = selected_df[
    selected_df["Operating Point"]
    == "Constrained Maximum F2"
].copy()


standard_leader = (
    standard_all
    .sort_values(
        [
            "F1-score",
            "Recall",
            "Precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)

best_f1_leader = (
    best_f1_all
    .sort_values(
        [
            "F1-score",
            "Recall",
            "FPR",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .iloc[0]
)

best_f2_leader = (
    best_f2_all
    .sort_values(
        [
            "F2-score",
            "Recall",
            "F1-score",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)

constrained_leader = (
    constrained_all
    .sort_values(
        [
            "F2-score",
            "Recall",
            "F1-score",
            "FPR",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .iloc[0]
)


leaders_df = pd.DataFrame([
    {
        "Selection Objective":
            "Best Standard F1",
        **standard_leader.to_dict(),
    },
    {
        "Selection Objective":
            "Best Threshold-Optimized F1",
        **best_f1_leader.to_dict(),
    },
    {
        "Selection Objective":
            "Best Unconstrained F2",
        **best_f2_leader.to_dict(),
    },
    {
        "Selection Objective":
            "Best Constrained F2",
        **constrained_leader.to_dict(),
    },
])

leaders_df.to_csv(
    CROSS_MODEL_LEADERS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 9. Save metadata
# ------------------------------------------------------------

metadata = {
    "Experiment":
        "IDS2018 Clean Validation V2",

    "Stage":
        "All top-five validation threshold analysis",

    "Models":
        list(probability_paths.keys()),

    "Threshold Grid": {
        "Minimum": float(
            THRESHOLDS.min()
        ),
        "Maximum": float(
            THRESHOLDS.max()
        ),
        "Step": 0.01,
        "Count": int(
            len(THRESHOLDS)
        ),
    },

    "Constrained Objective":
        "Maximum validation F2",

    "FPR Constraint":
        FPR_LIMIT,

    "Threshold Selection Split":
        "Validation",

    "Training Performed":
        False,

    "Test Set Used":
        False,
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Display selected operating points
# ------------------------------------------------------------

print("\n" + "=" * 112)
print("ALL TOP-FIVE SELECTED VALIDATION OPERATING POINTS")
print("=" * 112)

display(
    selected_df[
        [
            "Model",
            "Operating Point",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "FP",
            "FN",
            "False Negatives Reduced vs 0.50",
            "Additional False Positives vs 0.50",
        ]
    ]
    .sort_values(
        [
            "Operating Point",
            "F2-score",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(6)
)


print("\n" + "=" * 112)
print("CONSTRAINED SECURITY COMPARISON — FPR <= 5%")
print("=" * 112)

display(
    constrained_all[
        [
            "Model",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "FP",
            "FN",
            "False Negatives Reduced vs 0.50",
            "Additional False Positives vs 0.50",
        ]
    ]
    .sort_values(
        [
            "F2-score",
            "Recall",
            "F1-score",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .round(6)
)


print("\n" + "=" * 112)
print("CROSS-MODEL OPERATING-POINT LEADERS")
print("=" * 112)

display(
    leaders_df[
        [
            "Selection Objective",
            "Model",
            "Threshold",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "FP",
            "FN",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 11. Create updated Stage 4 archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_"
    "stage04_all_models_thresholds"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE4_ROOT,
)

print("\nSaved:")
print(FULL_SWEEP_PATH)
print(SELECTED_POINTS_PATH)
print(MODEL_SUMMARY_PATH)
print(CROSS_MODEL_LEADERS_PATH)
print(METADATA_PATH)

print("\nArchive created:")
print(archive_path)

print("\nTest set used:")
print(False)

FOUND XGBoost Tuned     : /kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/xgboost/results/xgboost_validation_probabilities.csv
FOUND LightGBM Tuned    : /kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/lightgbm/results/lightgbm_validation_probabilities.csv
FOUND CatBoost Tuned    : /kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/catboost/results/catboost_validation_probabilities.csv
FOUND MLP Tuned         : /kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/mlp/results/mlp_validation_probabilities.csv
FOUND 1D-CNN Tuned      : /kaggle/working/ids2018_clean_validation_v2/stage03_top5_tuning/cnn/results/cnn_validation_probabilities.csv

Evaluating XGBoost Tuned: 91 thresholds

Evaluating LightGBM Tuned: 91 thresholds

Evaluating CatBoost Tuned: 91 thresholds

Evaluating MLP Tuned: 91 thresholds

Evaluating 1D-CNN Tuned: 91 thresholds

ALL TOP-FIVE SELECTED VALIDATION OPERATING POINTS


,Model,Operating Point,Threshold,Accuracy,Precision,Recall,F1-score,F2-score,FPR,FNR,FP,FN,False Negatives Reduced vs 0.50,Additional False Positives vs 0.50
7,LightGBM Tuned,Constrained Maximum F2,0.26,0.932501,0.926735,0.903458,0.914948,0.908019,0.047986,0.096542,1382,1868,597,1178
3,XGBoost Tuned,Constrained Maximum F2,0.27,0.932480,0.926912,0.903199,0.914902,0.907844,0.047847,0.096801,1378,1873,568,1154
11,CatBoost Tuned,Constrained Maximum F2,0.27,0.932024,0.929564,0.898961,0.914006,0.904919,0.045764,0.101039,1318,1955,535,1110
15,MLP Tuned,Constrained Maximum F2,0.24,0.925876,0.936056,0.875342,0.904682,0.886847,0.040174,0.124658,1157,2412,461,1053
19,1D-CNN Tuned,Constrained Maximum F2,0.21,0.922200,0.927222,0.875084,0.900399,0.885037,0.046146,0.124916,1329,2417,476,1129
1,XGBoost Tuned,Maximum Validation F1,0.51,0.944796,0.988012,0.873223,0.927078,0.893997,0.007118,0.126777,205,2453,-12,-19
5,LightGBM Tuned,Maximum Validation F1,0.50,0.944568,0.988062,0.872603,0.926750,0.893485,0.007083,0.127397,204,2465,0,0
9,CatBoost Tuned,Maximum Validation F1,0.52,0.944090,0.989422,0.870174,0.925975,0.891667,0.006250,0.129826,180,2512,-22,-28
13,MLP Tuned,Maximum Validation F1,0.43,0.938275,0.993135,0.852292,0.917339,0.877172,0.003958,0.147708,114,2858,15,10
17,1D-CNN Tuned,Maximum Validation F1,0.65,0.936281,0.992617,0.847744,0.914478,0.873234,0.004236,0.152256,122,2946,-53,-78



CONSTRAINED SECURITY COMPARISON — FPR <= 5%


,Model,Threshold,Accuracy,Precision,Recall,F1-score,F2-score,FPR,FNR,FP,FN,False Negatives Reduced vs 0.50,Additional False Positives vs 0.50
7,LightGBM Tuned,0.26,0.932501,0.926735,0.903458,0.914948,0.908019,0.047986,0.096542,1382,1868,597,1178
3,XGBoost Tuned,0.27,0.932480,0.926912,0.903199,0.914902,0.907844,0.047847,0.096801,1378,1873,568,1154
11,CatBoost Tuned,0.27,0.932024,0.929564,0.898961,0.914006,0.904919,0.045764,0.101039,1318,1955,535,1110
15,MLP Tuned,0.24,0.925876,0.936056,0.875342,0.904682,0.886847,0.040174,0.124658,1157,2412,461,1053
19,1D-CNN Tuned,0.21,0.922200,0.927222,0.875084,0.900399,0.885037,0.046146,0.124916,1329,2417,476,1129



CROSS-MODEL OPERATING-POINT LEADERS


,Selection Objective,Model,Threshold,Precision,Recall,F1-score,F2-score,FPR,FNR,FP,FN
0,Best Standard F1,XGBoost Tuned,0.50,0.986925,0.873844,0.926948,0.894338,0.007778,0.126156,224,2441
1,Best Threshold-Optimized F1,XGBoost Tuned,0.51,0.988012,0.873223,0.927078,0.893997,0.007118,0.126777,205,2453
2,Best Unconstrained F2,XGBoost Tuned,0.16,0.809691,0.943976,0.871692,0.913670,0.149062,0.056024,4293,1084
3,Best Constrained F2,LightGBM Tuned,0.26,0.926735,0.903458,0.914948,0.908019,0.047986,0.096542,1382,1868



Saved:
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/all_models/results/all_top5_validation_threshold_sweep.csv
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/all_models/results/all_top5_selected_validation_operating_points.csv
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/all_models/results/all_top5_threshold_summary.csv
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/all_models/results/cross_model_threshold_leaders.csv
/kaggle/working/ids2018_clean_validation_v2/stage04_threshold/all_models/metadata/all_models_threshold_analysis_metadata.json

Archive created:
/kaggle/working/ids2018_clean_validation_v2_stage04_all_models_thresholds.zip

Test set used:
False


In [23]:
# ============================================================
# STAGE 5B — OBJECTIVE-SPECIFIC FINAL TEST COMPARISON
#
# Validation-selected operating points:
#   Balanced:
#       XGBoost threshold 0.51
#
#   Security-oriented, FPR <= 5%:
#       LightGBM threshold 0.26
#
# Standard threshold 0.50 is also reported for both models.
#
# No training
# No tuning
# No threshold selection using test data
# ============================================================

import json
import shutil
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)


# ------------------------------------------------------------
# 1. Frozen validation decisions
# ------------------------------------------------------------

XGB_STANDARD_THRESHOLD = 0.50
XGB_BALANCED_THRESHOLD = 0.51

LGBM_STANDARD_THRESHOLD = 0.50
LGBM_SECURITY_THRESHOLD = 0.26

FPR_CONSTRAINT = 0.05


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

XGB_TEST_PROBABILITY_PATH = (
    ROOT
    / "stage05_final_test"
    / "predictions"
    / "xgboost_final_test_probabilities.csv"
)

LGBM_MODEL_PATH = (
    ROOT
    / "stage03_top5_tuning"
    / "lightgbm"
    / "models"
    / "lightgbm_tuned.joblib"
)

ALL_MODEL_SELECTION_PATH = (
    ROOT
    / "stage04_threshold"
    / "all_models"
    / "results"
    / "all_top5_selected_validation_operating_points.csv"
)

STAGE_DIR = (
    ROOT
    / "stage05_final_test_comparison"
)

RESULTS_DIR = STAGE_DIR / "results"
PREDICTIONS_DIR = STAGE_DIR / "predictions"
METADATA_DIR = STAGE_DIR / "metadata"

for directory in [
    RESULTS_DIR,
    PREDICTIONS_DIR,
    METADATA_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

RESULT_PATH = (
    RESULTS_DIR
    / "objective_specific_final_test_results.csv"
)

LGBM_PROBABILITY_PATH = (
    PREDICTIONS_DIR
    / "lightgbm_final_test_probabilities.csv"
)

METADATA_PATH = (
    METADATA_DIR
    / "objective_specific_test_metadata.json"
)


# ------------------------------------------------------------
# 3. Verify test arrays
# ------------------------------------------------------------

required_variables = [
    "X_test",
    "y_test",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun Stage 1 only to recreate test arrays. "
        f"Missing variables: {missing_variables}"
    )

X_test_final = np.asarray(
    X_test,
    dtype=np.float32,
)

y_test_final = np.asarray(
    y_test,
    dtype=np.int32,
).reshape(-1)

assert X_test_final.shape == (60_186, 78)
assert y_test_final.shape == (60_186,)

print("Test features:", X_test_final.shape)
print("Test labels:", y_test_final.shape)


# ------------------------------------------------------------
# 4. Verify validation-frozen thresholds
# ------------------------------------------------------------

if not ALL_MODEL_SELECTION_PATH.exists():
    raise FileNotFoundError(
        "All-model validation threshold table missing:\n"
        f"{ALL_MODEL_SELECTION_PATH}"
    )

selection_df = pd.read_csv(
    ALL_MODEL_SELECTION_PATH
)

xgb_f1_row = selection_df[
    (
        selection_df["Model"]
        == "XGBoost Tuned"
    )
    & (
        selection_df["Operating Point"]
        == "Maximum Validation F1"
    )
]

lgbm_security_row = selection_df[
    (
        selection_df["Model"]
        == "LightGBM Tuned"
    )
    & (
        selection_df["Operating Point"]
        == "Constrained Maximum F2"
    )
]

if len(xgb_f1_row) != 1:
    raise ValueError(
        "Could not identify the frozen XGBoost "
        "maximum-validation-F1 point."
    )

if len(lgbm_security_row) != 1:
    raise ValueError(
        "Could not identify the frozen LightGBM "
        "constrained-F2 point."
    )

saved_xgb_threshold = float(
    xgb_f1_row.iloc[0]["Threshold"]
)

saved_lgbm_threshold = float(
    lgbm_security_row.iloc[0]["Threshold"]
)

if not np.isclose(
    saved_xgb_threshold,
    XGB_BALANCED_THRESHOLD,
):
    raise ValueError(
        "XGBoost threshold mismatch. "
        f"Validation file: {saved_xgb_threshold}; "
        f"configured: {XGB_BALANCED_THRESHOLD}."
    )

if not np.isclose(
    saved_lgbm_threshold,
    LGBM_SECURITY_THRESHOLD,
):
    raise ValueError(
        "LightGBM threshold mismatch. "
        f"Validation file: {saved_lgbm_threshold}; "
        f"configured: {LGBM_SECURITY_THRESHOLD}."
    )

print("\nFrozen balanced model:")
print("XGBoost Tuned at", XGB_BALANCED_THRESHOLD)

print("\nFrozen security model:")
print("LightGBM Tuned at", LGBM_SECURITY_THRESHOLD)


# ------------------------------------------------------------
# 5. Load existing XGBoost test probabilities
# ------------------------------------------------------------

if not XGB_TEST_PROBABILITY_PATH.exists():
    raise FileNotFoundError(
        "Existing XGBoost test probabilities missing:\n"
        f"{XGB_TEST_PROBABILITY_PATH}"
    )

xgb_probability_df = pd.read_csv(
    XGB_TEST_PROBABILITY_PATH
)

xgb_labels = np.asarray(
    xgb_probability_df["y_true"],
    dtype=np.int32,
).reshape(-1)

xgb_probabilities = np.asarray(
    xgb_probability_df[
        "attack_probability"
    ],
    dtype=np.float64,
).reshape(-1)

if not np.array_equal(
    xgb_labels,
    y_test_final,
):
    raise ValueError(
        "Existing XGBoost test labels do not match "
        "the recreated deterministic test split."
    )


# ------------------------------------------------------------
# 6. Generate LightGBM probabilities once
# ------------------------------------------------------------

if not LGBM_MODEL_PATH.exists():
    raise FileNotFoundError(
        "Tuned LightGBM model missing:\n"
        f"{LGBM_MODEL_PATH}"
    )

lightgbm_model = joblib.load(
    LGBM_MODEL_PATH
)

prediction_start = time.perf_counter()

lgbm_probabilities = (
    lightgbm_model.predict_proba(
        X_test_final
    )[:, 1]
)

lgbm_prediction_time = (
    time.perf_counter()
    - prediction_start
)

pd.DataFrame({
    "y_true": y_test_final,
    "attack_probability":
        lgbm_probabilities,
}).to_csv(
    LGBM_PROBABILITY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 7. Metric function
# ------------------------------------------------------------

def calculate_metrics(
    model_name,
    objective,
    labels,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    return {
        "Objective": objective,
        "Model": model_name,
        "Evaluation Split": "Holdout Test",
        "Threshold": float(threshold),

        "Accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "Precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),

        "Recall": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),

        "F1-score": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),

        "F2-score": fbeta_score(
            labels,
            predictions,
            beta=2,
            zero_division=0,
        ),

        "FPR": fpr,
        "FNR": fnr,

        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),

        "ROC-AUC": roc_auc_score(
            labels,
            probabilities,
        ),

        "PR-AUC": average_precision_score(
            labels,
            probabilities,
        ),
    }


# ------------------------------------------------------------
# 8. Evaluate frozen operating points
# ------------------------------------------------------------

results = [
    calculate_metrics(
        model_name="XGBoost Tuned",
        objective="Standard Reference",
        labels=y_test_final,
        probabilities=xgb_probabilities,
        threshold=XGB_STANDARD_THRESHOLD,
    ),

    calculate_metrics(
        model_name="XGBoost Tuned",
        objective="Maximum Validation F1",
        labels=y_test_final,
        probabilities=xgb_probabilities,
        threshold=XGB_BALANCED_THRESHOLD,
    ),

    calculate_metrics(
        model_name="LightGBM Tuned",
        objective="Standard Reference",
        labels=y_test_final,
        probabilities=lgbm_probabilities,
        threshold=LGBM_STANDARD_THRESHOLD,
    ),

    calculate_metrics(
        model_name="LightGBM Tuned",
        objective=(
            "Constrained Security Maximum F2"
        ),
        labels=y_test_final,
        probabilities=lgbm_probabilities,
        threshold=LGBM_SECURITY_THRESHOLD,
    ),
]

results_df = pd.DataFrame(
    results
)


# ------------------------------------------------------------
# 9. Add changes relative to each model's 0.50 point
# ------------------------------------------------------------

results_df[
    "False Negatives Reduced vs Model 0.50"
] = 0

results_df[
    "Additional False Positives vs Model 0.50"
] = 0

for model_name in [
    "XGBoost Tuned",
    "LightGBM Tuned",
]:

    model_rows = results_df[
        results_df["Model"] == model_name
    ]

    standard_row = model_rows[
        model_rows["Objective"]
        == "Standard Reference"
    ].iloc[0]

    standard_fn = int(
        standard_row["FN"]
    )

    standard_fp = int(
        standard_row["FP"]
    )

    indices = results_df[
        results_df["Model"] == model_name
    ].index

    results_df.loc[
        indices,
        "False Negatives Reduced vs Model 0.50",
    ] = (
        standard_fn
        - results_df.loc[
            indices,
            "FN",
        ].astype(int)
    )

    results_df.loc[
        indices,
        "Additional False Positives vs Model 0.50",
    ] = (
        results_df.loc[
            indices,
            "FP",
        ].astype(int)
        - standard_fp
    )


results_df.to_csv(
    RESULT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Metadata
# ------------------------------------------------------------

metadata = {
    "Experiment":
        "IDS2018 Clean Validation V2",

    "Stage":
        "Objective-specific final holdout comparison",

    "Balanced Operating Point": {
        "Model":
            "XGBoost Tuned",
        "Threshold":
            XGB_BALANCED_THRESHOLD,
        "Selection Criterion":
            "Maximum validation F1",
    },

    "Security Operating Point": {
        "Model":
            "LightGBM Tuned",
        "Threshold":
            LGBM_SECURITY_THRESHOLD,
        "Selection Criterion":
            "Maximum validation F2 subject to FPR <= 5%",
    },

    "Standard Threshold":
        0.50,

    "Model Selection Used Test":
        False,

    "Threshold Selection Used Test":
        False,

    "Training Performed":
        False,

    "Important Note":
        (
            "The holdout set was evaluated for final "
            "objective-specific reporting. No decisions "
            "were revised using holdout results."
        ),

    "LightGBM Prediction Time (sec)":
        lgbm_prediction_time,
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 11. Display
# ------------------------------------------------------------

print("\n" + "=" * 116)
print("OBJECTIVE-SPECIFIC FINAL HOLDOUT COMPARISON")
print("=" * 116)

display(
    results_df[
        [
            "Objective",
            "Model",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "TP",
            "TN",
            "FP",
            "FN",
            "False Negatives Reduced vs Model 0.50",
            "Additional False Positives vs Model 0.50",
            "ROC-AUC",
            "PR-AUC",
        ]
    ].round(6)
)

print(
    "\nNo model or threshold will be changed "
    "using these holdout results."
)


# ------------------------------------------------------------
# 12. Archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_"
    "stage05_objective_specific_test"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(RESULT_PATH)
print(LGBM_PROBABILITY_PATH)
print(METADATA_PATH)

print("\nArchive created:")
print(archive_path)

Test features: (60186, 78)
Test labels: (60186,)

Frozen balanced model:
XGBoost Tuned at 0.51

Frozen security model:
LightGBM Tuned at 0.26

OBJECTIVE-SPECIFIC FINAL HOLDOUT COMPARISON


,Objective,Model,Threshold,Accuracy,Precision,Recall,F1-score,F2-score,FPR,FNR,TP,TN,FP,FN,False Negatives Reduced vs Model 0.50,Additional False Positives vs Model 0.50,ROC-AUC,PR-AUC
0,Standard Reference,XGBoost Tuned,0.50,0.945901,0.989202,0.874928,0.928562,0.895620,0.006417,0.125072,21161,35769,231,3025,0,0,0.980191,0.977643
1,Maximum Validation F1,XGBoost Tuned,0.51,0.945884,0.989751,0.874390,0.928501,0.895260,0.006083,0.125610,21148,35781,219,3038,-13,-12,0.980191,0.977643
2,Standard Reference,LightGBM Tuned,0.50,0.945752,0.989701,0.874101,0.928316,0.895009,0.006111,0.125899,21141,35780,220,3045,0,0,0.980207,0.977691
3,Constrained Security Maximum F2,LightGBM Tuned,0.26,0.934702,0.929007,0.906806,0.917772,0.911161,0.046556,0.093194,21932,34324,1676,2254,791,1456,0.980207,0.977691



No model or threshold will be changed using these holdout results.

Saved:
/kaggle/working/ids2018_clean_validation_v2/stage05_final_test_comparison/results/objective_specific_final_test_results.csv
/kaggle/working/ids2018_clean_validation_v2/stage05_final_test_comparison/predictions/lightgbm_final_test_probabilities.csv
/kaggle/working/ids2018_clean_validation_v2/stage05_final_test_comparison/metadata/objective_specific_test_metadata.json

Archive created:
/kaggle/working/ids2018_clean_validation_v2_stage05_objective_specific_test.zip


In [24]:
# ============================================================
# STAGE 6 — DUAL-MODEL SHAP EXPLAINABILITY
#
# Models:
#   1. XGBoost — balanced/F1-oriented model
#   2. LightGBM — constrained security-oriented model
#
# Explanation data:
#   Shared deterministic stratified sample of 5,000
#   holdout-test records:
#       2,500 benign
#       2,500 attack
#
# This stage performs:
#   - No training
#   - No model selection
#   - No threshold selection
# ============================================================

import gc
import json
import shutil
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from IPython.display import display
from scipy.stats import spearmanr


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

BENIGN_SAMPLE_SIZE = 2_500
ATTACK_SAMPLE_SIZE = 2_500
TOTAL_SAMPLE_SIZE = (
    BENIGN_SAMPLE_SIZE
    + ATTACK_SAMPLE_SIZE
)

TOP_FEATURE_COUNT = 20

XGB_THRESHOLD = 0.51
LGBM_THRESHOLD = 0.26


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

XGB_MODEL_PATH = (
    ROOT
    / "stage03_top5_tuning"
    / "xgboost"
    / "models"
    / "xgboost_tuned.joblib"
)

LGBM_MODEL_PATH = (
    ROOT
    / "stage03_top5_tuning"
    / "lightgbm"
    / "models"
    / "lightgbm_tuned.joblib"
)

FEATURE_NAMES_PATH = (
    ROOT
    / "stage01_data"
    / "feature_names.json"
)

STAGE_DIR = (
    ROOT
    / "stage06_dual_shap"
)

RESULTS_DIR = STAGE_DIR / "results"
FIGURES_DIR = STAGE_DIR / "figures"
METADATA_DIR = STAGE_DIR / "metadata"

for directory in [
    RESULTS_DIR,
    FIGURES_DIR,
    METADATA_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


SAMPLE_MANIFEST_PATH = (
    RESULTS_DIR
    / "shared_shap_sample_manifest.csv"
)

XGB_SHAP_VALUES_PATH = (
    RESULTS_DIR
    / "xgboost_shap_values.npy"
)

LGBM_SHAP_VALUES_PATH = (
    RESULTS_DIR
    / "lightgbm_shap_values.npy"
)

XGB_TOP20_PATH = (
    RESULTS_DIR
    / "xgboost_shap_top20_features.csv"
)

LGBM_TOP20_PATH = (
    RESULTS_DIR
    / "lightgbm_shap_top20_features.csv"
)

GLOBAL_COMPARISON_PATH = (
    RESULTS_DIR
    / "xgboost_lightgbm_shap_global_comparison.csv"
)

TOP20_OVERLAP_PATH = (
    RESULTS_DIR
    / "xgboost_lightgbm_top20_overlap.csv"
)

METADATA_PATH = (
    METADATA_DIR
    / "dual_model_shap_metadata.json"
)


# ------------------------------------------------------------
# 3. Figure paths
# ------------------------------------------------------------

XGB_SUMMARY_PATH = (
    FIGURES_DIR
    / "xgboost_shap_summary_plot.png"
)

XGB_BAR_PATH = (
    FIGURES_DIR
    / "xgboost_shap_top20_bar_plot.png"
)

XGB_WATERFALL_PATH = (
    FIGURES_DIR
    / "xgboost_shap_attack_waterfall.png"
)

LGBM_SUMMARY_PATH = (
    FIGURES_DIR
    / "lightgbm_shap_summary_plot.png"
)

LGBM_BAR_PATH = (
    FIGURES_DIR
    / "lightgbm_shap_top20_bar_plot.png"
)

LGBM_WATERFALL_PATH = (
    FIGURES_DIR
    / "lightgbm_shap_attack_waterfall.png"
)


# ------------------------------------------------------------
# 4. Verify arrays and artifacts
# ------------------------------------------------------------

required_variables = [
    "X_test",
    "y_test",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Rerun Stage 1 only to recreate the test arrays. "
        f"Missing variables: {missing_variables}"
    )

required_files = [
    XGB_MODEL_PATH,
    LGBM_MODEL_PATH,
    FEATURE_NAMES_PATH,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required artifact not found:\n{path}"
        )


X_test_explain = np.asarray(
    X_test,
    dtype=np.float32,
)

y_test_explain = np.asarray(
    y_test,
    dtype=np.int32,
).reshape(-1)

assert X_test_explain.shape == (60_186, 78)
assert y_test_explain.shape == (60_186,)


with open(
    FEATURE_NAMES_PATH,
    "r",
    encoding="utf-8",
) as file:
    feature_names = json.load(file)

if len(feature_names) != 78:
    raise ValueError(
        f"Expected 78 feature names, "
        f"found {len(feature_names)}."
    )


# ------------------------------------------------------------
# 5. Shared deterministic stratified sample
# ------------------------------------------------------------

rng = np.random.default_rng(
    RANDOM_STATE
)

benign_indices = np.where(
    y_test_explain == 0
)[0]

attack_indices = np.where(
    y_test_explain == 1
)[0]

selected_benign = rng.choice(
    benign_indices,
    size=BENIGN_SAMPLE_SIZE,
    replace=False,
)

selected_attack = rng.choice(
    attack_indices,
    size=ATTACK_SAMPLE_SIZE,
    replace=False,
)

sample_indices = np.concatenate([
    selected_benign,
    selected_attack,
])

rng.shuffle(
    sample_indices
)

X_explain = X_test_explain[
    sample_indices
]

y_explain = y_test_explain[
    sample_indices
]

assert X_explain.shape == (
    TOTAL_SAMPLE_SIZE,
    78,
)

print("Shared explanation sample:", X_explain.shape)
print("Benign:", int(np.sum(y_explain == 0)))
print("Attack:", int(np.sum(y_explain == 1)))


# ------------------------------------------------------------
# 6. Load models
# ------------------------------------------------------------

xgb_model = joblib.load(
    XGB_MODEL_PATH
)

lgbm_model = joblib.load(
    LGBM_MODEL_PATH
)

# Explanation can run on CPU without altering
# the saved model artifacts.
try:
    xgb_model.set_params(
        device="cpu"
    )
except Exception:
    pass


# ------------------------------------------------------------
# 7. Generate probabilities for shared sample
# ------------------------------------------------------------

xgb_probabilities = (
    xgb_model.predict_proba(
        X_explain
    )[:, 1]
)

lgbm_probabilities = (
    lgbm_model.predict_proba(
        X_explain
    )[:, 1]
)

sample_manifest = pd.DataFrame({
    "Test Position": sample_indices,
    "True Label": y_explain,
    "XGBoost Attack Probability":
        xgb_probabilities,
    "LightGBM Attack Probability":
        lgbm_probabilities,
})

sample_manifest.to_csv(
    SAMPLE_MANIFEST_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Robust SHAP extraction function
# ------------------------------------------------------------

def extract_binary_shap_values(
    model,
    features,
):
    explainer = shap.TreeExplainer(
        model
    )

    try:
        raw_values = explainer.shap_values(
            features,
            check_additivity=False,
        )
    except TypeError:
        raw_values = explainer.shap_values(
            features
        )

    if isinstance(
        raw_values,
        list,
    ):
        values = np.asarray(
            raw_values[-1]
        )

    elif hasattr(
        raw_values,
        "values",
    ):
        values = np.asarray(
            raw_values.values
        )

    else:
        values = np.asarray(
            raw_values
        )

    if values.ndim == 3:

        if values.shape[-1] == 2:
            values = values[:, :, 1]

        elif values.shape[0] == 2:
            values = values[1]

    if values.shape != features.shape:
        raise ValueError(
            "Unexpected SHAP matrix shape. "
            f"Expected {features.shape}; "
            f"received {values.shape}."
        )

    expected_value = (
        explainer.expected_value
    )

    expected_array = np.asarray(
        expected_value
    ).reshape(-1)

    base_value = float(
        expected_array[-1]
    )

    return (
        explainer,
        values,
        base_value,
    )


# ------------------------------------------------------------
# 9. Calculate XGBoost SHAP values
# ------------------------------------------------------------

print("\nCalculating XGBoost SHAP values...")

(
    xgb_explainer,
    xgb_shap_values,
    xgb_base_value,
) = extract_binary_shap_values(
    model=xgb_model,
    features=X_explain,
)

np.save(
    XGB_SHAP_VALUES_PATH,
    xgb_shap_values,
)

print(
    "XGBoost SHAP shape:",
    xgb_shap_values.shape,
)


# ------------------------------------------------------------
# 10. Calculate LightGBM SHAP values
# ------------------------------------------------------------

print("\nCalculating LightGBM SHAP values...")

(
    lgbm_explainer,
    lgbm_shap_values,
    lgbm_base_value,
) = extract_binary_shap_values(
    model=lgbm_model,
    features=X_explain,
)

np.save(
    LGBM_SHAP_VALUES_PATH,
    lgbm_shap_values,
)

print(
    "LightGBM SHAP shape:",
    lgbm_shap_values.shape,
)


# ------------------------------------------------------------
# 11. Global feature importance
# ------------------------------------------------------------

xgb_mean_abs_shap = np.mean(
    np.abs(xgb_shap_values),
    axis=0,
)

lgbm_mean_abs_shap = np.mean(
    np.abs(lgbm_shap_values),
    axis=0,
)


comparison_df = pd.DataFrame({
    "Feature": feature_names,
    "XGBoost Mean Absolute SHAP":
        xgb_mean_abs_shap,
    "LightGBM Mean Absolute SHAP":
        lgbm_mean_abs_shap,
})


comparison_df[
    "XGBoost Normalized Importance"
] = (
    comparison_df[
        "XGBoost Mean Absolute SHAP"
    ]
    / comparison_df[
        "XGBoost Mean Absolute SHAP"
    ].sum()
)

comparison_df[
    "LightGBM Normalized Importance"
] = (
    comparison_df[
        "LightGBM Mean Absolute SHAP"
    ]
    / comparison_df[
        "LightGBM Mean Absolute SHAP"
    ].sum()
)


comparison_df[
    "XGBoost Rank"
] = (
    comparison_df[
        "XGBoost Mean Absolute SHAP"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

comparison_df[
    "LightGBM Rank"
] = (
    comparison_df[
        "LightGBM Mean Absolute SHAP"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

comparison_df[
    "Absolute Rank Difference"
] = np.abs(
    comparison_df["XGBoost Rank"]
    - comparison_df["LightGBM Rank"]
)

comparison_df = (
    comparison_df
    .sort_values(
        [
            "XGBoost Rank",
            "LightGBM Rank",
        ]
    )
    .reset_index(drop=True)
)

comparison_df.to_csv(
    GLOBAL_COMPARISON_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. Individual top-20 tables
# ------------------------------------------------------------

xgb_top_df = (
    comparison_df[
        [
            "Feature",
            "XGBoost Mean Absolute SHAP",
            "XGBoost Normalized Importance",
            "XGBoost Rank",
        ]
    ]
    .sort_values(
        "XGBoost Rank"
    )
    .head(TOP_FEATURE_COUNT)
    .reset_index(drop=True)
)

xgb_top_df.insert(
    0,
    "Display Rank",
    range(
        1,
        len(xgb_top_df) + 1,
    ),
)

xgb_top_df.to_csv(
    XGB_TOP20_PATH,
    index=False,
)


lgbm_top_df = (
    comparison_df[
        [
            "Feature",
            "LightGBM Mean Absolute SHAP",
            "LightGBM Normalized Importance",
            "LightGBM Rank",
        ]
    ]
    .sort_values(
        "LightGBM Rank"
    )
    .head(TOP_FEATURE_COUNT)
    .reset_index(drop=True)
)

lgbm_top_df.insert(
    0,
    "Display Rank",
    range(
        1,
        len(lgbm_top_df) + 1,
    ),
)

lgbm_top_df.to_csv(
    LGBM_TOP20_PATH,
    index=False,
)


# ------------------------------------------------------------
# 13. Top-20 overlap analysis
# ------------------------------------------------------------

xgb_top_features = set(
    xgb_top_df["Feature"]
)

lgbm_top_features = set(
    lgbm_top_df["Feature"]
)

shared_top_features = (
    xgb_top_features
    & lgbm_top_features
)

top_feature_union = (
    xgb_top_features
    | lgbm_top_features
)

jaccard_similarity = (
    len(shared_top_features)
    / len(top_feature_union)
)


overlap_df = comparison_df[
    comparison_df["Feature"].isin(
        top_feature_union
    )
].copy()

overlap_df[
    "In XGBoost Top 20"
] = overlap_df[
    "Feature"
].isin(
    xgb_top_features
)

overlap_df[
    "In LightGBM Top 20"
] = overlap_df[
    "Feature"
].isin(
    lgbm_top_features
)

overlap_df[
    "Shared Top 20"
] = (
    overlap_df[
        "In XGBoost Top 20"
    ]
    & overlap_df[
        "In LightGBM Top 20"
    ]
)

overlap_df = (
    overlap_df
    .sort_values(
        [
            "Shared Top 20",
            "XGBoost Rank",
            "LightGBM Rank",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

overlap_df.to_csv(
    TOP20_OVERLAP_PATH,
    index=False,
)


# ------------------------------------------------------------
# 14. Rank correlation
# ------------------------------------------------------------

rank_correlation, rank_p_value = (
    spearmanr(
        comparison_df[
            "XGBoost Rank"
        ],
        comparison_df[
            "LightGBM Rank"
        ],
    )
)


# ------------------------------------------------------------
# 15. Summary and bar plot function
# ------------------------------------------------------------

def save_summary_plots(
    shap_values,
    features,
    feature_names,
    summary_path,
    bar_path,
):
    plt.figure()

    shap.summary_plot(
        shap_values,
        features,
        feature_names=feature_names,
        max_display=TOP_FEATURE_COUNT,
        show=False,
    )

    plt.tight_layout()

    plt.savefig(
        summary_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


    plt.figure()

    shap.summary_plot(
        shap_values,
        features,
        feature_names=feature_names,
        plot_type="bar",
        max_display=TOP_FEATURE_COUNT,
        show=False,
    )

    plt.tight_layout()

    plt.savefig(
        bar_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


save_summary_plots(
    shap_values=xgb_shap_values,
    features=X_explain,
    feature_names=feature_names,
    summary_path=XGB_SUMMARY_PATH,
    bar_path=XGB_BAR_PATH,
)

save_summary_plots(
    shap_values=lgbm_shap_values,
    features=X_explain,
    feature_names=feature_names,
    summary_path=LGBM_SUMMARY_PATH,
    bar_path=LGBM_BAR_PATH,
)


# ------------------------------------------------------------
# 16. Select representative attacks
# ------------------------------------------------------------

def select_representative_attack(
    labels,
    probabilities,
    threshold,
):
    correctly_detected = np.where(
        (labels == 1)
        & (probabilities >= threshold)
    )[0]

    if len(correctly_detected) > 0:
        return int(
            correctly_detected[
                np.argmax(
                    probabilities[
                        correctly_detected
                    ]
                )
            ]
        )

    attack_positions = np.where(
        labels == 1
    )[0]

    return int(
        attack_positions[
            np.argmax(
                probabilities[
                    attack_positions
                ]
            )
        ]
    )


xgb_waterfall_position = (
    select_representative_attack(
        labels=y_explain,
        probabilities=xgb_probabilities,
        threshold=XGB_THRESHOLD,
    )
)

lgbm_waterfall_position = (
    select_representative_attack(
        labels=y_explain,
        probabilities=lgbm_probabilities,
        threshold=LGBM_THRESHOLD,
    )
)


# ------------------------------------------------------------
# 17. Waterfall plot function
# ------------------------------------------------------------

def save_waterfall_plot(
    shap_values,
    base_value,
    features,
    feature_names,
    sample_position,
    output_path,
):
    explanation = shap.Explanation(
        values=shap_values[
            sample_position
        ],
        base_values=base_value,
        data=features[
            sample_position
        ],
        feature_names=feature_names,
    )

    plt.figure()

    shap.plots.waterfall(
        explanation,
        max_display=TOP_FEATURE_COUNT,
        show=False,
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


save_waterfall_plot(
    shap_values=xgb_shap_values,
    base_value=xgb_base_value,
    features=X_explain,
    feature_names=feature_names,
    sample_position=xgb_waterfall_position,
    output_path=XGB_WATERFALL_PATH,
)

save_waterfall_plot(
    shap_values=lgbm_shap_values,
    base_value=lgbm_base_value,
    features=X_explain,
    feature_names=feature_names,
    sample_position=lgbm_waterfall_position,
    output_path=LGBM_WATERFALL_PATH,
)


# ------------------------------------------------------------
# 18. Save metadata
# ------------------------------------------------------------

metadata = {
    "Experiment":
        "IDS2018 Clean Validation V2",

    "Stage":
        "Dual-model post-evaluation SHAP analysis",

    "Models": {
        "Balanced Model":
            "XGBoost Tuned",
        "Balanced Threshold":
            XGB_THRESHOLD,
        "Security Model":
            "LightGBM Tuned",
        "Security Threshold":
            LGBM_THRESHOLD,
    },

    "Explanation Split":
        "Shared stratified sample from holdout test set",

    "Total Explanation Records":
        TOTAL_SAMPLE_SIZE,

    "Benign Explanation Records":
        BENIGN_SAMPLE_SIZE,

    "Attack Explanation Records":
        ATTACK_SAMPLE_SIZE,

    "Random State":
        RANDOM_STATE,

    "Feature Count":
        len(feature_names),

    "Top Features Reported":
        TOP_FEATURE_COUNT,

    "Top-20 Shared Feature Count":
        len(shared_top_features),

    "Top-20 Jaccard Similarity":
        float(jaccard_similarity),

    "All-Feature Spearman Rank Correlation":
        float(rank_correlation),

    "Spearman P-value":
        float(rank_p_value),

    "XGBoost Waterfall Test Position":
        int(
            sample_indices[
                xgb_waterfall_position
            ]
        ),

    "XGBoost Waterfall Probability":
        float(
            xgb_probabilities[
                xgb_waterfall_position
            ]
        ),

    "LightGBM Waterfall Test Position":
        int(
            sample_indices[
                lgbm_waterfall_position
            ]
        ),

    "LightGBM Waterfall Probability":
        float(
            lgbm_probabilities[
                lgbm_waterfall_position
            ]
        ),

    "Training Performed":
        False,

    "Model Selection Performed":
        False,

    "Threshold Selection Performed":
        False,
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

print("\n" + "=" * 94)
print("XGBOOST TOP 20 SHAP FEATURES")
print("=" * 94)

display(
    xgb_top_df.round(6)
)


print("\n" + "=" * 94)
print("LIGHTGBM TOP 20 SHAP FEATURES")
print("=" * 94)

display(
    lgbm_top_df.round(6)
)


print("\n" + "=" * 94)
print("SHAP FEATURE-RANKING AGREEMENT")
print("=" * 94)

print(
    "Shared top-20 features:",
    len(shared_top_features),
)

print(
    "Top-20 Jaccard similarity:",
    round(
        jaccard_similarity,
        6,
    ),
)

print(
    "All-feature Spearman correlation:",
    round(
        rank_correlation,
        6,
    ),
)

print(
    "Spearman p-value:",
    rank_p_value,
)


print("\nFigures:")

for path in [
    XGB_SUMMARY_PATH,
    XGB_BAR_PATH,
    XGB_WATERFALL_PATH,
    LGBM_SUMMARY_PATH,
    LGBM_BAR_PATH,
    LGBM_WATERFALL_PATH,
]:
    print(path)


# ------------------------------------------------------------
# 20. Archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_"
    "stage06_dual_shap"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nSaved:")
print(SAMPLE_MANIFEST_PATH)
print(XGB_TOP20_PATH)
print(LGBM_TOP20_PATH)
print(GLOBAL_COMPARISON_PATH)
print(TOP20_OVERLAP_PATH)
print(METADATA_PATH)

print("\nArchive created:")
print(archive_path)

Shared explanation sample: (5000, 78)
Benign: 2500
Attack: 2500

Calculating XGBoost SHAP values...
XGBoost SHAP shape: (5000, 78)

Calculating LightGBM SHAP values...
LightGBM SHAP shape: (5000, 78)

XGBOOST TOP 20 SHAP FEATURES


,Display Rank,Feature,XGBoost Mean Absolute SHAP,XGBoost Normalized Importance,XGBoost Rank
0,1,Init Fwd Win Byts,1.422182,0.187187,1
1,2,Fwd Seg Size Min,1.155656,0.152107,2
2,3,Dst Port,1.070499,0.140898,3
3,4,Fwd Pkt Len Max,0.393898,0.051845,4
4,5,Bwd Pkt Len Mean,0.260515,0.034289,5
5,6,RST Flag Cnt,0.246533,0.032448,6
6,7,Fwd Pkt Len Std,0.245038,0.032252,7
7,8,Bwd Pkt Len Max,0.182554,0.024028,8
8,9,Flow IAT Min,0.175497,0.023099,9
9,10,Fwd IAT Min,0.161499,0.021256,10



LIGHTGBM TOP 20 SHAP FEATURES


,Display Rank,Feature,LightGBM Mean Absolute SHAP,LightGBM Normalized Importance,LightGBM Rank
0,1,Init Fwd Win Byts,1.643697,0.211158,1
1,2,Fwd Seg Size Min,1.172440,0.150618,2
2,3,Dst Port,0.965930,0.124089,3
3,4,Fwd Pkt Len Std,0.566981,0.072837,4
4,5,Fwd Pkt Len Max,0.544401,0.069937,5
5,6,Init Bwd Win Byts,0.198953,0.025559,6
6,7,Bwd Pkt Len Mean,0.196670,0.025265,7
7,8,ECE Flag Cnt,0.174509,0.022418,8
8,9,Fwd IAT Min,0.152891,0.019641,9
9,10,RST Flag Cnt,0.135951,0.017465,10



SHAP FEATURE-RANKING AGREEMENT
Shared top-20 features: 15
Top-20 Jaccard similarity: 0.6
All-feature Spearman correlation: 0.854066
Spearman p-value: 2.839526154790284e-23

Figures:
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/figures/xgboost_shap_summary_plot.png
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/figures/xgboost_shap_top20_bar_plot.png
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/figures/xgboost_shap_attack_waterfall.png
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/figures/lightgbm_shap_summary_plot.png
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/figures/lightgbm_shap_top20_bar_plot.png
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/figures/lightgbm_shap_attack_waterfall.png

Saved:
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/results/shared_shap_sample_manifest.csv
/kaggle/working/ids2018_clean_validation_v2/stage06_dual_shap/results/xgboost_shap_top20_fea

In [25]:
# ============================================================
# STAGE 7 — PUBLICATION-READY FIGURES AND FINAL TABLES
#
# Generates:
#   1. Sixteen-model baseline F1 comparison
#   2. Tuned top-five F1 comparison
#   3. Constrained-security F2 comparison
#   4. XGBoost validation threshold trade-off
#   5. LightGBM validation threshold trade-off
#   6. Final objective-specific holdout comparison
#   7. XGBoost holdout confusion matrix
#   8. LightGBM security confusion matrix
#   9. SHAP feature-rank agreement
#  10. Publication CSV and LaTeX tables
#
# No training or model selection.
# ============================================================

import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import confusion_matrix


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "ids2018_clean_validation_v2"
)

STAGE_DIR = (
    ROOT
    / "stage07_publication_assets"
)

FIGURES_DIR = STAGE_DIR / "figures"
TABLES_DIR = STAGE_DIR / "tables"
METADATA_DIR = STAGE_DIR / "metadata"

for directory in [
    FIGURES_DIR,
    TABLES_DIR,
    METADATA_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


BASELINE16_PATH = (
    ROOT
    / "stage02_baselines"
    / "results"
    / "final_16_model_validation_ablation.csv"
)

TOP5_TUNED_PATH = (
    ROOT
    / "stage03_top5_tuning"
    / "combined_results"
    / "top5_tuned_validation_results.csv"
)

THRESHOLD_SWEEP_PATH = (
    ROOT
    / "stage04_threshold"
    / "all_models"
    / "results"
    / "all_top5_validation_threshold_sweep.csv"
)

SELECTED_THRESHOLD_PATH = (
    ROOT
    / "stage04_threshold"
    / "all_models"
    / "results"
    / "all_top5_selected_validation_operating_points.csv"
)

HOLDOUT_RESULTS_PATH = (
    ROOT
    / "stage05_final_test_comparison"
    / "results"
    / "objective_specific_final_test_results.csv"
)

XGB_TEST_PROBABILITY_PATH = (
    ROOT
    / "stage05_final_test"
    / "predictions"
    / "xgboost_final_test_probabilities.csv"
)

LGBM_TEST_PROBABILITY_PATH = (
    ROOT
    / "stage05_final_test_comparison"
    / "predictions"
    / "lightgbm_final_test_probabilities.csv"
)

SHAP_COMPARISON_PATH = (
    ROOT
    / "stage06_dual_shap"
    / "results"
    / "xgboost_lightgbm_shap_global_comparison.csv"
)

SHAP_FIGURES_SOURCE = (
    ROOT
    / "stage06_dual_shap"
    / "figures"
)


# ------------------------------------------------------------
# 2. Verify required inputs
# ------------------------------------------------------------

required_files = [
    BASELINE16_PATH,
    TOP5_TUNED_PATH,
    THRESHOLD_SWEEP_PATH,
    SELECTED_THRESHOLD_PATH,
    HOLDOUT_RESULTS_PATH,
    XGB_TEST_PROBABILITY_PATH,
    LGBM_TEST_PROBABILITY_PATH,
    SHAP_COMPARISON_PATH,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )

print("All required Stage 7 inputs are available.")


# ------------------------------------------------------------
# 3. Load tables
# ------------------------------------------------------------

baseline_df = pd.read_csv(
    BASELINE16_PATH
)

tuned_df = pd.read_csv(
    TOP5_TUNED_PATH
)

threshold_sweep_df = pd.read_csv(
    THRESHOLD_SWEEP_PATH
)

selected_threshold_df = pd.read_csv(
    SELECTED_THRESHOLD_PATH
)

holdout_df = pd.read_csv(
    HOLDOUT_RESULTS_PATH
)

shap_comparison_df = pd.read_csv(
    SHAP_COMPARISON_PATH
)


# ------------------------------------------------------------
# 4. Figure helper
# ------------------------------------------------------------

def save_current_figure(
    output_path,
    width=10,
    height=6,
):
    figure = plt.gcf()
    figure.set_size_inches(
        width,
        height,
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


# ------------------------------------------------------------
# 5. Figure 1 — 16-model validation F1 comparison
# ------------------------------------------------------------

baseline_plot_df = (
    baseline_df
    .sort_values(
        "F1-score",
        ascending=True,
    )
)

plt.figure()

plt.barh(
    baseline_plot_df["Model"],
    baseline_plot_df["F1-score"],
)

plt.xlabel("Validation F1-score")
plt.ylabel("Model")
plt.title(
    "Baseline Validation Performance Across 16 Models"
)

plt.xlim(
    max(
        0.0,
        baseline_plot_df["F1-score"].min() - 0.05,
    ),
    1.0,
)

plt.grid(
    axis="x",
    alpha=0.25,
)

save_current_figure(
    FIGURES_DIR
    / "figure01_baseline16_f1_comparison.png",
    width=10,
    height=8,
)


# ------------------------------------------------------------
# 6. Figure 2 — Tuned top-five comparison
# ------------------------------------------------------------

tuned_plot_df = (
    tuned_df
    .sort_values(
        "F1-score",
        ascending=True,
    )
)

plt.figure()

plt.barh(
    tuned_plot_df["Model"],
    tuned_plot_df["F1-score"],
)

plt.xlabel("Validation F1-score")
plt.ylabel("Tuned model")
plt.title(
    "Validation Performance of the Five Tuned Models"
)

plt.xlim(
    tuned_plot_df["F1-score"].min() - 0.01,
    tuned_plot_df["F1-score"].max() + 0.005,
)

plt.grid(
    axis="x",
    alpha=0.25,
)

save_current_figure(
    FIGURES_DIR
    / "figure02_tuned_top5_f1_comparison.png",
    width=9,
    height=5,
)


# ------------------------------------------------------------
# 7. Figure 3 — Constrained security comparison
# ------------------------------------------------------------

constrained_df = selected_threshold_df[
    selected_threshold_df["Operating Point"]
    == "Constrained Maximum F2"
].copy()

constrained_df = constrained_df.sort_values(
    "F2-score",
    ascending=True,
)

plt.figure()

plt.barh(
    constrained_df["Model"],
    constrained_df["F2-score"],
)

plt.xlabel("Validation F2-score")
plt.ylabel("Tuned model")
plt.title(
    "Security-Oriented Threshold Performance "
    "Under FPR ≤ 5%"
)

plt.xlim(
    constrained_df["F2-score"].min() - 0.02,
    constrained_df["F2-score"].max() + 0.005,
)

plt.grid(
    axis="x",
    alpha=0.25,
)

save_current_figure(
    FIGURES_DIR
    / "figure03_constrained_security_f2_comparison.png",
    width=9,
    height=5,
)


# ------------------------------------------------------------
# 8. Figure 4 — XGBoost threshold trade-off
# ------------------------------------------------------------

xgb_sweep = threshold_sweep_df[
    threshold_sweep_df["Model"]
    == "XGBoost Tuned"
].copy()

plt.figure()

plt.plot(
    xgb_sweep["Threshold"],
    xgb_sweep["Recall"],
    label="Recall",
)

plt.plot(
    xgb_sweep["Threshold"],
    xgb_sweep["Precision"],
    label="Precision",
)

plt.plot(
    xgb_sweep["Threshold"],
    xgb_sweep["F1-score"],
    label="F1-score",
)

plt.plot(
    xgb_sweep["Threshold"],
    xgb_sweep["F2-score"],
    label="F2-score",
)

plt.axvline(
    0.50,
    linestyle="--",
    label="Standard threshold 0.50",
)

plt.axvline(
    0.51,
    linestyle=":",
    label="Maximum-F1 threshold 0.51",
)

plt.xlabel("Decision threshold")
plt.ylabel("Metric value")
plt.title(
    "XGBoost Validation Threshold Trade-off"
)

plt.legend()
plt.grid(alpha=0.25)

save_current_figure(
    FIGURES_DIR
    / "figure04_xgboost_threshold_tradeoff.png",
    width=10,
    height=6,
)


# ------------------------------------------------------------
# 9. Figure 5 — LightGBM threshold trade-off
# ------------------------------------------------------------

lgbm_sweep = threshold_sweep_df[
    threshold_sweep_df["Model"]
    == "LightGBM Tuned"
].copy()

plt.figure()

plt.plot(
    lgbm_sweep["Threshold"],
    lgbm_sweep["Recall"],
    label="Recall",
)

plt.plot(
    lgbm_sweep["Threshold"],
    lgbm_sweep["Precision"],
    label="Precision",
)

plt.plot(
    lgbm_sweep["Threshold"],
    lgbm_sweep["F1-score"],
    label="F1-score",
)

plt.plot(
    lgbm_sweep["Threshold"],
    lgbm_sweep["F2-score"],
    label="F2-score",
)

plt.plot(
    lgbm_sweep["Threshold"],
    lgbm_sweep["FPR"],
    label="FPR",
)

plt.axhline(
    0.05,
    linestyle="--",
    label="FPR constraint 0.05",
)

plt.axvline(
    0.26,
    linestyle=":",
    label="Security threshold 0.26",
)

plt.xlabel("Decision threshold")
plt.ylabel("Metric value")
plt.title(
    "LightGBM Security-Oriented Threshold Trade-off"
)

plt.legend()
plt.grid(alpha=0.25)

save_current_figure(
    FIGURES_DIR
    / "figure05_lightgbm_threshold_tradeoff.png",
    width=10,
    height=6,
)


# ------------------------------------------------------------
# 10. Figure 6 — Final holdout comparison
# ------------------------------------------------------------

final_operating_df = holdout_df[
    holdout_df["Objective"].isin([
        "Maximum Validation F1",
        "Constrained Security Maximum F2",
    ])
].copy()

labels = [
    f"{row['Model']}\n"
    f"Threshold {row['Threshold']:.2f}"
    for _, row in final_operating_df.iterrows()
]

x_positions = np.arange(
    len(final_operating_df)
)

bar_width = 0.34

plt.figure()

plt.bar(
    x_positions - bar_width / 2,
    final_operating_df["F1-score"],
    width=bar_width,
    label="F1-score",
)

plt.bar(
    x_positions + bar_width / 2,
    final_operating_df["F2-score"],
    width=bar_width,
    label="F2-score",
)

plt.xticks(
    x_positions,
    labels,
)

plt.ylabel("Holdout-test score")
plt.title(
    "Final Objective-Specific Holdout Performance"
)

plt.ylim(
    min(
        final_operating_df[
            ["F1-score", "F2-score"]
        ].min()
    ) - 0.03,
    1.0,
)

plt.legend()
plt.grid(
    axis="y",
    alpha=0.25,
)

save_current_figure(
    FIGURES_DIR
    / "figure06_final_holdout_objective_comparison.png",
    width=9,
    height=6,
)


# ------------------------------------------------------------
# 11. Confusion-matrix function
# ------------------------------------------------------------

def save_confusion_matrix(
    y_true,
    probabilities,
    threshold,
    title,
    output_path,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    matrix = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    )

    plt.figure()

    plt.imshow(
        matrix,
    )

    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(2)

    plt.xticks(
        tick_marks,
        ["Benign", "Attack"],
    )

    plt.yticks(
        tick_marks,
        ["Benign", "Attack"],
    )

    plt.xlabel("Predicted class")
    plt.ylabel("True class")

    threshold_value = (
        matrix.max() / 2.0
    )

    for row in range(2):
        for column in range(2):
            plt.text(
                column,
                row,
                f"{matrix[row, column]:,}",
                horizontalalignment="center",
                verticalalignment="center",
                color=(
                    "white"
                    if matrix[row, column]
                    > threshold_value
                    else "black"
                ),
                fontsize=12,
            )

    save_current_figure(
        output_path,
        width=6,
        height=5,
    )


# ------------------------------------------------------------
# 12. Figure 7 — XGBoost confusion matrix
# ------------------------------------------------------------

xgb_test_df = pd.read_csv(
    XGB_TEST_PROBABILITY_PATH
)

save_confusion_matrix(
    y_true=np.asarray(
        xgb_test_df["y_true"],
        dtype=np.int32,
    ),
    probabilities=np.asarray(
        xgb_test_df["attack_probability"],
        dtype=np.float64,
    ),
    threshold=0.51,
    title=(
        "XGBoost Holdout Confusion Matrix "
        "(Threshold 0.51)"
    ),
    output_path=(
        FIGURES_DIR
        / "figure07_xgboost_holdout_confusion_matrix.png"
    ),
)


# ------------------------------------------------------------
# 13. Figure 8 — LightGBM confusion matrix
# ------------------------------------------------------------

lgbm_test_df = pd.read_csv(
    LGBM_TEST_PROBABILITY_PATH
)

save_confusion_matrix(
    y_true=np.asarray(
        lgbm_test_df["y_true"],
        dtype=np.int32,
    ),
    probabilities=np.asarray(
        lgbm_test_df["attack_probability"],
        dtype=np.float64,
    ),
    threshold=0.26,
    title=(
        "LightGBM Security Holdout Confusion Matrix "
        "(Threshold 0.26)"
    ),
    output_path=(
        FIGURES_DIR
        / "figure08_lightgbm_security_confusion_matrix.png"
    ),
)


# ------------------------------------------------------------
# 14. Figure 9 — SHAP feature-rank agreement
# ------------------------------------------------------------

rank_plot_df = shap_comparison_df.copy()

plt.figure()

plt.scatter(
    rank_plot_df["XGBoost Rank"],
    rank_plot_df["LightGBM Rank"],
)

maximum_rank = max(
    rank_plot_df["XGBoost Rank"].max(),
    rank_plot_df["LightGBM Rank"].max(),
)

plt.plot(
    [1, maximum_rank],
    [1, maximum_rank],
    linestyle="--",
    label="Identical rank",
)

plt.xlabel("XGBoost feature rank")
plt.ylabel("LightGBM feature rank")
plt.title(
    "Agreement Between XGBoost and LightGBM "
    "SHAP Feature Rankings"
)

plt.legend()
plt.grid(alpha=0.25)

save_current_figure(
    FIGURES_DIR
    / "figure09_shap_rank_agreement.png",
    width=7,
    height=7,
)


# ------------------------------------------------------------
# 15. Copy existing SHAP figures
# ------------------------------------------------------------

if SHAP_FIGURES_SOURCE.exists():

    for source_path in (
        SHAP_FIGURES_SOURCE.glob("*.png")
    ):
        destination_path = (
            FIGURES_DIR
            / source_path.name
        )

        shutil.copy2(
            source_path,
            destination_path,
        )


# ------------------------------------------------------------
# 16. Save publication-ready CSV tables
# ------------------------------------------------------------

baseline_publication = baseline_df[
    [
        "Rank",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "FPR",
        "FNR",
        "TP",
        "TN",
        "FP",
        "FN",
    ]
].copy()

baseline_publication.to_csv(
    TABLES_DIR
    / "table01_baseline16_validation_results.csv",
    index=False,
)


tuned_publication = tuned_df[
    [
        "Rank",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "FPR",
        "FNR",
        "TP",
        "TN",
        "FP",
        "FN",
    ]
].copy()

tuned_publication.to_csv(
    TABLES_DIR
    / "table02_tuned_top5_validation_results.csv",
    index=False,
)


constrained_publication = (
    selected_threshold_df[
        selected_threshold_df[
            "Operating Point"
        ]
        == "Constrained Maximum F2"
    ][
        [
            "Model",
            "Threshold",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "F2-score",
            "FPR",
            "FNR",
            "FP",
            "FN",
            "False Negatives Reduced vs 0.50",
            "Additional False Positives vs 0.50",
        ]
    ]
    .sort_values(
        "F2-score",
        ascending=False,
    )
)

constrained_publication.to_csv(
    TABLES_DIR
    / "table03_constrained_security_validation_results.csv",
    index=False,
)


holdout_publication = holdout_df[
    [
        "Objective",
        "Model",
        "Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "F2-score",
        "FPR",
        "FNR",
        "TP",
        "TN",
        "FP",
        "FN",
        "ROC-AUC",
        "PR-AUC",
    ]
].copy()

holdout_publication.to_csv(
    TABLES_DIR
    / "table04_objective_specific_holdout_results.csv",
    index=False,
)


# ------------------------------------------------------------
# 17. Save LaTeX versions
# ------------------------------------------------------------

def save_latex_table(
    dataframe,
    output_path,
    caption,
    label,
):
    latex_text = dataframe.to_latex(
        index=False,
        float_format="%.4f",
        caption=caption,
        label=label,
        escape=True,
    )

    output_path.write_text(
        latex_text,
        encoding="utf-8",
    )


save_latex_table(
    baseline_publication,
    TABLES_DIR
    / "table01_baseline16_validation_results.tex",
    caption=(
        "Validation performance of the sixteen "
        "baseline intrusion-detection models."
    ),
    label="tab:baseline16",
)

save_latex_table(
    tuned_publication,
    TABLES_DIR
    / "table02_tuned_top5_validation_results.tex",
    caption=(
        "Validation performance of the five "
        "hyperparameter-tuned models."
    ),
    label="tab:tuned_top5",
)

save_latex_table(
    constrained_publication,
    TABLES_DIR
    / "table03_constrained_security_validation_results.tex",
    caption=(
        "Security-oriented validation operating points "
        "selected under an FPR constraint of 5 percent."
    ),
    label="tab:security_thresholds",
)

save_latex_table(
    holdout_publication,
    TABLES_DIR
    / "table04_objective_specific_holdout_results.tex",
    caption=(
        "Objective-specific performance on the "
        "holdout test set."
    ),
    label="tab:holdout_results",
)


# ------------------------------------------------------------
# 18. Save Stage 7 metadata
# ------------------------------------------------------------

metadata = {
    "Experiment":
        "IDS2018 Clean Validation V2",

    "Stage":
        "Publication-ready asset generation",

    "Figures Generated":
        len(
            list(
                FIGURES_DIR.glob("*.png")
            )
        ),

    "CSV Tables Generated":
        len(
            list(
                TABLES_DIR.glob("*.csv")
            )
        ),

    "LaTeX Tables Generated":
        len(
            list(
                TABLES_DIR.glob("*.tex")
            )
        ),

    "Balanced Model":
        "XGBoost Tuned",

    "Balanced Threshold":
        0.51,

    "Security Model":
        "LightGBM Tuned",

    "Security Threshold":
        0.26,

    "Training Performed":
        False,

    "Model Selection Performed":
        False,

    "Threshold Selection Performed":
        False,
}

METADATA_PATH = (
    METADATA_DIR
    / "publication_assets_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 19. Display output inventory
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("PUBLICATION FIGURES")
print("=" * 92)

for path in sorted(
    FIGURES_DIR.glob("*.png")
):
    print(path)


print("\n" + "=" * 92)
print("PUBLICATION TABLES")
print("=" * 92)

for path in sorted(
    TABLES_DIR.iterdir()
):
    print(path)


# ------------------------------------------------------------
# 20. Create Stage 7 archive
# ------------------------------------------------------------

archive_base = (
    "/kaggle/working/"
    "ids2018_clean_validation_v2_"
    "stage07_publication_assets"
)

archive_file = Path(
    archive_base + ".zip"
)

if archive_file.exists():
    archive_file.unlink()

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    STAGE_DIR,
)

print("\nArchive created:")
print(archive_path)

All required Stage 7 inputs are available.

PUBLICATION FIGURES
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figure01_baseline16_f1_comparison.png
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figure02_tuned_top5_f1_comparison.png
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figure03_constrained_security_f2_comparison.png
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figure04_xgboost_threshold_tradeoff.png
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figure05_lightgbm_threshold_tradeoff.png
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figure06_final_holdout_objective_comparison.png
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figure07_xgboost_holdout_confusion_matrix.png
/kaggle/working/ids2018_clean_validation_v2/stage07_publication_assets/figures/figur